[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap04/cap04_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 4 Morphologie mathématique et segmentation d'images

Ce chapitre présente deux thèmes fondamentaux du traitement numérique des images (TNI) : la **morphologie mathématique** et la **segmentation d'images**. La morphologie mathématique fournit un cadre théorique fondé sur la théorie des ensembles pour analyser, affiner et quantifier la forme des objets dans les images binaires et en niveaux de gris, au moyen d'opérateurs fondamentaux tels que l'**érosion** et la **dilatation**. La segmentation, quant à elle, vise à partitionner l'image en régions d'intérêt, en séparant les objets du fond et en produisant des représentations adaptées à l'analyse et à l'interprétation.

Le chapitre débute par le **seuillage**, l'une des techniques de segmentation les plus importantes, en introduisant la méthode automatique d'**Otsu** et en revisit l'analyse des histogrammes au moyen de la variance interclasses, présentée au chapitre 1. Ensuite, sont étudiés les principaux opérateurs de la morphologie mathématique, notamment l'érosion, la dilatation, l'ouverture, la fermeture et la reconstruction morphologique, qui permettent d'affiner les masques binaires et de préserver les structures pertinentes des objets. Enfin, sont présentées des techniques de segmentation fondée sur les régions, telles que l'**étiquetage des composantes connexes**, la **transformée de distance** et l'algorithme ***watershed*** basé sur les marqueurs, aboutissant à l'extraction de descripteurs géométriques et à la génération de *bounding boxes* compatibles avec les systèmes modernes de détection d'objets.

## 4.1 Objectifs

À la fin de ce chapitre, vous serez en mesure de :

* **Appliquer la seuillage :** Comprendre le critère automatique d'Otsu par maximisation de la variance interclasses ($\sigma_B^2$) et sélectionner des stratégies de prétraitement adaptées pour faciliter la segmentation ;
* **Maîtriser la morphologie binaire :** Comprendre et appliquer l'érosion ($A\ominus B$) et la dilatation ($A\oplus B$) comme opérateurs fondamentaux, en dérivant l'ouverture ($A\circ B$), la fermeture ($A\bullet B$) et les opérations basées sur la reconstruction morphologique, comme `mm.clohole` et `mm.edgeoff` ;
* **Appliquer la morphologie en niveaux de gris :** Utiliser le gradient morphologique et les filtres *top-hat* pour le rehaussement et l'analyse de structures locales ;
* **Étiqueter les composantes connexes :** Identifier et séparer les régions connectées dans les images binaires au moyen d'algorithmes d'étiquetage ;
* **Appliquer la transformée de distance :** Interpréter et calculer les distances au fond en utilisant des approches morphologiques et des métriques géométriques ;
* **Segmenter par régions :** Construire des *pipelines* de segmentation basés sur des marqueurs en utilisant la transformée de distance et l'algorithme ***watershed*** ;
* **Extraire des descripteurs géométriques :** Calculer des propriétés telles que l'aire, le périmètre, le centroïde, la circularité et les *bounding boxes* au moyen de `cv2.connectedComponentsWithStats` et `cv2.findContours` ;
* **Relier le TNI et la vision par ordinateur :** Comprendre comment les descripteurs extraits par segmentation peuvent être convertis dans des formats utilisés par les détecteurs modernes, comme YOLO.

In [1]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm
import numpy as np

✅ Environnement prêt. Morph : 1.1.9 | OpenCV : 5.0.0


## 4.2 Seuilletage

Le **seuilletage** (*thresholding*) est l'une des formes les plus simples et efficaces de segmentation d'images. Son objectif est de classer chaque pixel en deux classes d'intensité, généralement associées à *objet* et *fond* :

<a id="eq-04-limiar"></a>
$$
g(x,y) =
\begin{cases}
255, & \text{si } f(x,y) > T \\
0,   & \text{sinon}
\end{cases} \tag{4.1}
$$


où $f(x,y)$ représente l'intensité du pixel dans l'image originale et $g(x,y)$ l'image binaire résultante.

Le choix du seuil $T$ est important pour la qualité de la segmentation. La méthode d'**Otsu** détermine automatiquement le seuil optimal en maximisant la **variance interclasses** $\sigma_B^2$ définie par :

<a id="eq-04-otsu"></a>
$$
\sigma_B^2(T) =
w_0(T)\,w_1(T)\,
\bigl[\mu_0(T)-\mu_1(T)\bigr]^2 \tag{4.2}
$$


où :

- $w_0(T)$ et $w_1(T)$ sont les probabilités cumulées des classes fond et objet ;
- $\mu_0(T)$ et $\mu_1(T)$ sont les moyennes d'intensité de ces classes ;
- $\sigma_B^2(T)$ représente la variance interclasses pour un seuil donné $T$.

La méthode fonctionne mieux lorsque l'histogramme présente deux groupes d'intensités relativement séparés. Pour cela, l'algorithme évalue tous les seuils possibles de l'image — typiquement dans l'intervalle $[0,255]$ pour des images 8 bits — et sélectionne la valeur qui maximise la variance entre classes, notée $\sigma_B^2$ :

$$
T^* =
\arg\max_{T \in [0,255]}
\sigma_B^2(T)
$$

> ### 📝 Otsu suppose des histogrammes bimodaux
>
> La méthode d'Otsu produit de meilleurs résultats lorsque l'histogramme présente deux pics bien définis (*bimodalité*), correspondant au fond et à l'objet. Plus la séparation entre ces pics est grande et plus le maximum de $\sigma_B^2$ est prononcé, plus le seuil obtenu tend à être fiable.
>
> Dans les images avec un éclairage non uniforme ou de multiples régions d'intensité, les techniques de **seuilletage adaptatif** — dans lesquelles le seuil est calculé localement — produisent généralement des segmentations plus robustes.
>
> L'indice $B$ dans $\sigma_B^2$ signifie ***between classes*** (*entre classes*). Ainsi, $\sigma_B^2$ représente la **variance entre les classes** (*between-class variance*).

### 4.2.1 Image de pièces de monnaie

L’image utilisée pour pratiquer la segmentation est une photographie d’une collection de pièces de monnaie de différents pays et époques ([Figure 4.1](#fig-04-coins)). Crédit : GAZI.MD.AHAD (CC BY-SA 4.0). Elle présente des objets circulaires aux contours bien définis, ce qui la rend idéale pour illustrer le seuillage, les opérateurs morphologiques, la transformée de distance, le *watershed* et les descripteurs de forme.

In [2]:
import os

url     = "https://upload.wikimedia.org/wikipedia/commons/2/25/GAZI.MD.AHAD_11.jpg"
caminho = "imagens/coins.jpg"

if not os.path.exists(caminho):
    os.makedirs("imagens", exist_ok=True)
    img_obj = mm.read(url, pil=True)
    mm.write(img_obj, caminho)
else:
    img_obj = mm.read(caminho, pil=True)

img_coins_color = np.array(img_obj)
img_coins_gray  = mm.gray(img_coins_color)

print(f"Dimensions [y,x,c] : {img_coins_color.shape}")
mm.show(img_coins_color, scale=30)

Dimensions [y,x,c] : (2560, 1920, 3)


<Figure size 576x768 with 1 Axes>

**Figure 4.1:** Image avec des pièces de différents types. Crédit : GAZI.MD.AHAD (CC BY-SA 4.0).


### 4.2.2 Escolha do Pré-processamento para o Otsu

A qualidade do método de Otsu depende diretamente do quão **bimodal** é o histograma da imagem de entrada. A imagem original das moedas apresenta iluminação não uniforme e moedas escuras (cobre oxidado) com intensidades próximas às do fundo escuro, tornando o histograma pouco bimodal.

Para minimizar essas limitações, serão avaliadas duas técnicas clássicas de pré-processamento, apresentadas no capítulo anterior, aplicadas antes da etapa de limiarização. A [Tableau 4.1](#tbl-preproc-otsu) resume as características de cada abordagem. Essas técnicas buscam aumentar a separação entre objeto e fundo, tornando o histograma mais próximo de uma distribuição bimodal.

<a id="tbl-preproc-otsu"></a>

**Tabela 4.1:** Técnicas de pré-processamento avaliadas para melhorar a separação entre moedas e fundo antes da aplicação do método de Otsu.

| Técnica | O que faz | Quando usar |
|:--------|:----------|:------------|
| **CLAHE** | Equalização de histograma adaptativa local | Baixo contraste global ou regional |
| **Gaussiano** | Suavização por convolução com gaussiana | Ruído de alta frequência (textura do fundo) |


A função `cv2.createCLAHE(clipLimit, tileGridSize)` divide a imagem em blocos e aplica equalização de histograma em cada um deles, limitando a amplificação de ruído por meio do parâmetro `clipLimit`. Já o filtro Gaussiano (`cv2.GaussianBlur`) suaviza texturas finas que poderiam criar falsos picos no histograma.

Para comparar objetivamente qual pré-processamento produz a melhor entrada para o método de Otsu, serão apresentados, para cada versão da imagem, a própria imagem, o histograma com o limiar ótimo $T^*$ destacado, a curva $\sigma_B^2(T)$ e o resultado da binarização.

> ### 📝 Critério de comparação
>
> A versão que apresentar o maior valor de $\sigma_B^2$ em seu pico fornece a melhor separação entre as classes de fundo e objeto e, consequentemente, a melhor entrada para o método de Otsu ([Figure 4.2](#fig-04-otsu-comparacao-histogramas)).

In [3]:
import cv2, io
import matplotlib.pyplot as plt

def otsu_criterio(img):
    h=mm.hist(img); p=h/h.sum();                  # histogramme et probabilités
    sigma2=np.zeros(len(p)) 
    for T in range(1,len(p)):                     # parcourt les seuils
        w0,w1=p[:T].sum(),p[T:].sum()             # probabilités des classes
        if w0*w1==0: continue                     # évite la division par zéro
        mu0=(np.arange(T)*p[:T]).sum()/w0         # moyenne fond
        mu1=(np.arange(T,len(p))*p[T:]).sum()/w1  # moyenne objet
        sigma2[T]=w0*w1*(mu0-mu1)**2              # σ²B(T)
    return sigma2,np.argmax(sigma2)               # courbe et T optimal

def fig2img(fig):
    b=io.BytesIO(); fig.savefig(b,format='png',dpi=100)   # figure → buffer
    plt.close(fig); b.seek(0)
    return np.array(plt.imread(b))                        # buffer → array

def plot_curve(y,T,title,ylabel,color):
    fig,ax=plt.subplots(figsize=(4,3))
    ax.plot(y,color=color) if ylabel=="σ²B" else ax.bar(range(len(y)),y,color=color,width=1)
    ax.axvline(T,color='red',lw=2,label=f"T*={T}")       # seuil optimal
    ax.set(xlabel="T" if ylabel=="σ²B" else "Intensidade",ylabel=ylabel)
    ax.legend(fontsize=8); plt.tight_layout()
    return fig2img(fig)

# ── Pré-traitements ───────────────────────────────────────────────────────
clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
img_clahe = clahe.apply(img_coins_gray)
img_gauss = cv2.GaussianBlur(img_clahe,(5,5),0)
imgs0=[("Original",img_coins_gray),
       ("CLAHE",img_clahe),
       ("CLAHE+Gauss",img_gauss)]

# ── Tableau comparatif ───────────────────────────────────────────────────────
print(f"{'Versão':<18}{'T*':>6}{'σ²B pico':>14}")
print("-"*40)

imgs,titles=[],[]
for nome,img in imgs0:
    sigma2,T=otsu_criterio(img)                           # calcule σ²B(T)
    print(f"{nome:<18}{T:>6}{sigma2[T]:>14.4e}")
    imgs += [
        img,                                              # image
        plot_curve(mm.hist(img),T,f"Hist T*={T}","Freq.","steelblue"),
        plot_curve(sigma2,T,"σ²B(T)","σ²B","darkorange"),
        mm.threshold(img)                                # Otsu final
    ]
    titles += [nome,f"Hist T*={T}","σ²B(T)",f"Otsu T*={T}"]

# ── Affichage final ───────────────────────────────────────────────────────────
mm.show(imgs,titles=titles,cols=4,figsize=(12,12),dpi=200)

Versão                T*      σ²B pico
----------------------------------------


Original             105    1.9437e+03


CLAHE                122    2.4695e+03


CLAHE+Gauss          123    2.4283e+03


<Figure size 2400x2400 with 12 Axes>

**Figure 4.2:** Comparaison des pré-traitements : image | histogramme+T* | σ²B(T) | Otsu. La meilleure séparation bimodale indique le seuil le plus fiable.


### 4.2.3 Résultat : CLAHE comme meilleur prétraitement

L’analyse de la [Figure 4.2](#fig-04-otsu-comparacao-histogramas) indique que le **CLAHE** a obtenu la valeur la plus élevée de variance inter-classes ($\sigma_B^2 \approx 2{,}47 \times 10^3$), avec un seuil optimal $T^* = 122$. Bien que la combinaison **CLAHE+Gaussien** ait produit un résultat très similaire ($\sigma_B^2 \approx 2{,}43 \times 10^3$, $T^* = 123$), le critère quantitatif de la méthode d’Otsu favorise légèrement l’utilisation du CLAHE seul.

Sur le plan visuel, les images binarisées obtenues avec le CLAHE et le CLAHE+Gaussien sont pratiquement équivalentes. La différence entre les deux approches devient plus évidente dans l’analyse des histogrammes et des valeurs de $\sigma_B^2(T)$ que dans l’inspection directe des segmentations résultantes. Ainsi, le choix du CLAHE repose principalement sur la maximisation de la séparation statistique entre les classes de fond et d’objet.


> ### 💡 Interprétation des résultats
>
> Remarquez que les prétraitements avec CLAHE et CLAHE+Gaussien produisent des histogrammes et des seuils optimaux très proches ($T^*=122$ et $T^*=123$). Par conséquent, les images binarisées résultantes sont également très similaires. Dans ce cas, la décision ne repose pas sur des différences visuelles marquantes, mais sur le critère objectif de la méthode d’Otsu : la valeur la plus élevée de $\sigma_B^2$ indique la meilleure séparation entre les classes.

## 4.3 Morphologie mathématique

La **morphologie mathématique** est une théorie basée sur les ensembles, utilisée pour analyser la forme et la structure des objets dans les images. Contrairement aux filtres linéaires présentés au chapitre 3, les opérateurs morphologiques sont **non linéaires**, car ils reposent sur des opérations de minimum, de maximum et d'inclusion spatiale, plutôt que sur des combinaisons linéaires d'intensités. Ces opérateurs agissent sur le voisinage de chaque pixel au moyen d'un **élément structurant** $\mathbb{B}$, chargé de définir la forme et la taille de la région analysée.

Dans les images binaires et en niveaux de gris avec des éléments plans, l'élément structurant translaté à la position $x$ est défini spatialement comme :

$$
\mathbb{B}_x = \{ x + b \mid b \in \mathbb{B} \}
$$

Dans les régions de bord de l'image, une partie de l'ensemble $\mathbb{B}_x$ peut dépasser le domaine physique de la scène ($\mathbb{E}$). Pour garantir la cohérence mathématique des opérateurs primitifs à ces frontières, on suppose théoriquement que l'espace extérieur au domaine de l'image est rempli avec l'**élément neutre** de l'opération correspondante (infini positif pour l'érosion et infini négatif pour la dilatation), empêchant ainsi que l'environnement externe corrompe les structures internes de l'objet.

Lorsque l'élément structurant associe des poids à ses éléments — c'est-à-dire $b: \mathbb{B} \to \mathbb{Z}$ — il est appelé **fonction structurante** ou **élément structurant non plan**.

Développée par Matheron et Serra dans les années 1960 pour les images binaires, puis étendue aux niveaux de gris, la morphologie mathématique fonde des opérateurs tels que le gradient morphologique, le *top-hat*, le *watershed* et la transformée de distance, tous dérivés de deux primitifs : l'**érosion** et la **dilatation** [@matheron1975random; Serra (1982)].

### 4.3.1 Érosion et Dilatation

Les deux opérateurs primitifs sont définis de manière unifiée pour les images en niveaux de gris ($f: \mathbb{E} \to \mathbb{Z}$) et, par restriction au domaine $\{0,1\}$, également pour les images binaires.

#### 4.3.1.1 Érosion

L'**érosion** d'une image $f$ par un élément structurant $b: \mathbb{B} \to \mathbb{Z}$ est définie formellement par :

<a id="eq-04-erosao"></a>
$$
\varepsilon_b(f)(x) = (f \ominus b)(x) = \min_{z \in \mathbb{B}}\{\, f(x + z) - b(z) \,\}, \quad \forall\, x \in \mathbb{E} \tag{4.3}
$$


En pratique, l'érosion remplace l'intensité du pixel $x$ par la valeur minimale résultant de la différence entre l'image et l'élément structurant dans le voisinage défini par le domaine $\mathbb{B}$. Des valeurs positives dans les poids de $b(z)$ forcent le résultat local vers le bas, « creusant » plus profondément le relief de l'image et intensifiant l'érosion.

Dans le cas **plat** (où les poids sont nuls à l'intérieur du domaine, c'est-à-dire $b \equiv 0$), l'expression se simplifie en le minimum local pur :

$$
\varepsilon_B(f)(x) = \min\{\, f(y) : y \in \mathbb{B}_x \,\}
$$

Dans les images binaires, cette opération équivaut à exiger que l'ensemble $\mathbb{B}$, translaté à la coordonnée $x$, soit **complètement contenu** dans l'objet $A$ :

$$
A \ominus \mathbb{B} = \{\, z \in \mathbb{E} \mid \mathbb{B}_z \subseteq A \,\}
$$

**Effet visuel :** *Rétrécit* les objets et les structures claires, éliminant les protubérances, les pics brillants ou les bruits qui sont géométriquement plus petits que le domaine $\mathbb{B}$.

#### 4.3.1.2 Implémentation de l’érosion

La version didactique `mm.ero0` implémente le cas particulier de l’érosion avec **élément structurant plat**. Pour chaque pixel $(y,x)$, la fonction parcourt les voisins spatiaux autorisés par $B$ et stocke la plus petite valeur trouvée dans l’image d’entrée $f$, reproduisant directement l’opération de minimum local décrite dans la [Équation 4.3](#eq-04-erosao) pour $b \equiv 0$.

Notez que les valeurs des voisins sont toujours lues de manière statique à partir de l’image originale $f$ ; la matrice de sortie $g$ est utilisée exclusivement pour enregistrer le minimum accumulé du voisinage courant. Ainsi, le résultat final est invariant par rapport à l’ordre de balayage des pixels (que ce soit par lignes ou par colonnes).

La fonction auxiliaire `_viz` calcule les coordonnées des voisins valides à l’intérieur des limites physiques de l’image. Sur les bords, l’initialisation de l’accumulateur à `255` émule exactement le remplissage par élément neutre exigé par la théorie. La fonction d’interface `mm.ero` recourt quant à elle à l’implémentation native et optimisée d’OpenCV (`cv2.erode`) lorsque l’élément structurant est plat, basculant vers la routine générale `mm.ero1` si l’élément possède des poids topographiques.

L’exemple informatique suivant illustre l’application d’un élément structurant en croix (`mm.secross()`) mis en évidence dans la [Figure 4.3](#fig-04-elemento-cruz), en comparant l’exécution de la variante didactique en boucle (`mm.ero0`) avec le moteur de calcul d’OpenCV (`mm.ero`).

In [4]:
B_cruz = mm.secross()
mm.drawImgPlt(B_cruz, scale=20)

<Figure size 180x180 with 1 Axes>

**Figure 4.3:** Élément structurant en forme de croix ($B_{\\text{cruz}}$) utilisé pour la connectivité-4.


In [5]:
def _viz(f, B, y, x):
    """Génère (vy, vx, b_val) pour chaque voisin valide de (y,x)."""
    H, W = f.shape
    Bh, Bw = B.shape
    oh, ow = -Bh/2 + 0.5, -Bw/2 + 0.5  # offsets fixes
    for by, bx in np.ndindex(Bh, Bw):
        vy, vx = int(y + by + oh), int(x + bx + ow)
        if 0 <= vy < H and 0 <= vx < W:
            yield vy, vx, B[by, bx]

def ero(f, Bc=np.zeros((3,3),dtype='uint8')):
    """Érosion (OpenCV ou avec poids)."""
    try:    return cv2.erode(f, Bc)
    except: return mm.ero1(f, Bc)

def ero0(f, Bc=np.ones((3,3),dtype='uint8')):
    """Érosion classique sans poids."""
    g = np.empty_like(f)
    for y in range(f.shape[0]):
        for x in range(f.shape[1]):
            g[y,x] = 255
            for vy,vx,bv in mm._viz(f,Bc,y,x):
                if bv and g[y,x] > f[vy,vx]: g[y,x] = f[vy,vx]
    return g

# Script de Test et de Validation
B = mm.secross()
print("Élément structurant B :")
print(mm.drawImage(B))

f = mm.randomImage(5,5)
print("Image originale f :")
print(mm.drawImage(f))

print("Érosion plane didactique (ero0) :")
print(mm.drawImage(ero0(f, B)))

print("Érosion optimisée OpenCV (ero) :")
print(mm.drawImage(ero(f, B)))

Élément structurant B :
0 1 0 
1 1 1 
0 1 0 

Image originale f :
0 3 5 2 0 
8 8 4 8 5 
4 3 2 9 3 
6 3 9 9 9 
8 0 8 8 7 

Érosion plane didactique (ero0) :
0 0 2 0 0 
0 3 2 2 0 
3 2 2 2 3 
3 0 2 8 3 
0 0 0 7 7 

Érosion optimisée OpenCV (ero) :
0 0 2 0 0 
0 3 2 2 0 
3 2 2 2 3 
3 0 2 8 3 
0 0 0 7 7 



La fonction `_viz` utilise `yield` pour générer chaque voisin à la demande, sans stocker tous les résultats en mémoire dans une liste. Dans l’expérience ci-dessous, une fenêtre de 3000×3000 produit 9 millions de voisins. L’implémentation basée sur une liste a consommé plus de 1 Go de RAM et a pris environ 46 s, tandis que la version avec `yield` a consommé une mémoire négligeable et s’est exécutée en 38 s. En traitement d’images, les générateurs sont particulièrement utiles pour parcourir de grandes voisinages de manière efficace.

In [6]:
import tracemalloc

def lista(n): return [(i, i) for i in range(n)]
def gera(n):
    for i in range(n): yield i, i

for nome, f in [("LISTA", lista), ("YIELD", gera)]:
    tracemalloc.start()
    sum(x+y for x,y in f(9_000_000))
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"{nome} : {pico/1024/1024:.1f} Mo ")

LISTA : 830.8 Mo 


YIELD : 0.0 Mo 


#### 4.3.1.3 Dilatation

La **Dilatation** d'une image $f$ par une fonction structurante $b: \mathbb{B} \to \mathbb{Z}$ est définie formellement par :

<a id="eq-04-dilatacao"></a>
$$
\delta_b(f)(x) = (f \oplus b)(x) = \max_{z \in \mathbb{B}}\{\, f(x - z) + b(z) \,\}, \quad \forall\, x \in \mathbb{E} \tag{4.4}
$$


En pratique, la dilatation remplace l'intensité du pixel $x$ par la plus grande valeur résultant de la somme entre l'image et l'élément structurant dans le voisinage défini. L'argument d'inversion spatiale ($x - z$) indique que la dilatation évalue implicitement l'élément transposé (réfléchi) $\hat{b}$, propriété fondamentale pour assurer la dualité mathématique par rapport à l'érosion.

Dans le cas **plat** (où les poids sont nuls à l'intérieur du domaine, c'est-à-dire $b \equiv 0$), l'expression se réduit au maximum local pur :

$$
\delta_B(f)(x) = \max\{\, f(y) : y \in \mathbb{B}_x \,\}
$$

Dans les images binaires, cette opération équivaut à exiger que l'ensemble réfléchi $\hat{\mathbb{B}}$, translaté à la coordonnée $x$, possède une **intersection non vide** avec l'objet $A$ :

$$
A \oplus \mathbb{B} = \{\, z \in \mathbb{E} \mid \hat{\mathbb{B}}_z \cap A \neq \varnothing \,\}
$$

**Effet visuel :** *Étend* les structures claires de l'image, augmente le remplissage des objets, connecte les composantes proches et élimine les canaux, fosses sombres ou vallées qui sont géométriquement plus petits que le domaine $\mathbb{B}$.

#### 4.3.1.4 Implémentation de la dilatation

La version didactique `mm.dil0` implémente le cas particulier de dilatation avec **élément structurant plan**. Pour chaque pixel $(y,x)$, la fonction parcourt les voisins spatiaux autorisés par $B$ et stocke la plus grande valeur trouvée dans l’image d’entrée $f$, reproduisant directement l’opération de maximum local pour $b \equiv 0$.

Avant de lancer le balayage spatial, l’élément structurant subit une réflexion géométrique au moyen de l’instruction `np.flip(Bc)` afin de construire explicitement la matrice transposée $\hat{B}$ exigée par la théorie. Pour des masques parfaitement symétriques (tels que des croix, des carrés et des disques centrés à l’origine), cette réflexion ne modifie pas l’arrangement des pixels ; toutefois, pour les éléments asymétriques, cette étape est strictement nécessaire afin de garantir l’équivalence avec les définitions formelles et de préserver les lois de dualité.

Comme constaté pour l’opérateur d’érosion, les valeurs des voisins sont toujours lues de manière statique à partir de la matrice originale $f$, tandis que la matrice de sortie $g$ agit uniquement comme le registre du maximum accumulé du voisinage. Aux frontières de l’image, l’initialisation de l’accumulateur à `0` émule avec exactitude le remplissage externe par élément neutre ($-\infty$, ou zéro dans les représentations sur 8 bits), garantissant que les bords physiques de la scène soient dilatés en parfaite conformité avec le standard adopté par OpenCV.

L’exemple informatique ci-dessous illustre l’application pratique d’un élément en croix (`mm.secross()`), validant la cohérence entre la logique en boucles (`mm.dil0`) et la méthode native industrielle (`mm.dil`).

In [7]:
def dil(f, Bc=np.zeros((3,3),dtype='uint8')):
    """Dilatation (OpenCV ou avec des poids)."""
    try:    return cv2.dilate(f, Bc)
    except: return mm.dil1(f, Bc)

def dil0(f, Bc=np.zeros((3,3),dtype='uint8')):
    """Dilatation plane suivant rigoureusement la théorie."""
    g = np.empty_like(f) 
    Bc = np.flip(Bc)     # réflexion explicite : B̂
    for y in range(f.shape[0]):
        for x in range(f.shape[1]):
            g[y,x] = 0 # Initialise avec la valeur minimale pour chercher le maximum
            for vy,vx,bv in mm._viz(f,Bc,y,x):
                if bv and g[y,x] < f[vy,vx]:
                    g[y,x] = f[vy,vx]
    return g

# Script de test et de validation
B = mm.secross()
print("Élément structurant B :")
print(mm.drawImage(B))

f = mm.randomImage(5,5)
print("Image originale f :")
print(mm.drawImage(f))

print("Dilatation plane didactique (dil0) :")
print(mm.drawImage(dil0(f, B)))

print("Dilatation optimisée OpenCV (dil) :")
print(mm.drawImage(dil(f, B)))

Élément structurant B :
0 1 0 
1 1 1 
0 1 0 

Image originale f :
6 4 3 1 2 
7 7 2 9 7 
7 2 7 6 7 
0 3 6 2 4 
1 0 2 4 7 

Dilatation plane didactique (dil0) :
7 7 4 9 7 
7 7 9 9 9 
7 7 7 9 7 
7 6 7 6 7 
1 3 6 7 7 

Dilatation optimisée OpenCV (dil) :
7 7 4 9 7 
7 7 9 9 9 
7 7 7 9 7 
7 6 7 6 7 
1 3 6 7 7 



> ### 📝 Note : Le Confront des Signes ($f(x+z)$ vs $f(x-z)$)
>
> Comparez les définitions formelles de l'érosion ([Équation 4.3](#eq-04-erosao)) et de la dilatation ([Équation 4.4](#eq-04-dilatacao)). Considérez un élément structurant asymétrique à droite $\mathbb{B}=\{0,1\}$ (origine et un pixel à droite) appliqué à la position $x=10$.
>
> 1. **Dans l'érosion** ([Équation 4.3](#eq-04-erosao)) :
>
>    $$
>    \min\{f(x+z)-b(z)\}
>    $$
>
>    * $z=0 \Rightarrow f(10+0)=\mathbf{f(10)}$
>    * $z=1 \Rightarrow f(10+1)=\mathbf{f(11)}$
>
>    L'opérateur consulte le pixel courant ($10$) et le pixel à droite ($11$), préservant l'orientation originale de $\mathbb{B}$.
>
> 2. **Dans la dilatation** ([Équation 4.4](#eq-04-dilatacao)) :
>
>    $$
>    \max\{f(x-z)+b(z)\}
>    $$
>
>    * $z=0 \Rightarrow f(10-0)=\mathbf{f(10)}$
>    * $z=1 \Rightarrow f(10-1)=\mathbf{f(9)}$
>
>    En raison du signe négatif ($-z$), avancer dans l'élément structurant correspond à reculer dans l'image, ce qui fait que la dilatation consulte le pixel à gauche ($9$).
>
> La fonction `_viz`, utilisée dans `morph.py`, génère des voisins par déplacements additifs de la forme $x+z$. Pour cette raison, l'implémentation de `mm.dil0` reflète au préalable l'élément structurant au moyen de `np.flip(B)`. Après la réflexion, le balayage basé sur $x+z$ accède alors exactement aux mêmes points définis par l'expression théorique $f(x-z)$ de la dilatation dans [Équation 4.4](#eq-04-dilatacao).
>
> Pour les éléments structurants symétriques (comme les disques, les carrés et les croix centrées), la réflexion ne modifie pas le masque. En revanche, pour les éléments asymétriques, cette étape est indispensable pour que l'implémentation reproduise correctement la définition mathématique de la dilatation et préserve la dualité érosion–dilatation.

> ### 📝 Dualité érosion–dilatation
>
> L'érosion et la dilatation sont **duales par complémentation**. Cela signifie qu'un opérateur peut être entièrement obtenu à partir de l'autre, à condition d'agir sur le complément de l'image en utilisant l'élément structurant réfléchi $\hat{B}$ :
>
> $$
> (A \ominus B)^c = A^c \oplus \hat{B} \quad \Longleftrightarrow \quad A \ominus B = (A^c \oplus \hat{B})^c
> $$
>
> De manière analogue, la **dilatation peut également être obtenue à partir de l'érosion** :
>
> $$
> (A \oplus B)^c = A^c \ominus \hat{B} \quad \Longleftrightarrow \quad A \oplus B = (A^c \ominus \hat{B})^c
> $$
>
> En termes pratiques, l'érosion d'un objet peut être obtenue par la dilatation de son complément, suivie de la complémentation du résultat (et vice-versa). Dans l'implémentation du paquet `morph.py`, les versions didactiques `mm.ero0` et `mm.dil0` rendent explicite cette structure au moyen de boucles (*loops*), tandis que `mm.ero` et `mm.dil` délèguent les opérations à OpenCV pour une plus grande efficacité computationnelle.

> ### 📝 Conditions aux limites et images finies
>
> En morphologie mathématique classique, définie sur un domaine infini (typiquement $\mathbb{Z}^2$), cette dualité est exacte. Dans les images numériques, cependant, on travaille avec des matrices finies, et le résultat dépend de la manière dont sont traités les pixels situés hors de l'image.
>
> Pour que les identités de dualité restent valides, le complément doit être défini par rapport au même univers et les conditions aux limites adoptées pour l'érosion et pour la dilatation doivent être complémentaires entre elles. Par exemple, si l'érosion suppose que les pixels externes appartiennent à l'objet ($255$), alors la dilatation appliquée au complément doit supposer que ces mêmes pixels externes appartiennent au fond ($0$).
>
> Lorsque différentes stratégies de remplissage sont utilisées (réplication, réflexion, valeur constante, etc.), la dualité théorique peut cesser d'être satisfaite exactement dans les régions proches des bords de l'image.

Pour illustrer numériquement les opérateurs morphologiques et la dualité érosion–dilatation, la [Figure 4.4](#fig-04-ero-dil-didatico) présente une image binaire 10×10 traitée avec un élément structurant en forme de « L ». Dans l'implémentation de `morph.py`, l'origine de $B$ est fixée au centre géométrique du masque — position $(1,1)$ pour un *kernel* 3×3 — et doit correspondre à un élément actif pour que l'érosion se comporte correctement (comme discuté précédemment). L'élément structurant $B_L$ défini ci-dessous satisfait cette condition. La [Figure 4.5](#fig-04-sim-04-erosao) complète l'analyse avec un simulateur interactif de l'érosion, permettant de visualiser le déplacement de l'élément structurant sur l'image et d'identifier les positions où il reste complètement contenu dans l'objet.

In [8]:
A = np.array([
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,1,1,1,0,0,0,0],
    [0,0,1,1,1,1,1,0,0,0],
    [0,1,1,1,1,1,1,1,0,0],
    [0,1,1,1,1,1,1,1,0,0],
    [0,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,0,0],
    [0,0,0,1,1,1,1,0,0,0],
    [0,0,0,0,1,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0]], dtype=np.uint8) * 255

B_L = np.array([[1,0,0],
                [1,1,0],
                [1,1,0]], dtype=np.uint8)   # origine au centre (1,1), élément actif

# Élément structurant réfléchi B̂
B_hat = np.flip(B_L)
print("Élément structurant B_L :")
print(mm.drawImg(B_L))
print("Élément structurant réfléchi B̂ :")
print(mm.drawImg(B_hat))    

# Érosion de A par B_L
img_ero0 = mm.ero0(A, B_L)

# Validation de la dualité : (A ⊖ B)^c = A^c ⊕ B̂
A_c   = 255 - A           # complément de A
ero_c = 255 - img_ero0    # complément de l’érosion — côté gauche
dil_Ac = mm.dil0(A_c, B_hat)             # côté droit

dualidade_ok = np.array_equal(ero_c, dil_Ac)
print(f"Dualité (A ⊖ B)^c == A^c ⊕ B̂ : {dualidade_ok}")
# Transposé (réflexion sur les deux axes) : B̂ utilisé dans la dilatation de la dualité
B_hat = np.flip(B_L)
mm.show(
    [A, A_c, img_ero0, ero_c, dil_Ac],
    titles=["A", "Aᶜ", "A ⊖ B", "(A ⊖ B)ᶜ", "Aᶜ ⊕ B̂"],
    cols=5,
    figsize=(15, 3)
)

Élément structurant B_L :
1 0 0 
1 1 0 
1 1 0 

Élément structurant réfléchi B̂ :
0 1 1 
0 1 1 
0 0 1 

Dualité (A ⊖ B)^c == A^c ⊕ B̂ : True


<Figure size 2250x450 with 5 Axes>

**Figure 4.4:** Érosion et dilatation sur une image binaire 10×10 avec un élément structurant « L » asymétrique 3×3. Validation de la dualité érosion–dilatation.


In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulateur : Érosion morphologique</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">A ⊖ B_L · décalages via _viz</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cliquez sur une cellule du canevas pour déplacer le noyau B_L ou utilisez les curseurs pour tester l'inclusion des contenus.</p>

    <!-- Estatísticas Principais -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:16px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Position X (col)</div>
        <div id="sim-04-erosao_mX" style="font-size:16px;font-weight:700;font-family:monospace;color:#2980b9;">4</div>
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Position Y (ligne)</div>
        <div id="sim-04-erosao_mY" style="font-size:16px;font-weight:700;font-family:monospace;color:#2980b9;">4</div>
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Pixel Érosion</div>
        <div id="sim-04-erosao_mEro" style="font-size:16px;font-weight:700;font-family:monospace;color:#27ae60;">255</div>
      </div>
    </div>

    <!-- Área Gráfica e Controles Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(260px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Canvas Interativo -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <canvas id="sim-04-erosao_Canvas" style="display:block;max-width:100%;height:auto;border-radius:8px;border:1px solid #e4dcc8;cursor:crosshair;background:#ffffff;margin:0 auto;"></canvas>
        <div style="font-size:10px;color:#8a8371;margin-top:8px;">🖱️ Cliquez sur une cellule pour déplacer le noyau B_L</div>
      </div>

      <!-- Painel de Controles e Status -->
      <div style="display:flex;flex-direction:column;gap:12px;">
        
        <!-- Status de Inclusão -->
        <div id="sim-04-erosao_stBox" style="border-radius:10px;border:1.5px solid #27ae60;padding:10px 12px;background:#eafaf1;transition:all 0.15s ease;">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:4px;">
            <span style="font-size:16px;">🟢</span>
            <div id="sim-04-erosao_stTitle" style="font-size:11px;font-weight:700;color:#27ae60;">Succès : contenu !</div>
          </div>
          <div id="sim-04-erosao_stDesc" style="font-size:10.5px;color:#27ae60;line-height:1.4;">Le pixel reçoit 1 (255) dans l'image érodée.</div>
        </div>

        <!-- Sliders de Posição -->
        <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 12px;">
          <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Contrôles de coordonnées</span>
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:6px;">
            <label for="sim-04-erosao_slX" style="font-size:10.5px;color:#8a8371;width:40px;flex-shrink:0;">X (col)</label>
            <input type="range" id="sim-04-erosao_slX" min="0" max="9" step="1" value="4" style="flex:1;cursor:pointer;">
            <span id="sim-04-erosao_vX" style="font-size:11px;font-family:monospace;font-weight:700;min-width:14px;text-align:right;">4</span>
          </div>
          <div style="display:flex;align-items:center;gap:8px;">
            <label for="sim-04-erosao_slY" style="font-size:10.5px;color:#8a8371;width:40px;flex-shrink:0;">Y (ligne)</label>
            <input type="range" id="sim-04-erosao_slY" min="0" max="9" step="1" value="4" style="flex:1;cursor:pointer;">
            <span id="sim-04-erosao_vY" style="font-size:11px;font-family:monospace;font-weight:700;min-width:14px;text-align:right;">4</span>
          </div>
        </div>

        <button id="sim-04-erosao_rstBtn" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;width:100%;transition:all 0.15s ease;">↩ Réinitialiser la position (4, 4)</button>

        <!-- Kernel B_L Informativo -->
        <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 12px;text-align:center;">
          <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Noyau B_L (3×3)</span>
          <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ebf4fd;border:1px solid #2980b9;border-radius:4px;color:#2980b9;">★</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          </div>
          <div id="sim-04-erosao_offsetInfo" style="font-size:10px;font-family:monospace;color:#5e5a4a;line-height:1.5;background:#ffffff;border:1px solid #e4dcc8;border-radius:6px;padding:6px 8px;margin-top:6px;text-align:left;"></div>
        </div>

      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSim04Ero(root){
    if (!root || root.dataset.sim04EroInit) return;
    root.dataset.sim04EroInit = "1";

    const ero_COLS = 10, ero_ROWS = 10, ero_CELL = 32, ero_PAD = 14;
    const ero_W = ero_COLS * ero_CELL + ero_PAD * 2, ero_H = ero_ROWS * ero_CELL + ero_PAD * 2;
    const ero_cv = root.querySelector('#sim-04-erosao_Canvas');
    ero_cv.width = ero_W; 
    ero_cv.height = ero_H;
    const ero_ctx = ero_cv.getContext('2d');

    const ero_A = [
      [0,0,0,0,0,0,0,0,0,0],
      [0,0,0,1,1,1,0,0,0,0],
      [0,0,1,1,1,1,1,0,0,0],
      [0,1,1,1,1,1,1,1,0,0],
      [0,1,1,1,1,1,1,1,0,0],
      [0,1,1,1,1,1,1,0,0,0],
      [0,0,1,1,1,1,1,1,0,0],
      [0,0,0,1,1,1,1,0,0,0],
      [0,0,0,0,1,0,0,0,0,0],
      [0,0,0,0,0,0,0,0,0,0]
    ];

    const ero_B = [
      [1,0,0],
      [1,1,0],
      [1,1,0]
    ];
    const ero_Bh = 3, ero_Bw = 3;

    function ero_vizOffsets(){
      const offs = [];
      for(let by=0; by<ero_Bh; by++) {
        for(let bx=0; bx<ero_Bw; bx++){
          const dr = by - Math.floor(ero_Bh/2);
          const dc = bx - Math.floor(ero_Bw/2);
          offs.push({dr, dc, bv: ero_B[by][bx], by, bx});
        }
      }
      return offs;
    }
    
    const ero_OFFSETS = ero_vizOffsets();
    const ero_ACTIVE  = ero_OFFSETS.filter(o => o.bv === 1);
    const ero_INACTIVE= ero_OFFSETS.filter(o => o.bv === 0);

    function ero_computeEro(){
      const e = Array.from({length:ero_ROWS}, () => new Array(ero_COLS).fill(0));
      for(let y=0; y<ero_ROWS; y++) {
        for(let x=0; x<ero_COLS; x++) {
          let ok = true;
          for(const {dr,dc,bv} of ero_OFFSETS){
            if(!bv) continue;
            const vy = y + dr, vx = x + dc;
            if(vy < 0 || vy >= ero_ROWS || vx < 0 || vx >= ero_COLS || !ero_A[vy][vx]) { 
              ok = false; 
              break; 
            }
          }
          e[y][x] = ok ? 1 : 0;
        }
      }
      return e;
    }
    
    const ero_ERO = ero_computeEro();
    let ero_cx = 4, ero_cy = 4;

    function ero_updateOffsetInfo(){
      const lines = ero_ACTIVE.map(o => {
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        const inBounds = vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS;
        const inA = inBounds && ero_A[vy][vx];
        const mark = inA ? '✓' : '✗';
        return 'B[' + o.by + '][' + o.bx + '] → (' + vy + ',' + vx + ') ' + mark;
      });
      
      root.querySelector('#sim-04-erosao_offsetInfo').innerHTML =
        '<span style="color:#8a8371; font-size:9.5px">offsets ativos @ (y=' + ero_cy + ', x=' + ero_cx + '):</span><br>' +
        lines.map(l => '<span style="color:' + (l.endsWith('✓') ? '#27ae60' : '#c0392b') + '">' + l + '</span>').join('<br>');
    }

    function ero_draw(){
      ero_ctx.clearRect(0, 0, ero_W, ero_H);

      const activeMap = new Map();
      for(const o of ero_ACTIVE){
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        if(vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS) activeMap.set(vy + ',' + vx, ero_A[vy][vx]);
      }
      const inactiveSet = new Set();
      for(const o of ero_INACTIVE){
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        if(vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS) inactiveSet.add(vy + ',' + vx);
      }

      const contained = ero_ACTIVE.every(o => {
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        return vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS && ero_A[vy][vx];
      });

      for(let r=0; r<ero_ROWS; r++) {
        for(let c=0; c<ero_COLS; c++) {
          const x = ero_PAD + c * ero_CELL, y = ero_PAD + r * ero_CELL;
          const key = r + ',' + c;
          const isActive = activeMap.has(key);
          const isInactive = !isActive && inactiveSet.has(key);
          const isCenter = r === ero_cy && c === ero_cx;
          const inA = ero_A[r][c];
          const inE = ero_ERO[r][c];
          
          let fill, stroke, lw=0.5, dash=[];

          if(isActive){
            const inside = activeMap.get(key);
            fill = inside ? 'rgba(243, 156, 18, 0.35)' : 'rgba(192, 57, 43, 0.35)';
            stroke = inside ? '#b9770e' : '#c0392b';
            lw = 2;
          } else if(isCenter){
            fill = 'rgba(41, 128, 185, 0.25)';
            stroke = '#2980b9'; lw = 2; dash = [4,3];
          } else if(isInactive){
            fill = 'rgba(235, 244, 253, 0.4)';
            stroke = '#b9770e'; lw = 1; dash = [3,3];
          } else if(inE){
            fill = 'rgba(39, 174, 96, 0.15)';
            stroke = '#27ae60';
          } else if(inA){
            fill = 'rgba(142, 68, 173, 0.15)';
            stroke = '#8e44ad';
          } else {
            fill = '#fafaf7';
            stroke = '#e4dcc8';
          }

          ero_ctx.setLineDash(dash);
          ero_ctx.fillStyle = fill; 
          ero_ctx.fillRect(x+1, y+1, ero_CELL-2, ero_CELL-2);
          ero_ctx.strokeStyle = stroke; 
          ero_ctx.lineWidth = lw; 
          ero_ctx.strokeRect(x+0.5, y+0.5, ero_CELL-1, ero_CELL-1);
          ero_ctx.setLineDash([]);

          ero_ctx.textAlign = 'center'; 
          ero_ctx.textBaseline = 'middle';
          
          if(isCenter && !isActive){
            ero_ctx.fillStyle = '#2980b9';
            ero_ctx.font = '13px monospace';
            ero_ctx.fillText('★', x+ero_CELL/2, y+ero_CELL/2);
          } else if(isActive){
            const inside = activeMap.get(key);
            ero_ctx.fillStyle = inside ? '#7d5a00' : '#78281f';
            ero_ctx.font = 'bold 11px monospace';
            ero_ctx.fillText(inA ? '1' : '0', x+ero_CELL/2, y+ero_CELL/2);
          } else if(inA){
            ero_ctx.fillStyle = '#4a235a';
            ero_ctx.font = '11px monospace';
            ero_ctx.fillText('1', x+ero_CELL/2, y+ero_CELL/2);
          } else {
            ero_ctx.fillStyle = '#8a8371';
            ero_ctx.font = '11px monospace';
            ero_ctx.fillText('0', x+ero_CELL/2, y+ero_CELL/2);
          }

          if(r===0){ero_ctx.fillStyle='#8a8371'; ero_ctx.font='9px monospace'; ero_ctx.textAlign='center'; ero_ctx.fillText(c, x+ero_CELL/2, ero_PAD/2);}
          if(c===0){ero_ctx.fillStyle='#8a8371'; ero_ctx.font='9px monospace'; ero_ctx.textAlign='center'; ero_ctx.fillText(r, ero_PAD/2, y+ero_CELL/2);}
        }
      }

      root.querySelector('#sim-04-erosao_mX').textContent = ero_cx;
      root.querySelector('#sim-04-erosao_mY').textContent = ero_cy;
      const ev = ero_ERO[ero_cy][ero_cx];
      const mEro = root.querySelector('#sim-04-erosao_mEro');
      mEro.textContent = ev ? '255' : '0';
      mEro.style.color = ev ? '#27ae60' : '#c0392b';
      root.querySelector('#sim-04-erosao_vX').textContent = ero_cx;
      root.querySelector('#sim-04-erosao_vY').textContent = ero_cy;
      root.querySelector('#sim-04-erosao_slX').value = ero_cx;
      root.querySelector('#sim-04-erosao_slY').value = ero_cy;

      const box = root.querySelector('#sim-04-erosao_stBox');
      const title = root.querySelector('#sim-04-erosao_stTitle');
      const desc = root.querySelector('#sim-04-erosao_stDesc');
      
      if(contained){
        box.style.background = '#eafaf1'; box.style.borderColor = '#27ae60';
        title.textContent = 'Sucesso: contido!'; title.style.color = '#27ae60';
        desc.textContent = 'Pixel recebe 1 (255) na imagem erodida.'; desc.style.color = '#27ae60';
      } else {
        box.style.background = '#fdecea'; box.style.borderColor = '#c0392b';
        title.textContent = 'Aviso: B_L sai do objeto!'; title.style.color = '#c0392b';
        desc.textContent = 'Pixel recebe 0 na imagem erodida.'; desc.style.color = '#c0392b';
      }
      
      ero_updateOffsetInfo();
    }

    ero_cv.addEventListener('click', function(e){
      const rect = ero_cv.getBoundingClientRect();
      const sx = ero_W / rect.width, sy = ero_H / rect.height;
      const c = Math.floor(((e.clientX - rect.left) * sx - ero_PAD) / ero_CELL);
      const r = Math.floor(((e.clientY - rect.top) * sy - ero_PAD) / ero_CELL);
      if(c >= 0 && c < ero_COLS && r >= 0 && r < ero_ROWS){
        ero_cx = c; ero_cy = r; 
        ero_draw();
      }
    });
    
    root.querySelector('#sim-04-erosao_slX').addEventListener('input', function(){ ero_cx = +this.value; ero_draw(); });
    root.querySelector('#sim-04-erosao_slY').addEventListener('input', function(){ ero_cy = +this.value; ero_draw(); });
    root.querySelector('#sim-04-erosao_rstBtn').addEventListener('click', function(){ ero_cx = 4; ero_cy = 4; ero_draw(); });
    
    ero_draw();
  }

  function tryInitSim04Ero(){
    var root = document.getElementById('sim-04-erosao');
    if (root) initSim04Ero(root); else setTimeout(tryInitSim04Ero, 200);
  }
  tryInitSim04Ero();
})();
</script>
</div>
""")

**Figure 4.5:** Simulateur : Érosion Morphologique (A ⊖ B_L)


<figure id="fig-04-sim-04-erosao">
  <img src="imagens/fig-04-sim-04-erosao.png" alt=" Simulateur : Érosion Morphologique (A ⊖ B_L) " style="max-width:80%" />
  <figcaption><strong>Figure 4.5:</strong>  Simulateur : Érosion Morphologique (A ⊖ B_L) </figcaption>
</figure>

### 4.3.2 Ouverture et Fermeture

En combinant l'érosion et la dilatation, on obtient deux opérateurs d'une grande utilité pratique : l'**ouverture** et la **fermeture**, définis par les Équations [Équation 4.5](#eq-04-abertura) et [Équation 4.6](#eq-04-fechamento).. Leurs principaux effets sont résumés dans la [Tableau 4.2](#tbl-04-open-close)..

**Ouverture** (*opening*) — érosion suivie d'une dilatation par le même $B$ :

<a id="eq-04-abertura"></a>
$$
A \circ B = (A \ominus B) \oplus B \tag{4.5}
$$


**Fermeture** (*closing*) — dilatation suivie d'une érosion par le même $B$ :

<a id="eq-04-fechamento"></a>
$$
A \bullet B = (A \oplus B) \ominus B \tag{4.6}
$$


En pratique, OpenCV applique le même élément structurant lors des deux étapes. Pour les éléments structurants symétriques (les plus courants), cette implémentation coïncide avec la définition mathématique présentée ci-dessus.

<a id="tbl-04-open-close"></a>

**Tabela 4.2:** Propriétés de l'ouverture et de la fermeture.

| Opérateur | Séquence | Effet principal |
|:--------:|:----------|:-----------------|
| Ouverture $A \circ B$ | érosion → dilatation | Supprime les structures incapables de contenir l'élément structurant ; lisse les contours externes |
| Fermeture $A \bullet B$ | dilatation → érosion | Comble les trous plus petits que $B$ ; lisse les contours internes |


**Propriété importante :** les deux sont **idempotents**. Par exemple,

$$
(A \circ B) \circ B = A \circ B,
$$

c'est-à-dire qu'après la première application, de nouvelles applications du même opérateur ne modifient plus le résultat.

#### 4.3.2.1 Implémentation de l'ouverture et de la fermeture

Contrairement à l'érosion et à la dilatation, l'ouverture et la fermeture n'introduisent pas
de nouveaux mécanismes computationnels. Toutes deux sont obtenues par la composition séquentielle
des opérateurs primitifs déjà présentés :

```python
def open0(f, B):
    return mm.dil0(mm.ero0(f, B), B)

def close0(f, B):
    return mm.ero0(mm.dil0(f, B), B)
```

La fonction `mm.open` délègue l'opération à `cv2.morphologyEx(..., MORPH_OPEN, B)`,
tandis que `mm.close` utilise `cv2.morphologyEx(..., MORPH_CLOSE, B)`, produisant
le même résultat de manière plus efficace.

L'ouverture hérite de l'érosion la capacité de supprimer les structures plus petites que
l'élément structurant et de la dilatation la restauration partielle des régions
préservées. La fermeture réalise le processus inverse : elle étend d'abord les
objets puis restaure leurs dimensions originales, comblant les lacunes et les
trous plus petits que l'élément structurant.

#### 4.3.2.2 Filtre Séquentiel Alterné

En pratique, l'ouverture et la fermeture sont souvent appliquées en séquence pour éliminer simultanément le bruit externe et remplir les trous internes. La fonction `mm.asf` (*filtre séquentiel alterné*) généralise cette stratégie en appliquant des ouvertures et des fermetures en alternance avec des éléments structurants progressivement plus grands. Les séquences disponibles sont présentées dans la [Tableau 4.3](#tbl-04-asf).

<a id="tbl-04-asf"></a>

**Tabela 4.3:** Séquences du filtre séquentiel alterné `mm.asf`.

| Séquence | Ordre                             | Utilisation typique                                          |
| :------: | :-------------------------------- | :----------------------------------------------------------- |
|   `'OC'` | ouverture → fermeture             | élimine le bruit externe avant de remplir les petits trous    |
|   `'CO'` | fermeture → ouverture             | remplit les petits trous avant d'éliminer le bruit externe    |
|  `'OCO'` | ouverture → fermeture → ouverture | accentue l'élimination du bruit externe                       |
|  `'COC'` | fermeture → ouverture → fermeture | accentue le remplissage des trous et des lacunes              |


Le paramètre `n` contrôle le nombre d'échelles utilisées par le filtre. À chaque itération $i$, l'élément structurant est agrandi par somme de Minkowski (`mm.sesum(b, i)`), produisant une séquence de filtres morphologiques de plus en plus englobants. Contrairement à une seule ouverture ou fermeture avec un grand élément structurant, l'ASF effectue un lissage progressif à plusieurs échelles, préservant mieux la géométrie des objets pertinents tout en éliminant les structures plus petites. La [Figure 4.6](#fig-04-open-close) présente un exemple d'application de l'ouverture, de la fermeture et de l'ASF sur l'image des pièces de monnaie.

In [10]:
img_bin    = mm.threshold(img_clahe)
B_disk     = mm.sedisk(19)

img_open  = mm.open(img_bin, B_disk)             # érosion → dilatation
img_close = mm.close(img_bin, B_disk)            # dilatation → érosion
img_oc    = mm.close(img_open, B_disk)           # ouverture suivie de fermeture
img_asf   = mm.asf(img_bin, 'OC', mm.sedisk(3), n=7)  
# ASF : disque de base 3×3, croît à chaque itération

mm.show(
    [img_bin, img_open, img_close, img_oc, img_asf],
    titles=["Binarisation d’Otsu (CLAHE)", "Ouverture (A∘B)", "Fermeture (A∙B)",
            "Ouverture→Fermeture", "ASF-OC (n=7)"],
    cols=5, figsize=(15, 12)
)

<Figure size 2250x1800 with 5 Axes>

**Figure 4.6:** Ouverture, fermeture, composition et filtre séquentiel alterné appliqués à la binarisation d’Otsu des pièces avec CLAHE. Élément structurant : disque 13×13.


### 4.3.3 Opérateurs Géodésiques

Les opérateurs géodésiques introduisent une contrainte supplémentaire aux opérateurs morphologiques classiques au moyen d'une image de contrôle appelée **masque** $g$. Au lieu de permettre à l'érosion ou à la dilatation de se propager librement à travers l'image, le résultat de chaque itération est limité point par point par les valeurs du masque, restreignant l'évolution de l'opération aux régions autorisées.

#### 4.3.3.1 Dilatation Géodésique

La **dilatation géodésique** d’une image marqueur $f$ sous une image masque $g$, à l’aide d’un élément structurant plat $b$, est définie par :

<a id="eq-04-cdil"></a>
$$
f \oplus_g b = (f \oplus b) \wedge g, \tag{4.7}
$$


où $\wedge$ représente le minimum point par point.

En d’autres termes, on effectue d’abord une dilatation conventionnelle sur le marqueur, puis le résultat est restreint par le masque $g$. Ainsi, la propagation ne peut jamais dépasser les régions autorisées par le masque.

La formulation classique de la dilatation géodésique suppose que le marqueur est contenu dans le masque, c’est-à-dire $f \le g$, garantissant que l’évolution de l’opération reste toujours limitée par le masque.

#### 4.3.3.2 Implémentation de la dilatation géodésique

La fonction `mm.cdil` implémente directement cet opérateur et permet d'exécuter plusieurs itérations consécutives :

```python
def cdil(f, g, b=np.zeros((3,3),dtype='uint8'), n=1):
    """Dilatação geodésica do marcador f sob a máscara g."""
    y = f.copy()
    for _ in range(n):
        y = np.minimum(cv2.dilate(y, b), g)
    return y
```

L'instruction `np.minimum(cv2.dilate(y, b), g)` implémente exactement la définition mathématique de la dilatation géodésique, c'est-à-dire $(y \oplus b)\wedge g$.

Lorsque $n=1$, la fonction exécute une seule dilatation géodésique. Pour $n>1$, le résultat de chaque étape devient le marqueur de l'étape suivante, produisant une propagation progressive contrôlée par le masque.

Le simulateur interactif [Figure 4.7](#fig-04-sim-04-cdil) permet de suivre, étape par étape, la propagation du marqueur $f$ le long des couloirs du labyrinthe. À chaque itération de `mm.cdil`, le front de dilatation avance vers les cellules voisines libres — celles où $g = 1$ —, tandis que les murs ($g = 0$) restent infranchissables. Le nombre de pas nécessaires pour que le marqueur atteigne la sortie correspond exactement à la longueur géodésique du chemin le plus court à l'intérieur du masque, mettant en évidence la connexion directe entre la dilatation géodésique itérée et la notion de distance dans les graphes.

In [11]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-cdil" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulateur : Dilatation géodésique dans le labyrinthe</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">δ_g^(n)(f)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    
    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:14px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#2c2c2a;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Mur</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#eafaf1;border:1px solid #a3e4d7;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Chemin libre (g)</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#27ae60;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Marqueur f</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#a3e4d7;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Propagation</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#fef9e7;border:1px solid #f8c471;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Sortie</span></div>
    </div>

    <!-- Canvas Centralizado -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <canvas id="sim-04-cdil_Canvas" width="435" height="435" style="width:100%;max-width:435px;display:block;border-radius:12px;border:1px solid #e4dcc8;background:#ffffff;"></canvas>
    </div>

    <!-- Controles e Informações do Passo -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;gap:6px;flex-wrap:wrap;">
        <button id="sim-04-cdil_btnReset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↺ Réinitialiser</button>
        <button id="sim-04-cdil_btnPrev" disabled style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">‹ Précédent</button>
        <button id="sim-04-cdil_btnNext" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">Suivant ›</button>
        <button id="sim-04-cdil_btnPlay" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">▶ Animer</button>
      </div>
      <div id="sim-04-cdil_info" style="font-size:11px;font-family:monospace;color:#26241d;flex:1;min-width:180px;text-align:right;">Étape 0 — marqueur initial f (entrée)</div>
    </div>

  </div>
</div>

<script>
(function() {
  function initSim04Cdil(root){
    if (!root || root.dataset.sim04CdilInit) return;
    root.dataset.sim04CdilInit = "1";

    const G = [
      [0,0,0,0,0,0,0,0,0,0,0,0],
      [0,1,1,1,0,1,1,1,1,1,1,0],
      [0,0,0,1,0,1,0,0,0,0,1,0],
      [0,1,0,1,0,1,0,1,1,0,1,0],
      [0,1,0,1,1,1,0,1,0,0,1,0],
      [0,1,0,0,0,0,0,1,0,1,1,0],
      [0,1,1,1,1,1,1,1,0,1,0,0],
      [0,0,0,0,0,0,1,0,0,1,0,0],
      [0,1,1,1,1,0,1,1,1,1,0,0],
      [0,1,0,0,1,0,0,0,0,1,0,0],
      [0,1,1,1,1,1,1,1,1,1,0,0],
      [0,0,0,0,0,0,0,0,0,0,0,0]
    ];
    const ROWS = G.length, COLS = G[0].length;
    const START = [1,1];
    const EXIT  = [10,9];

    function dilate(prev, mask) {
      const next = prev.map(r => [...r]);
      const dirs = [[-1,0],[1,0],[0,-1],[0,1]];
      for (let r=0; r<ROWS; r++)
        for (let c=0; c<COLS; c++)
          if (prev[r][c]) {
            for (const [dr,dc] of dirs) {
              const nr=r+dr, nc=c+dc;
              if (nr>=0&&nr<ROWS&&nc>=0&&nc<COLS&&mask[nr][nc])
                next[nr][nc]=1;
            }
          }
      return next;
    }

    function emptyGrid() { return Array.from({length:ROWS},()=>new Array(COLS).fill(0)); }

    let steps = [];
    function buildSteps() {
      steps = [];
      let cur = emptyGrid();
      cur[START[0]][START[1]] = 1;
      steps.push(cur.map(r=>[...r]));
      let prev = null;
      while (JSON.stringify(cur) !== JSON.stringify(prev)) {
        prev = cur.map(r=>[...r]);
        cur = dilate(cur, G);
        steps.push(cur.map(r=>[...r]));
        if (steps.length > 80) break;
      }
    }

    buildSteps();
    let idx = 0, playing = false, timer = null;

    const cv = root.querySelector('#sim-04-cdil_Canvas');
    const ctx = cv.getContext('2d');

    const C = {
      wall:    '#2c2c2a',
      free:    '#eafaf1',
      freeBd:  '#a3e4d7',
      marker:  '#27ae60',
      trail:   '#a3e4d7',
      exit:    '#fef9e7',
      exitBd:  '#f8c471',
      bg:      '#fafaf7',
      text:    '#5e5a4a',
      grid:    '#e4dcc8',
    };

    function draw() {
      const W = cv.width, H = cv.height;
      const cw = W / COLS, ch = H / ROWS;
      ctx.clearRect(0,0,W,H);
      ctx.fillStyle = C.bg;
      ctx.fillRect(0,0,W,H);

      const cur = steps[idx];
      const prev = idx > 0 ? steps[idx-1] : null;

      for (let r=0; r<ROWS; r++) {
        for (let c=0; c<COLS; c++) {
          const x = c*cw, y = r*ch;
          const isExit = r===EXIT[0]&&c===EXIT[1];

          if (!G[r][c]) {
            ctx.fillStyle = C.wall;
            ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
          } else {
            if (isExit) {
              ctx.fillStyle = C.exit;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
              ctx.strokeStyle = C.exitBd;
              ctx.lineWidth = 1.5;
              ctx.strokeRect(x+1.5,y+1.5,cw-3,ch-3);
            } else {
              ctx.fillStyle = C.free;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
              ctx.strokeStyle = C.freeBd;
              ctx.lineWidth = 0.5;
              ctx.strokeRect(x+0.5,y+0.5,cw-1,ch-1);
            }
            if (prev && prev[r][c]) {
              ctx.fillStyle = C.trail;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
            }
            if (cur[r][c]) {
              ctx.fillStyle = C.marker;
              ctx.fillRect(x+2,y+2,cw-4,ch-4);
            }
          }
        }
      }

      ctx.strokeStyle = C.grid;
      ctx.lineWidth = 0.5;
      for (let r=0; r<=ROWS; r++) { ctx.beginPath(); ctx.moveTo(0,r*ch); ctx.lineTo(W,r*ch); ctx.stroke(); }
      for (let c=0; c<=COLS; c++) { ctx.beginPath(); ctx.moveTo(c*cw,0); ctx.lineTo(c*cw,H); ctx.stroke(); }

      ctx.fillStyle = '#ffffff';
      ctx.font = 'bold 11px monospace';
      ctx.textAlign = 'center';
      ctx.textBaseline = 'middle';
      const [sr,sc] = START;
      ctx.fillText('E', (sc+0.5)*cw, (sr+0.5)*ch);
      const [er,ec] = EXIT;
      ctx.fillStyle = '#b9770e';
      ctx.fillText('S', (ec+0.5)*cw, (er+0.5)*ch);

      const reached = steps[idx][EXIT[0]][EXIT[1]];
      let msg = '';
      if (idx === 0) msg = 'Passo 0 — marcador inicial f (entrada)';
      else if (reached) msg = 'Passo ' + idx + ' — saída alcançada! 🎉';
      else msg = 'Passo ' + idx + ' — propagação geodésica';

      root.querySelector('#sim-04-cdil_info').textContent = msg;
      root.querySelector('#sim-04-cdil_btnPrev').disabled = idx === 0;
      root.querySelector('#sim-04-cdil_btnNext').disabled = idx === steps.length-1;
    }

    function step(d) {
      idx = Math.max(0, Math.min(steps.length-1, idx+d));
      draw();
    }

    function reset() {
      stopPlay();
      idx = 0;
      draw();
    }

    function togglePlay() {
      playing ? stopPlay() : startPlay();
    }

    function startPlay() {
      playing = true;
      root.querySelector('#sim-04-cdil_btnPlay').textContent = '⏸ Pausar';
      timer = setInterval(() => {
        if (idx >= steps.length-1) { stopPlay(); return; }
        idx++;
        draw();
      }, 350);
    }

    function stopPlay() {
      playing = false;
      clearInterval(timer);
      root.querySelector('#sim-04-cdil_btnPlay').textContent = '▶ Animar';
    }

    root.querySelector('#sim-04-cdil_btnReset').addEventListener('click', reset);
    root.querySelector('#sim-04-cdil_btnPrev').addEventListener('click', () => step(-1));
    root.querySelector('#sim-04-cdil_btnNext').addEventListener('click', () => step(1));
    root.querySelector('#sim-04-cdil_btnPlay').addEventListener('click', togglePlay);

    draw();
  }

  function tryInitSim04Cdil(){
    var root = document.getElementById('sim-04-cdil');
    if (root) initSim04Cdil(root); else setTimeout(tryInitSim04Cdil, 200);
  }
  tryInitSim04Cdil();
})();
</script>
""")

**Figure 4.7:** Simulateur interactif de dilatation géodésique : le marqueur f (vert) se propage pas à pas le long des chemins libres du masque g, sans traverser les murs.


<figure id="fig-04-sim-04-cdil">
  <img src="imagens/fig-04-sim-04-cdil.png" alt=" Simulateur interactif de dilatation géodésique : le marqueur f (vert) se propage pas à pas le long des chemins libres du masque g, sans traverser les murs. " style="max-width:80%" />
  <figcaption><strong>Figure 4.7:</strong>  Simulateur interactif de dilatation géodésique : le marqueur f (vert) se propage pas à pas le long des chemins libres du masque g, sans traverser les murs. </figcaption>
</figure>

#### 4.3.3.3 Érosion géodésique

De manière duale, l'**érosion géodésique** d'une image marqueur $f$ sous une image masque $g$ est définie par :

<a id="eq-04-cero"></a>
$$
f \ominus_g b = (f \ominus b) \vee g, \tag{4.8}
$$


où $\vee$ représente l'opérateur de maximum point à point.

Dans ce cas, l'érosion conventionnelle du marqueur est suivie d'une contrainte inférieure imposée par le masque. Ainsi, aucun pixel du résultat ne peut prendre une valeur inférieure au pixel correspondant du masque.

La formulation classique de l'érosion géodésique présuppose la condition duale

$$
f \ge g,
$$

de sorte que le masque agisse comme borne inférieure pendant tout le processus.

#### 4.3.3.4 Implémentation de l'érosion géodésique

La fonction statique `mm.cero` matérialise cet opérateur :

```python
staticmethod
def cero(f, g, b=np.zeros((3,3),dtype='uint8'), n=1):
    """Érosion géodésique du marqueur f sous le masque g."""
    y = f.copy()
    for _ in range(n):
        y = np.maximum(cv2.erode(y, b), g)
    return y
```

L'instruction `np.maximum(cv2.erode(y, b), g)` implémente directement l'expression $(y \ominus b)\vee g$.

Tout comme pour la dilatation géodésique, le paramètre $n$ définit combien d'érosions géodésiques successives seront calculées avant de retourner l'image finale.

> ### 📝 Relation avec la reconstruction morphologique
>
> La reconstruction morphologique présentée dans la section suivante est obtenue par l'application itérative de la dilatation géodésique (`mm.cdil`) jusqu'à ce qu'un point fixe soit atteint, c'est-à-dire jusqu'à ce qu'aucune cellule ne change de valeur entre deux itérations consécutives. En d'autres termes, la reconstruction consiste en une séquence de dilatations géodésiques successives qui se propagent à l'intérieur du masque jusqu'à ce qu'il n'y ait plus de modifications.
>
> De manière duale, il est également possible de définir des reconstructions basées sur l'érosion géodésique au moyen d'applications successives de `mm.cero`.

#### 4.3.3.5 Exemple : propagation dans un labyrinthe via dualité

La [Figure 4.8](#fig-04-cero-labirinto) illustre la résolution du problème de connectivité d’un labyrinthe en utilisant la dualité morphologique au moyen des fonctions `mm.cero` et `mm.suprec`.

Au lieu de propager un marqueur à travers les couloirs libres par des dilatations géodésiques, le problème est formulé dans le domaine complémentaire. Initialement, le masque original est inversé,

$$
g = 1 - g_{orig},
$$

ce qui fait que les murs prennent la valeur 1 et les couloirs la valeur 0. De manière analogue, le marqueur est construit dans ce même domaine complémentaire, contenant une unique valeur 0 à la position de l’entrée du labyrinthe et la valeur 1 sur les autres pixels.

En utilisant l’élément structurant en croix (`mm.secross()`), l’érosion géodésique agit sur le marqueur complémenté. À chaque itération de `mm.cero`, la région connectée au marqueur initial subit des érosions successives, tandis que le masque impose une borne inférieure qui empêche la propagation à travers les murs du labyrinthe.

Les images intermédiaires montrent des états de l’évolution après différents nombres d’itérations (`n=5`, `n=12`, et `n=22`). Le résultat final est obtenu par la reconstruction géodésique par érosion (`mm.suprec`), qui applique des érosions géodésiques successives jusqu’à atteindre un point fixe, c’est-à-dire une situation où aucune modification supplémentaire ne se produit entre deux itérations consécutives.

Dans le domaine complémentaire, la région reconstruite correspond exactement à l’ensemble des couloirs connectés à l’entrée du labyrinthe. Ainsi, la connectivité entre l’entrée et la sortie peut être déterminée directement à partir de l’image reconstruite.

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 1. Masque original g (0 = Mur, 1 = Couloir)
g_orig = np.array([
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 1, 1, 1],
    [0, 0, 0, 0, 0, 1, 0, 1, 0, 1],
    [0, 1, 1, 1, 0, 1, 1, 1, 0, 1],
    [0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0]], dtype=np.uint8)

# Inversion globale au début (Dualité Morphologique)
g = 1 - g_orig  # Maintenant : 1 = Mur, 0 = Couloir
f = np.ones((10, 10), dtype=np.uint8)
f[0, 1] = 0     # Graine injectée comme 0 dans le couloir

# Élément structurant en croix
B_cruz = mm.secross()

# Traitement direct via mm.cero et mm.suprec dans le domaine inversé
passo_5    = mm.cero(f, g, B_cruz, n=5)
passo_12   = mm.cero(f, g, B_cruz, n=12)
passo_22   = mm.cero(f, g, B_cruz, n=22)
ponto_fixo = mm.suprec(f, g, B_cruz)

# --- CONFIGURATION DE L'AFFICHAGE GRAPHIQUE ---

titles = [
    "Máscara (~g)", "Marcador (~f)", 
    "Avanço (n=5)", "Avanço (n=12)", 
    "Avanço (n=22)", "~mm.suprec"
]
images = [g, f, passo_5, passo_12, passo_22, ponto_fixo]

fig, axes = plt.subplots(1, 6, figsize=(16, 4), facecolor='#fcfcfc')

# Carte de couleurs adaptée au domaine complémentaire :
# Dans le domaine inversé : 1 = Mur (Bleu Foncé)
# Où g == 0 et image == 1 = Couloir Libre (Gris Clair)
# Où g == 0 et image == 0 = Onde Géodésique Active (Or)
cmap_pipeline = mcolors.ListedColormap(['#ffc13b', '#e0e0e0', '#1e3d59'])

for i, ax in enumerate(axes):
    if i == 0 or i == 1:
        # Pour les conditions initiales (~g et ~f)
        cmap_init = mcolors.ListedColormap(['#e0e0e0', '#1e3d59'])
        ax.imshow(images[i], cmap=cmap_init, vmin=0, vmax=1)
    else:
        # Rendu basé sur les états complémentaires
        render_step = np.zeros_like(g, dtype=np.uint8)
        render_step[g == 1] = 2           # Mur (1 original de la carte de couleurs)
        render_step[g == 0] = 1           # Couloir par défaut
        render_step[images[i] == 0] = 0   # Onde active (0 original de la carte de couleurs)
        ax.imshow(render_step, cmap=cmap_pipeline, vmin=0, vmax=2)
    
    ax.set_title(titles[i], fontsize=10, fontweight='bold', color='#1e3d59', pad=12)
    ax.set_xticks(np.arange(-0.5, 10, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 10, 1), minor=True)
    ax.grid(which='minor', color='#ffffff', linestyle='-', linewidth=1)
    ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
    
    # Indicateurs graphiques d'Entrée et de Sortie
    ax.plot(1, 0, marker='v', color='#2ecc71', markersize=7, markeredgewidth=1.5)
    ax.plot(8, 9, marker='o', color='#e74c3c', markersize=6, fillstyle='none', markeredgewidth=2)

plt.tight_layout()
plt.show()

<Figure size 4800x1200 with 6 Axes>

**Figure 4.8:** Résolution de labyrinthe via Opérations Complémentaires : en inversant le masque et le marqueur au début du *pipeline*, le flux est résolu directement dans le domaine complémentaire via mm.cero et mm.suprec, éliminant les ré-inversions redondantes.


### 4.3.4 Reconstruction morphologique

La **reconstruction morphologique** propage une image **marqueur** $f$ à l’intérieur d’une image **masque** $g$, en garantissant que le résultat ne dépasse jamais les valeurs d’intensité imposées par le masque. L’opérateur fondamental qui rend possible cette propagation contenue est la **dilatation géodésique**, définie par la [Équation 4.7](#eq-04-cdil).

La reconstruction est obtenue par l’application itérative de cette dilatation conditionnée. Initialement, le marqueur est limité par le masque pour établir l’état initial :

<a id="eq-04-infrec"></a>
$$
X^{(0)} = f \wedge g,
$$

et les itérations suivantes sont définies de manière récursive par :

$$
X^{(k)} = (X^{(k-1)} \oplus b) \wedge g.
$$

La séquence croît de manière monotone jusqu’à atteindre un point fixe, produisant la **reconstruction morphologique par dilatation** (également connue dans la littérature sous le nom d’*inf-reconstruction*) :

$$
R_g^\delta(f) = \lim_{k\to\infty} X^{(k)} = X^{(k)} \quad \text{quand} \quad X^{(k)} = X^{(k-1)}. \tag{4.9}
$$


La montée itérative est interrompue dès que la stabilité est atteinte, c’est-à-dire lorsque deux itérations consécutives produisent des matrices avec des valeurs absolument identiques.

#### 4.3.4.1 Implémentation de la reconstruction morphologique

La routine didactique `mm.infrec` implémente directement l'algorithme itératif de point fixe. Initialement, le marqueur effectif initial $X^{(0)}$ est déterminé par l'opération `np.minimum(f, g)`. Pour garantir que la boucle de vérification exécute la première itération sans déclencher de fausses convergences prématurées, la variable de contrôle de l'itération précédente (`y1`) est initialisée avec une valeur sentinelle hors du domaine des données (ou simplement avec une matrice qui force la première exécution).

```python
def infrec(f, g, b=np.zeros((3,3), dtype='uint8')):
    """Inf-reconstruction : dilate le marqueur (f ∧ g) jusqu'à convergence sous le masque g."""
    y = np.minimum(f, g)
    # Initialise y1 avec des valeurs impossibles pour forcer l'entrée dans la boucle
    y1 = np.full_like(f, 256, dtype=np.int16) 
    while not np.array_equal(y, y1):
        y1 = y.copy()
        # Applique la dilatation géodésique : (y ⊕ b) ∧ g
        y = np.minimum(cv2.dilate(y, b), g)
    return y.astype('uint8')
```

À l'intérieur de la boucle `while`, la variable `y` stocke l'estimation courante de la reconstruction $X^{(k)}$, tandis que `y1` conserve l'image de l'étape immédiatement précédente $X^{(k-1)}$. L'instruction de contrôle conditionnelle `np.minimum(cv2.dilate(y, b), g)` traduit fidèlement la dilatation géodésique théorique, où l'expansion morphologique conventionnelle pilotée par OpenCV est immédiatement « élaguée » et limitée par les barrières d'intensité du masque $g$. La boucle cesse lorsqu'aucune modification de pixel n'est enregistrée entre les étapes.

#### 4.3.4.2 Vantagens da reconstrução morfológica

A reconstrução morfológica é significativamente mais robusta que a abertura convencional porque é capaz de remover estruturas indesejadas sem distorcer ou alterar a morfologia dos objetos que devem ser preservados.

Enquanto a abertura clássica suaviza cantos, elimina pontas e deforma contornos devido à imposição geométrica rígida do elemento estruturante, a reconstrução geodésica utiliza a máscara para recuperar com exatidão os limites e formatos originais dos objetos que possuem conectividade com o marcador original.

Em termos intuitivos, o marcador atua como uma semente de contágio que se expande progressivamente, mas apenas trafegando pelas regiões permitidas pela máscara. Componentes que não possuem nenhuma intersecção com o marcador jamais serão reconstruídas (sendo eliminadas), enquanto as componentes tocadas pela semente expandem-se até restaurar integralmente a sua geometria original.

Esse comportamento discriminatório e conservativo é ilustrado na [Figure 4.9](#fig-04-reconstrucao-didatica), utilizando o elemento estruturante em cruz apresentado na [Figure 4.3](#fig-04-elemento-cruz)..

In [13]:
# 1. Définition du masque (g) : image 10x10 avec deux objets isolés
g = np.array([
    [0,0,0,0,0,0,0,0,0,0],
    [0,1,1,1,0,0,0,1,1,0],
    [0,1,1,1,1,0,0,1,1,0],
    [0,1,1,1,1,0,0,0,0,0],
    [0,1,1,1,1,0,0,0,0,0],
    [0,1,1,1,0,0,0,0,0,0],
    [0,0,1,1,1,0,0,1,0,0],
    [0,0,1,1,1,0,0,1,1,0],
    [0,0,0,1,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0]], dtype=np.uint8) * 255

B_cruz = mm.secross()

# 2. Génération du marqueur (f)
f = mm.ero(g, mm.sebox())

# 3. Itérations de la dilatation conditionnée automatisées en boucle
iteracoes = []
titulos_iteracoes = []
img_atual = f.copy()

for i in range(1, 6):
    img_add = img_atual*80
    img_atual = mm.cdil(img_atual, g, B_cruz)
    iteracoes.append(img_add+img_atual)
    titulos_iteracoes.append(f"Dilatação Cond. (n={i})")

# Reconstruction géodésique finale via la bibliothèque (point de contrôle)
img_reconstruida = mm.infrec(f, g, B_cruz)

# Vérification de convergence (l'étape 5 doit être identique à la reconstruction finale)
convergencia_ok = np.array_equal(img_reconstruida, iteracoes[-1])
print(f"✅ La reconstruction a atteint la stabilité à l'itération 5 : {convergencia_ok}")

# 4. Affichage du pipeline évolutif (assemblage dynamique des listes)
mm.show(
    [g, f] + iteracoes + [img_reconstruida],
    titles=["Máscara (g)", "Marcador (f)"] + titulos_iteracoes + ["Reconstrução Final R_g(f)"],
    cols=8,
    figsize=(18, 3)
)

✅ La reconstruction a atteint la stabilité à l'itération 5 : False


<Figure size 2700x450 with 8 Axes>

**Figure 4.9:** *Pipeline* de Reconstruction Morphologique par Dilatation Conditionnée : le masque contient deux objets, le marqueur isole uniquement le noyau de l


### 4.3.5 Preenchimento de Buracos e Remoção de Bordas

Dois operadores baseados em reconstrução morfológica completam o *pipeline* de limpeza binária. Suas características principais são resumidas na [Tableau 4.4](#tbl-04-clohole-edgeoff).

**Preenchimento de buracos** (`mm.clohole`) remove cavidades completamente cercadas pelo objeto, independentemente do tamanho, sem alterar os contornos externos. O procedimento atua no complemento da imagem, utilizando como marcador uma restrição da moldura (*frame*) ao fundo:

<a id="eq-04-clohole"></a>
$$
\text{clohole}(f) = \bigl(R_{f^c}^\delta(\text{frame}(f) \wedge f^c)\bigr)^c \tag{4.10}
$$


Em termos operacionais, primeiro reconstrói-se o fundo externo e, em seguida, aplica-se a complementação para recuperar os objetos com buracos preenchidos.

**Remoção de objetos de borda** (`mm.edgeoff`) elimina todos os objetos que tocam a borda da imagem, preservando apenas os componentes totalmente internos. O marcador é obtido pela interseção entre a moldura (*frame*) e os objetos da imagem:

<a id="eq-04-edgeoff"></a>
$$
\text{edgeoff}(f) = f \setminus R_f^\delta(\text{frame}(f) \wedge f) \tag{4.11}
$$


<a id="tbl-04-clohole-edgeoff"></a>

**Tabela 4.4:** Comparação entre os operadores *clohole* e *edgeoff*.

| Operador | Marcador | Máscara | Efeito |
|:---------|:---------|:--------|:-------|
| `mm.clohole` | *frame* restrito ao fundo ($f^c$) | $f^c$ | Preenche buracos internos |
| `mm.edgeoff` | *frame* restrito ao objeto ($f$) | $f$ | Remove objetos conectados à borda |


A evolução passo a passo dessas transformações geodésicas pode ser acompanhada nas figuras a seguir. A [Figure 4.10](#fig-04-clohole-didatico) ilustra o mecanismo de inundação controlada do operador `mm.clohole`, no qual a reconstrução ocorre a partir do fundo externo e impede a propagação para regiões internas não conectadas ao exterior, resultando no preenchimento consistente das cavidades internas. Em contrapartida, a [Figure 4.11](#fig-04-edgeoff-didatico) detalha a dinâmica do operador `mm.edgeoff`, em que apenas os componentes conectados à borda são reconstruídos e posteriormente removidos, preservando exclusivamente os objetos totalmente contidos no interior da imagem.

A conectividade da propagação geodésica é controlada pelo elemento estruturante: `mm.sebox()` (vizinhança de 8) inclui conexões diagonais, enquanto `mm.secross()` (vizinhança de 4) as exclui. Consequentemente, a escolha do elemento estruturante afeta quais componentes são alcançados pela reconstrução e, portanto, quais serão preservados ou removidos.

#### 4.3.5.1 Conformité avec l’implémentation

Les définitions ci-dessus sont directement alignées avec l’implémentation dans `morph.py`, reproduite ci-dessous :

```python
staticmethod
def clohole(f, b=np.ones((3,3),dtype='uint8')):
    # marqueur restreint au fond de l'image
    marqueur = mm.frame(f, border=1) & mm.neg(f)
    return mm.neg(mm.infrec(marqueur, mm.neg(f), b))

staticmethod
def edgeoff(f, b=np.ones((3,3),dtype='uint8')):
    # marqueur restreint aux objets de l'image
    marqueur = mm.frame(f, border=1) & f
    return mm.subm(f, mm.infrec(marqueur, f, b))
```

Ces implémentations rendent explicite le fait que les deux opérateurs sont des instances directes de la reconstruction morphologique par dilatation géodésique avec `mm.infrec`, ne différant que par le choix du marqueur et du masque : `clohole` agit sur le complément de l’image, tandis que `edgeoff` agit directement sur le domaine des objets.

In [14]:
# Fonction auxiliaire unifiée pour générer le pipeline itératif avec mise en évidence visuelle
def gerar_pipeline_reconstrucao(marcador, mascara, kernel, passos=4, fator=80):
    iteracoes, titulos = [], []
    img_atual = marcador.copy()
    for i in range(1, passos + 1):
        img_visual = img_atual * fator
        img_atual = mm.cdil(img_atual, mascara, kernel)
        iteracoes.append(img_visual + img_atual)
        titulos.append(f"Iter. (n={i})")
    return iteracoes, titulos

# Image binaire 10x10 avec 3 scénarios : objet avec trou, objet isolé et objet sur le bord
f = np.array([
    [0,0,0,0,0,0,0,0,0,0],
    [0,1,1,1,1,0,0,0,0,1],
    [0,1,0,0,1,0,0,1,0,1],
    [0,1,0,0,1,0,0,1,0,1],
    [0,1,1,1,1,0,0,1,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,1,1,1,0,1,1,1,0],
    [0,0,1,1,1,0,1,0,1,0],
    [0,0,1,1,1,0,1,1,1,0],
    [0,0,0,0,0,0,0,0,0,1]], dtype=np.uint8) * 255

B_cruz = mm.secross()
f_c = mm.neg(f)              # Utilise l'opérateur de négation natif de mm

# ── PIPELINE CLOHOLE ──────────────────────────────────────────────────────────
# Le marqueur du clohole théorique est le bord externe 
marcador_ch = mm.frame(f, border=1) 
iters_ch, tits_ch = gerar_pipeline_reconstrucao(marcador_ch, f_c, B_cruz, passos=5)

# Résultat final calculé en détail et validé avec la fonction native mm.clohole
img_clohole = mm.neg(mm.infrec(marcador_ch, f_c, B_cruz))
print(f"✅ Validation clohole : {np.array_equal(img_clohole, mm.clohole(f))}")

mm.show(
    [f, marcador_ch] + iters_ch + [img_clohole],
    titles=["f original", "Marcador (Borda)"] + tits_ch + ["clohole(f)"],
    cols=8, figsize=(18, 3), axis=True
)

✅ Validation clohole : True


<Figure size 2700x450 with 8 Axes>

**Figure 4.10:** *Pipeline* de remplissage de trous (*clohole*) : le marqueur est obtenu à partir du bord de l


In [15]:
# ── PIPELINE EDGEOFF ──────────────────────────────────────────────────────────
B_box = mm.sebox()
marcador_eo = marcador_ch & f
iters_eo, tits_eo = gerar_pipeline_reconstrucao(marcador_eo, f, B_box, passos=5)

# Résultat final par définition et validation native
img_edgeoff =  mm.edgeoff(f, B_box)
print(f"✅ Validation edgeoff : {np.array_equal(img_edgeoff, mm.edgeoff(f))}")

mm.show(
    [f, marcador_eo] + iters_eo + [img_edgeoff],
    titles=["f original", "Marcador (Borda)"] + tits_eo + ["edgeoff(f)"],
    cols=8, figsize=(18, 3)
)

✅ Validation edgeoff : True


<Figure size 2700x450 with 8 Axes>

**Figure 4.11:** *Pipeline* de suppression des structures de bord (*edgeoff*) : le marqueur capture les racines connectées aux extrémités de la matrice, la reconstruction délimite l


### 4.3.6 Pipeline de Limpeza Binária com CLAHE

Com base na análise anterior — na qual o CLAHE produziu o maior valor da variância interclasses ($\sigma_B^2 \approx 2{,}47 \times 10^3$), ver [Figure 4.2](#fig-04-otsu-comparacao-histogramas), e o operador `mm.clohole` mostrou-se eficaz no preenchimento das cavidades internas —, o *pipeline* final de segmentação, ilustrado na [Figure 4.12](#fig-04-pipeline-clahe), é estruturado pelo seguinte fluxo computacional:

$$
\text{gray}
\xrightarrow{\text{CLAHE}}
\xrightarrow{\text{Otsu}}
\xrightarrow{\text{open}}
\xrightarrow{\text{clohole}}
\xrightarrow{\text{open}}
\xrightarrow{\text{edgeoff}}
\text{segmentação}
$$

Após a etapa de `mm.clohole`, aplica-se uma segunda abertura morfológica com um elemento estruturante maior (`mm.sedisk(33)`, disco de diâmetro 33). Essa operação remove pequenas regiões residuais e artefatos que possam ter permanecido após a segmentação. Em particular, o preenchimento geodésico pode transformar pequenas cavidades isoladas em componentes conectados ao objeto, tornando conveniente uma etapa adicional de filtragem baseada em tamanho. O diâmetro foi escolhido de modo que as moedas permaneçam capazes de conter o elemento estruturante, enquanto componentes significativamente menores sejam eliminados.

Nesta imagem, nenhuma moeda está conectada à borda da matriz. Consequentemente, a aplicação de `mm.edgeoff` não altera o resultado obtido após a segunda abertura. Ainda assim, essa etapa é mantida no *pipeline* por robustez, pois em outras imagens podem existir objetos parcialmente visíveis ou conectados às bordas, que devem ser removidos antes da etapa de análise.

> ### 💡 Por que a abertura após o clohole?
>
> O operador `mm.clohole` preenche todas as cavidades fechadas presentes nos objetos segmentados. Em algumas situações, pequenas regiões indesejadas podem permanecer após essa etapa ou tornar-se conectadas aos objetos principais. A abertura morfológica subsequente remove componentes menores que o elemento estruturante, preservando as moedas devido ao seu tamanho significativamente maior.

In [16]:
# ── Pré-traitement : uniquement CLAHE
img_clahe0 = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img_coins_gray)

# ── Étape 1 : Binarisation Otsu
img_bin = mm.threshold(img_clahe0)

# ── Étape 2 : Ouverture — supprime les bruits blancs de fond
img_open = mm.open(img_bin, mm.sedisk(9))

# ── Étape 3 : clohole — ferme tous les trous internes
img_hole = mm.clohole(img_open)

# ── Étape 4 : Ouverture (grand noyau) — supprime les artefacts du clohole
img_limpo = mm.open(img_hole, mm.sedisk(33))

# ── Étape 5 : edgeoff — supprime les objets qui touchent le bord
img_final = mm.edgeoff(img_limpo, border=1)

mm.show(
    [img_clahe0,  img_bin,               img_open,
     img_hole,    img_limpo,             img_final],
    titles=["CLAHE",            "Otsu",                "Ouverture (r=9)",
            "clohole",          "Ouverture (r=33)",     "Final (edgeoff)"],
    cols=6, figsize=(18, 6)
)

<Figure size 2700x900 with 6 Axes>

**Figure 4.12:** Pipeline complet de segmentation avec CLAHE : Otsu → ouverture (r=9) → clohole → ouverture (r=33) → edgeoff.


### 4.3.7 Morphologie en niveaux de gris

Les opérateurs morphologiques s'étendent naturellement aux images en niveaux de gris. Dans cette formulation, l'érosion et la dilatation agissent directement sur les niveaux d'intensité de l'image. Pour des éléments structurants plats ($b \equiv 0$), l'érosion correspond au **minimum local** et la dilatation au **maximum local** dans le voisinage défini par l'élément structurant.

L'interprétation intuitive est simple : l'érosion **assombrit** les régions en remplaçant chaque pixel par la plus petite valeur présente dans son voisinage, tandis que la dilatation **éclaircit** les régions en utilisant la plus grande valeur disponible. La combinaison de ces opérateurs permet de construire des transformations capables de rehausser les contours, de supprimer les tendances d'éclairage et de mettre en évidence les structures locales.

Trois opérateurs dérivés sont particulièrement utiles :

**Gradient morphologique** — met en évidence les contours comme la différence entre la dilatation et l'érosion :

<a id="eq-04-gradiente-morf"></a>
$$
\text{grad}_B(f) = (f \oplus B) - (f \ominus B) \tag{4.12}
$$


***Top-hat*** — met en évidence les structures brillantes plus petites que l'élément structurant (différence entre l'image originale et son ouverture) :

<a id="eq-04-tophat"></a>
$$
\text{top-hat}_B(f) = f - (f \circ B) \tag{4.13}
$$


***Black-hat*** — met en évidence les structures sombres plus petites que l'élément structurant (différence entre la fermeture et l'image originale) :

<a id="eq-04-blackhat"></a>
$$
\text{black-hat}_B(f) = (f \bullet B) - f \tag{4.14}
$$


Le *Top-hat* extrait les détails brillants qui ne survivent pas à l'ouverture, tandis que le *Black-hat* révèle les détails sombres supprimés par la fermeture. Quant au gradient morphologique, il rehausse les transitions abruptes d'intensité, produisant une représentation similaire à celle d'un détecteur de contours.

Pour comprendre le mécanisme de ces opérateurs au niveau local, la [Figure 4.13](#fig-04-sim-04-operadores-av) présente un simulateur interactif de morphologie en tons de gris. Le simulateur permet de modifier librement l'élément structurant, de visualiser son déplacement sur l'image et de suivre simultanément le profil unidimensionnel des intensités. Ainsi, il devient possible d'observer directement comment l'érosion sélectionne les minima locaux, comment la dilatation sélectionne les maxima locaux et comment le gradient morphologique émerge de la différence entre ces deux opérateurs.

Le bouton situé dans le coin supérieur droit permet d'alterner entre la visualisation originale en tons de gris et une représentation en fausses couleurs (*colormap*) pour les trois types de gradients uniquement. La version colorée facilite la perception visuelle des variations d'intensité, rendant plus évidente l'action des opérateurs morphologiques sur les maxima, les minima et les transitions locales de l'image.

Les opérateurs présentés sont disponibles dans `morph.py` via les fonctions `mm.gradm`, `mm.tophat` et `mm.blackhat`.

In [17]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-operadores-av" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-operadores-av * { box-sizing: border-box; }
  #sim-04-operadores-av canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-04-operadores-av button { font-size: 11px; padding: 6px 10px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 4px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-operadores-av button:hover { background: #e8dfcf; }
  #sim-04-operadores-av button.gc_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  
  #sim-04-operadores-av button.gc_toggle_btn { background: #ebf4fd; border-color: #2980b9; color: #2980b9; font-weight: 700; }
  #sim-04-operadores-av button.gc_toggle_btn:hover { background: #d4e6fc; }
  
  .gc_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .gc_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 8px; padding: 8px; text-align: center; }
  .gc_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .gc_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  #sim04_seCanvas { cursor: pointer; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎛️ Simulateur avancé de morphologie mathématique</span>
  <button id="sim04_btn_toggle_view" class="gc_toggle_btn" onclick="sim04_toggleViewMode()">
    Visualisation: Gris réel
  </button>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Barra de Modos de Operação -->
  <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:14px; justify-content:center;">
    <button id="sim04_btn_orig" class="gc_active" onclick="sim04_setMode('orig')">Original f</button>
    <button id="sim04_btn_ero" onclick="sim04_setMode('ero')">Érosion</button>
    <button id="sim04_btn_dil" onclick="sim04_setMode('dil')">Dilatation</button>
    <button id="sim04_btn_open" onclick="sim04_setMode('open')">Ouverture (∘)</button>
    <button id="sim04_btn_close" onclick="sim04_setMode('close')">Fermeture (•)</button>
    <button id="sim04_btn_grad" onclick="sim04_setMode('grad')">Gradient</button>
    <button id="sim04_btn_tophat" onclick="sim04_setMode('tophat')">Top-hat</button>
    <button id="sim04_btn_bhat" onclick="sim04_setMode('bhat')">Black-hat</button>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(260px, 1fr)); gap:16px; align-items:start; margin-bottom:14px;">
    
    <!-- Canvas Principal -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
      <canvas id="sim04_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Déplacez la souris pour mettre à jour le profil 1D de la ligne correspondante
      </div>
    </div>

    <!-- Painel Lateral de Estatísticas, Descrição e Kernel B -->
    <div style="display:flex; flex-direction:column; gap:12px;">
      
      <!-- Estatísticas / Valores do Pixel -->
      <div style="display:grid; grid-template-columns:repeat(2, 1fr); gap:8px;">
        <div class="gc_stat_box"><div class="gc_stat_label">X (col)</div><div id="sim04_sX" class="gc_stat_value">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label">Y (ligne)</div><div id="sim04_sY" class="gc_stat_value">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label">f(x,y)</div><div id="sim04_sOrig" class="gc_stat_value" style="color:#27ae60;">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label" id="sim04_sLabel">valeur</div><div id="sim04_sVal" class="gc_stat_value" style="color:#2980b9;">—</div></div>
      </div>

      <!-- Descrição Teórica -->
      <div id="sim04_desc" class="gc_panel" style="font-size:11px; color:#5e5a4a; line-height:1.5; min-height:45px;"></div>

      <!-- Editor do Elemento Estruturante B -->
      <div class="gc_panel" style="text-align:center;">
        <div style="font-size:10.5px; font-weight:700; color:#5e5a4a; margin-bottom:8px; text-transform:uppercase; letter-spacing:.3px;">
          Élément B (Cliquer pour modifier)
        </div>
        <canvas id="sim04_seCanvas" width="114" height="114" style="margin:0 auto; display:block; border-radius:6px; border:1px solid #e4dcc8; background:#ffffff;"></canvas>
      </div>

      <!-- Fórmulas Auxiliares -->
      <div class="gc_panel" style="font-size:9.5px; font-family:monospace; color:#5e5a4a; line-height:1.5;">
        <strong style="color:#26241d;">Ouverture (f∘B):</strong> dil(ero(f))<br>
        <strong style="color:#26241d;">Fermeture (f•B):</strong> ero(dil(f))<br>
        <strong style="color:#26241d;">Gradient:</strong> dil(f) − ero(f)<br>
        <strong style="color:#26241d;">Top-hat:</strong> f − (f∘B)<br>
        <strong style="color:#26241d;">Black-hat:</strong> (f•B) − f
      </div>

    </div>

  </div>

  <!-- Perfil 1D na Parte Inferior -->
  <div style="background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
    <canvas id="sim04_profileCanvas" width="660" height="110" style="border-radius:8px; border:1px solid #e4dcc8; width:100%; height:auto; background:#ffffff;"></canvas>
    <div id="sim04_profileLabel" style="font-size:10.5px; color:#8a8371; margin-top:6px; font-family:monospace;">Profil 1D de la ligne: Aucun (passez la souris sur l'image)</div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04Advanced(root){
    if (!root || root.dataset.sim04AdvancedInit) return;
    root.dataset.sim04AdvancedInit = "1";

    const ROWS = 40, COLS = 40, CELL = 8, PAD = 6;
    const W = COLS * CELL + PAD * 2, H = ROWS * CELL + PAD * 2;

    const cv = root.querySelector('#sim04_Canvas');
    cv.width = W; cv.height = H;
    const ctx = cv.getContext('2d');

    let gc_viewMode = 'gray';
    let gc_mode = 'orig';
    let gc_hx = -1, gc_hy = 19;

    const f = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
    const cx1 = 13, cy1 = 14, r1 = 9, cx2 = 28, cy2 = 14, r2 = 7, cx3 = 20, cy3 = 28, r3 = 8;
    for (let r=0; r<ROWS; r++) {
      for (let c=0; c<COLS; c++) {
        let v = 30;
        const d1 = Math.hypot(c - cx1, r - cy1), d2 = Math.hypot(c - cx2, r - cy2), d3 = Math.hypot(c - cx3, r - cy3);
        if (d1 < r1) v = Math.round(200 - 90 * (d1 / r1));
        if (d2 < r2) v = Math.max(v, Math.round(180 - 70 * (d2 / r2)));
        if (d3 < r3) v = Math.max(v, Math.round(160 - 50 * (d3 / r3)));
        if (Math.hypot(c - 33, r - 8) < 2.2) v = 230;
        if (Math.hypot(c - 6, r - 32) < 2.2) v = Math.min(v, 20);
        f[r][c] = Math.max(0, Math.min(255, v));
      }
    }

    const B_SIZE = 5;
    const B = Array.from({length: B_SIZE}, () => new Array(B_SIZE).fill(0));
    const rad = 2;
    for (let r=0; r<B_SIZE; r++) {
      for (let c=0; c<B_SIZE; c++) {
        if (Math.hypot(c - rad, r - rad) <= rad) B[r][c] = 1;
      }
    }

    const MORPH_DATA = {};

    function recalculateMorphology() {
      const Boff = [];
      const center = Math.floor(B_SIZE / 2);
      for (let r=0; r<B_SIZE; r++) {
        for (let c=0; c<B_SIZE; c++) {
          if (B[r][c]) Boff.push([r - center, c - center]);
        }
      }
      if (Boff.length === 0) Boff.push([0, 0]);

      function e_op(img) {
        const res = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
        for (let r=0; r<ROWS; r++) {
          for (let c=0; c<COLS; c++) {
            let mn = 255;
            for (const [dr, dc] of Boff) {
              const vr = r + dr, vc = c + dc;
              const val = (vr >= 0 && vr < ROWS && vc >= 0 && vc < COLS) ? img[vr][vc] : 255;
              if (val < mn) mn = val;
            }
            res[r][c] = mn;
          }
        }
        return res;
      }

      function d_op(img) {
        const res = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
        for (let r=0; r<ROWS; r++) {
          for (let c=0; c<COLS; c++) {
            let mx = 0;
            for (const [dr, dc] of Boff) {
              const vr = r + dr, vc = c + dc;
              const val = (vr >= 0 && vr < ROWS && vc >= 0 && vc < COLS) ? img[vr][vc] : 0;
              if (val > mx) mx = val;
            }
            res[r][c] = mx;
          }
        }
        return res;
      }

      function diff(a, b) {
        return Array.from({length: ROWS}, (_, r) => Array.from({length: COLS}, (_, c) => Math.max(0, Math.min(255, a[r][c] - b[r][c]))));
      }

      MORPH_DATA.orig = f;
      MORPH_DATA.ero = e_op(f);
      MORPH_DATA.dil = d_op(f);
      MORPH_DATA.open = d_op(MORPH_DATA.ero);
      MORPH_DATA.close = e_op(MORPH_DATA.dil);
      MORPH_DATA.grad = diff(MORPH_DATA.dil, MORPH_DATA.ero);
      MORPH_DATA.tophat = diff(f, MORPH_DATA.open);
      MORPH_DATA.bhat = diff(MORPH_DATA.close, f);
    }

    const MODES_CONFIG = {
      orig: {label: 'f(x,y)', color: '#27ae60', desc: 'Imagem original f.'},
      ero: {label: 'ero(x,y)', color: '#c0392b', desc: 'Erosão: Encolhe estruturas claras de acordo com a geometria de B.'},
      dil: {label: 'dil(x,y)', color: '#2980b9', desc: 'Dilatação: Expande estruturas claras preenchendo falhas.'},
      open: {label: 'open(x,y)', color: '#8e44ad', desc: 'Abertura: Suaviza contornos, elimina pequenos ruídos e picos brilhantes isolados.'},
      close: {label: 'close(x,y)', color: '#d35400', desc: 'Fechamento: Preenche pequenos canais ou buracos escuros interiores.'},
      grad: {label: 'grad(x,y)', color: '#16a085', desc: 'Gradiente morfológico: Destaca as bordas físicas dos objetos.'},
      tophat: {label: 'th(x,y)', color: '#b9770e', desc: 'Top-hat: Isola elementos brilhantes menores que B.'},
      bhat: {label: 'bh(x,y)', color: '#2c3e50', desc: 'Black-hat: Isola fossas escuras ou vales menores que B.'}
    };

    window.sim04_toggleViewMode = function() {
      if (gc_viewMode === 'gray') {
        gc_viewMode = 'colormap';
        root.querySelector('#sim04_btn_toggle_view').innerHTML = 'Visualização: Falsa Cor';
      } else {
        gc_viewMode = 'gray';
        root.querySelector('#sim04_btn_toggle_view').innerHTML = 'Visualização: Cinza Real';
      }
      gc_draw();
    };

    window.sim04_setMode = function(m) {
      gc_mode = m;
      Object.keys(MODES_CONFIG).forEach(k => {
        const btn = root.querySelector('#sim04_btn_' + k);
        if (btn) btn.classList.toggle('gc_active', k === m);
      });
      const cfg = MODES_CONFIG[m];
      root.querySelector('#sim04_sLabel').textContent = cfg.label;
      root.querySelector('#sim04_sVal').style.color = cfg.color;
      root.querySelector('#sim04_desc').textContent = cfg.desc;
      gc_draw();
      gc_drawProfile();
    };

    function gc_imgToGray(data) {
      const id = ctx.createImageData(W, H);
      for (let r=0; r<ROWS; r++) {
        for (let c=0; c<COLS; c++) {
          const v = data[r][c];
          const x = PAD + c * CELL, y = PAD + r * CELL;
          let rc = v, gc = v, bc = v;

          if (gc_viewMode === 'colormap' && (gc_mode === 'grad' || gc_mode === 'tophat' || gc_mode === 'bhat')) {
            if (v > 0) {
              rc = Math.min(255, v * 7);       
              gc = Math.min(255, v * 3.5);     
              bc = Math.max(40, 255 - v * 4.5); 
            } else { rc = 250; gc = 250; bc = 247; }
          }

          for (let dy=0; dy<CELL; dy++) {
            for (let dx=0; dx<CELL; dx++) {
              const idx = 4 * ((y + dy) * W + (x + dx));
              id.data[idx] = rc; id.data[idx+1] = gc; id.data[idx+2] = bc; id.data[idx+3] = 255;
            }
          }
        }
      }
      return id;
    }

    function gc_draw() {
      const data = MORPH_DATA[gc_mode];
      const id = gc_imgToGray(data);
      ctx.putImageData(id, 0, 0);

      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 0.5;
      for (let c=0; c<=COLS; c++) { ctx.beginPath(); ctx.moveTo(PAD + c * CELL, PAD); ctx.lineTo(PAD + c * CELL, PAD + ROWS * CELL); ctx.stroke(); }
      for (let r=0; r<=ROWS; r++) { ctx.beginPath(); ctx.moveTo(PAD, PAD + r * CELL); ctx.lineTo(PAD + COLS * CELL, PAD + r * CELL); ctx.stroke(); }

      if (gc_hy >= 0 && gc_hy < ROWS) {
        ctx.strokeStyle = 'rgba(192, 57, 43, 0.4)'; ctx.lineWidth = 1;
        ctx.beginPath(); ctx.moveTo(PAD, PAD + gc_hy * CELL + CELL / 2); ctx.lineTo(PAD + COLS * CELL, PAD + gc_hy * CELL + CELL / 2); ctx.stroke();
      }

      if (gc_hx >= 0 && gc_hy >= 0) {
        ctx.strokeStyle = 'rgba(41, 128, 185, 0.9)'; ctx.lineWidth = 1.5;
        ctx.strokeRect(PAD + gc_hx * CELL + 0.5, PAD + gc_hy * CELL + 0.5, CELL - 1, CELL - 1);
      }

      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 1;
      ctx.strokeRect(PAD, PAD, COLS * CELL, ROWS * CELL);
    }

    const seCanvas = root.querySelector('#sim04_seCanvas');
    const seCtx = seCanvas.getContext('2d');
    const SE_CELL = 20, SE_PAD = 7;

    function gc_drawSE() {
      seCtx.clearRect(0, 0, 114, 114);
      const center = Math.floor(B_SIZE / 2);
      for (let r=0; r<B_SIZE; r++) {
        for (let c=0; c<B_SIZE; c++) {
          const x = SE_PAD + c * SE_CELL, y = SE_PAD + r * SE_CELL;
          seCtx.fillStyle = B[r][c] ? '#2980b9' : '#fafaf7';
          seCtx.fillRect(x, y, SE_CELL - 2, SE_CELL - 2);
          seCtx.strokeStyle = '#e4dcc8';
          seCtx.strokeRect(x, y, SE_CELL - 2, SE_CELL - 2);
          if (r === center && c === center) {
            seCtx.fillStyle = '#ffffff'; seCtx.font = 'bold 11px monospace';
            seCtx.textAlign = 'center'; seCtx.textBaseline = 'middle';
            seCtx.fillText('★', x + (SE_CELL - 2) / 2, y + (SE_CELL - 2) / 2);
          }
        }
      }
    }

    seCanvas.addEventListener('click', function(e) {
      const rect = seCanvas.getBoundingClientRect();
      const scaleX = seCanvas.width / rect.width;
      const scaleY = seCanvas.height / rect.height;
      const cx = (e.clientX - rect.left) * scaleX;
      const cy = (e.clientY - rect.top) * scaleY;
      const c = Math.floor((cx - SE_PAD) / SE_CELL);
      const r = Math.floor((cy - SE_PAD) / SE_CELL);
      if (c >= 0 && c < B_SIZE && r >= 0 && r < B_SIZE) {
        B[r][c] = B[r][c] ? 0 : 1;
        recalculateMorphology();
        gc_drawSE();
        gc_draw();
        gc_drawProfile();
      }
    });

    function gc_drawProfile() {
      const pc = root.querySelector('#sim04_profileCanvas');
      const pctx = pc.getContext('2d');
      pctx.clearRect(0, 0, 660, 110);
      
      if (gc_hy < 0 || gc_hy >= ROWS) return;

      root.querySelector('#sim04_profileLabel').innerHTML = `<span style='color:#c0392b; font-weight:bold;'>Perfil 1D da Linha ${gc_hy}</span> — Tracejado: Original f(x) | Cor da Aba: Operação Atual`;

      const dataOrig = MORPH_DATA.orig[gc_hy];
      const dataCurrent = MORPH_DATA[gc_mode][gc_hy];
      const currentColor = MODES_CONFIG[gc_mode].color;
      
      const stepX = 660 / (COLS - 1);
      
      function drawLine(arrayData, color, width, isDash = false) {
        pctx.strokeStyle = color; pctx.lineWidth = width;
        pctx.beginPath();
        if (isDash) pctx.setLineDash([4, 4]); else pctx.setLineDash([]);
        for (let c=0; c<COLS; c++) {
          const x = c * stepX;
          const y = 95 - (arrayData[c] / 255) * 85; 
          if (c === 0) pctx.moveTo(x, y); else pctx.lineTo(x, y);
        }
        pctx.stroke();
      }

      pctx.setLineDash([]);
      pctx.strokeStyle = '#e4dcc8'; pctx.lineWidth = 0.5;
      for (let h=0; h<=4; h++) {
        let yVal = 10 + h * 21.25;
        pctx.beginPath(); pctx.moveTo(0, yVal); pctx.lineTo(660, yVal); pctx.stroke();
      }

      if (gc_mode !== 'orig') {
        drawLine(dataOrig, 'rgba(39, 174, 96, 0.4)', 1.5, true);
      }
      drawLine(dataCurrent, currentColor, 2.5, false);

      pctx.setLineDash([]);
      pctx.fillStyle = '#8a8371'; pctx.font = '9px monospace';
      pctx.fillText('Intensidade (255)', 5, 10);
      pctx.fillText('Fundo (0)', 5, 104);
    }

    cv.addEventListener('mousemove', function(e) {
      const rect = cv.getBoundingClientRect();
      const sx = W / rect.width, sy = H / rect.height;
      const c = Math.floor(((e.clientX - rect.left) * sx - PAD) / CELL);
      const r = Math.floor(((e.clientY - rect.top) * sy - PAD) / CELL);
      if (c >= 0 && c < COLS && r >= 0 && r < ROWS) {
        gc_hx = c; gc_hy = r;
        root.querySelector('#sim04_sX').textContent = c;
        root.querySelector('#sim04_sY').textContent = r;
        root.querySelector('#sim04_sOrig').textContent = f[r][c];
        root.querySelector('#sim04_sVal').textContent = MORPH_DATA[gc_mode][r][c];
      }
      gc_draw();
      gc_drawProfile();
    });

    recalculateMorphology();
    sim04_setMode('orig');
    gc_drawSE();
  }

  function tryInitSim04Advanced(){
    var root = document.getElementById('sim-04-operadores-av');
    if (root) initSim04Advanced(root); else setTimeout(tryInitSim04Advanced, 200);
  }
  tryInitSim04Advanced();
})();
</script>
""")

**Figure 4.13:** Simulateur interactif avancé de morphologie avec élément structurant modifiable et profil 1D.


<figure id="fig-04-sim-04-operadores-av">
  <img src="imagens/fig-04-sim-04-operadores-av.png" alt=" Simulateur interactif avancé de morphologie avec élément structurant modifiable et profil 1D. " style="max-width:80%" />
  <figcaption><strong>Figure 4.13:</strong>  Simulateur interactif avancé de morphologie avec élément structurant modifiable et profil 1D. </figcaption>
</figure>

La [Figure 4.14](#fig-04-morf-gc-histogramas) illustre les effets de ces opérateurs sur l’image des pièces et leurs histogrammes respectifs. On observe que l’érosion déplace la distribution vers des intensités plus faibles, tandis que la dilatation la déplace vers des intensités plus élevées. Le gradient concentre les valeurs dans les régions de contour, et les opérateurs *Top-hat* et *Black-hat* produisent des histogrammes fortement concentrés à de faibles niveaux d’intensité, car seules de petites structures locales sont mises en évidence.

In [18]:
import io
import matplotlib.pyplot as plt
import numpy as np

def fig2img(fig):
    b = io.BytesIO(); fig.savefig(b, format='png', dpi=100); plt.close(fig); b.seek(0)
    return (plt.imread(b)[:, :, :3] * 255).astype(np.uint8)

def plot_hist(img, title):
    fig, ax = plt.subplots(figsize=(4, 3))
    h = mm.hist(img)
    # CORRECTION : range utilise len(h) dynamiquement pour correspondre au retour de la bibliothèque
    ax.bar(range(len(h)), h, color='steelblue', width=1, edgecolor='steelblue')
    ax.set(xlim=(0, 255)); plt.tight_layout()
    return fig2img(fig)

# 1. Traitement morphologique de base
B = mm.sedisk(19)
operadores = [
    ("Original", img_coins_gray),
    ("Erosão", mm.ero(img_coins_gray, B)),
    ("Dilatação", mm.dil(img_coins_gray, B)),
    ("Gradiente Morf.", mm.gradm(img_coins_gray, B)),
    ("Top-hat", mm.tophat(img_coins_gray, B)),
    ("Black-hat", mm.blackhat(img_coins_gray, B))
]

# 2. Assemblage dynamique de la paire : [Image, Histogramme]
imgs, titles = [], []
for nome, img in operadores:
    imgs += [img, plot_hist(img, f"Hist. {nome}")]
    titles += [nome, f"Hist. {nome}"]

# 3. Affichage final en grille de deux colonnes (Image | Histogramme)
mm.show(imgs, titles=titles, cols=4, figsize=(10, 8))

<Figure size 1500x1200 with 12 Axes>

**Figure 4.14:** Morphologie en niveaux de gris et leurs histogrammes respectifs. L


Les opérateurs morphologiques présentés précédemment seront désormais utilisés comme outils de raffinement et de génération de marqueurs pour des méthodes de segmentation plus avancées, présentées ci-après.

## 4.4 Segmentation d'Images : Fondements et Taxinomie

La **segmentation d'images** consiste à diviser l'image en régions associées à des objets ou à des structures d'intérêt. En TNI, elle représente la transition entre le traitement de bas niveau — comme le filtrage et le rehaussement — et des étapes d'analyse plus avancées, telles que l'extraction de caractéristiques, la reconnaissance et l'interprétation de la scène.

Formellement, l'objectif de la segmentation consiste à décomposer le domaine spatial complet d'une image, noté $\Omega$, en une partition de sous-ensembles $\{R_1, R_2, \ldots, R_n\}$ qui satisfait simultanément les critères de **complétude** et de **disjonction** :

<a id="eq-04-particao"></a>
$$
\bigcup_{i=1}^{n} R_i = \Omega, \qquad R_i \cap R_j = \emptyset \quad \forall\, i \neq j \tag{4.15}
$$


Outre les propriétés de complétude et de disjonction exprimées dans la [Équation 4.15](#eq-04-particao), chaque sous-région $R_i$ doit constituer un domaine **homogène** selon un prédicat de similarité défini sur des propriétés locales — intensité, couleur ou texture — et, simultanément, être **distincte** des régions adjacentes.

Les techniques de segmentation peuvent être organisées en différentes familles. Dans ce chapitre, l'accent sera mis sur les approches résumées dans la [Tableau 4.5](#tbl-04-segmentacao), fondées principalement sur des critères d'intensité, de connectivité et de proximité spatiale.

<a id="tbl-04-segmentacao"></a>

**Tabela 4.5:** Taxinomie simplifiée des principales approches de segmentation et d'affinement étudiées dans ce chapitre.

| Approche | Critère de Segmentation | Opérateurs de Référence |
| :--- | :--- | :--- |
| **Seuillage** | Partitionnement de l'espace des intensités | Critère d'Otsu, seuillage global et local |
| **Morphologie Mathématique** | Relations spatiales définies par des éléments/fonctions structurants | Érosion, dilatation, ouverture, fermeture et reconstruction |
| **Basée sur les Régions** | Homogénéité locale et connectivité spatiale | Étiquetage des composantes connexes, Transformée de Distance et *Watershed* |


Jusqu'à ce point, le développement pratique s'est concentré sur le **seuillage**, au moyen de la combinaison entre l'égalisation adaptative CLAHE et la méthode globale d'Otsu. Cette étape a été complétée par des opérateurs de **reconstruction morphologique** basés sur des dilatations géodésiques, implémentés par les fonctions `mm.infrec`, `mm.clohole` et `mm.edgeoff`, produisant un masque binaire propre et adapté à l'analyse.

Cependant, dans les scénarios où des objets distincts apparaissent connectés dans le masque binaire — que ce soit par contact physique, chevauchement partiel ou par de minces ponts de pixels produits par la segmentation —, le seuillage cesse d'être suffisant pour individualiser chaque objet. Dans ces cas, de multiples objets composent une seule composante connexe, ce qui complique les étapes ultérieures de mesure et d'interprétation.

Pour surmonter cette limitation, les prochaines sections introduisent trois outils complémentaires : l'**Étiquetage des Composantes Connexes**, la **Transformée de Distance** et l'algorithme de segmentation par ***Watershed*** basé sur des marqueurs. Ensemble, ces techniques permettent de séparer les objets adjacents, d'identifier les régions individuellement et d'extraire des descripteurs géométriques cohérents pour une analyse quantitative.

### 4.4.1 Étiquetage

L'**étiquetage des composantes connexes** (*connected component labeling*) est l'opérateur qui attribue un identifiant entier unique à chaque ensemble de pixels appartenant à la même composante connexe dans une image binaire.

> ### 📝 Définition formelle
>
> Étant donné une image binaire $f$ et une relation de connectivité définie par un élément structurant $B$ (typiquement la connectivité-4 ou la connectivité-8), l'algorithme d'étiquetage de la [Figure 4.15](#fig-04-sim-alg-rotulagem2) produit une image $g$ dans laquelle tous les pixels appartenant à la même composante connexe reçoivent le même étiquette entière positive, tandis que les pixels appartenant à des composantes distinctes reçoivent des étiquettes différentes.

La connectivité définit quels pixels sont considérés comme voisins directs d'un pixel $(x,y)$. Les définitions les plus utilisées sont :

* **Connectivité-4 :** ne considère que les quatre voisins orthogonaux (nord, sud, est et ouest).
* **Connectivité-8 :** considère les quatre voisins orthogonaux et les quatre voisins diagonaux, totalisant huit voisins.

Le choix de la connectivité influence directement la formation des composantes connexes et, par conséquent, le résultat de l'étiquetage, comme illustré dans la [Figure 4.17](#fig-04-rotulacao-didatico). Un exemple supplémentaire peut être exploré de manière interactive dans le simulateur présenté dans la [Figure 4.16](#fig-04-sim-04-rotulacao).

L'implémentation dans `morph.py` met à disposition deux versions de cet opérateur. La fonction `mm.label0` reproduit explicitement l'algorithme de *flood-fill* à l'aide d'une pile et permet de contrôler la connectivité via l'élément structurant adopté. Quant à `mm.label`, elle délègue l'opération à l'implémentation optimisée d'OpenCV (`cv2.connectedComponents`). Dans les deux cas, le résultat est une image étiquetée dans laquelle chaque composante connexe reçoit un identifiant entier distinct.

In [19]:
# @title { display-mode: "form" }
# Préfixe exclusif pour éviter les conflits avec d'autres cellules du notebook
PREFIX = "lbl2"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#EEEDFE;color:#534AB7}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Algoritmo de rotulação por flood-fill com pilha — painel interativo com HTML e SVG</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-stack-2" aria-hidden="true"></i></div>
    <div><div class="algo-title">Flood-fill com pilha</div><div class="algo-sub">Rotulagem de componentes conexas</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Criar imagem de saída <em>g</em>, inicializada com <code>zeros</code>, mesma dimensão de <em>f</em>.</div></div>
    <div class="step"><div class="step-num num-teal">2</div><div class="step-text">Inicializar contador de rótulos (cor) <code>cor ← 1</code>.</div></div>
    <div class="step"><div class="step-num num-purple">3</div><div class="step-text">Percorrer <em>f</em> em <strong>ordem raster</strong> (coordenadas <code>x</code> e <code>y</code>) até encontrar uma semente: pixel ativo (<code>f[x,y] ≠ 0</code>) ainda não rotulado (<code>g[x,y] = 0</code>).</div></div>
    <div class="step"><div class="step-num num-amber">4</div><div class="step-text">Inserir a semente encontrada na pilha <code>pilha ← [[x,y]]</code>.</div></div>
    <div class="step">
      <div class="step-num num-coral">5</div>
      <div class="step-text">Enquanto a pilha contiver coordenadas (<code>while pilha</code>):
        <div class="step-sub">
          <div class="step-sub-item">Desempilhar pixel atual: <code>i, j ← pilha.pop()</code> e atribuir o rótulo: <code>g[i,j] ← cor</code>.</div>
          <div class="step-sub-item">Buscar vizinhos usando o iterador <code>mm._viz(f,b,i,j)</code>. Se o vizinho for ativo no elemento estruturante (<code>bv ≠ 0</code>), ativo na imagem (<code>f[vy,vx] ≠ 0</code>) e não rotulado (<code>g[vy,vx] = 0</code>), empilhá-lo.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-blue">6</div><div class="step-text">Pilha vazia ⟹ Toda a componente conexa atual foi explorada e rotulada com sucesso.</div></div>
    <div class="step"><div class="step-num num-blue">7</div><div class="step-text">Incrementar o rótulo para a próxima componente: <code>cor ← cor + 1</code> e continuar a varredura raster.</div></div>
    <div class="note">A conectividade (4 ou 8 vizinhos) é definida unicamente pela matriz morfológica <code>b</code> passada como parâmetro, alterando os pixels retornados em <code>mm._viz</code>.</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Criar matriz de rótulos <code>g</code> preenchida com zeros (fundo). Definir rótulo inicial <code>cor ← 1</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-purple">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#534AB7">Varredura Raster</div>
      <div class="card-text">Percorrer a matriz bidimensional linha por linha, localizando pixels pertencentes ao objeto que ainda não possuem rótulo.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-amber">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#854F0B">Semente inicial</div>
      <div class="card-text">Ao achar um pixel válido, inicializar a estrutura LIFO de busca: <code>pilha = [[x, y]]</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Expansão por Flood-Fill</div>
      <div class="card-text">Enquanto houver elementos na pilha:</div>
      <div class="card-sub">
        <div class="card-sub-item">Extrair <code>(i, j)</code> via <code>pop()</code> e marcar <code>g[i, j] = cor</code>.</div>
        <div class="card-sub-item">Inspecionar vizinhança geométrica e adicionar novos candidatos à pilha.</div>
      </div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">05</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Próxima Componente</div>
      <div class="card-text">Pilha esvaziada ⟹ Incrementar indexador <code>cor ← cor + 1</code> para diferenciar o próximo objeto isolado.</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Alocação Espacial</div><div class="tl-desc"><code>g ← zeros_like(f)</code> e definição do primeiro identificador: <code>cor ← 1</code>.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div><div class="tl-title">Varredura Bidimensional</div><div class="tl-desc">Laços encadeados varrendo as dimensões <code>h</code> e <code>w</code> da imagem.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Descoberta de Objeto</div><div class="tl-desc">Filtro condicional localiza pixel ativo não indexado e cria a <code>pilha</code> semente.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">04–05</span></div>
    <div>
      <div class="tl-title">Preenchimento por Região (Flood-fill)</div>
      <div class="while-box">
        <div class="while-head">while pilha</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Remover último da pilha <code>(i,j)</code> e aplicar rótulo atual.</div>
          <div class="step-sub-item">Empilhar vizinhos conectados que atendam aos critérios morfológicos de <code>b</code>.</div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">06</span></div>
    <div><div class="tl-title">Fechamento do Objeto</div><div class="tl-desc">Pilha vazia determina o fim do isolamento daquela componente.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">07</span></div>
    <div><div class="tl-title">Atualização do Rótulo</div><div class="tl-desc">Incremento linear: <code>cor ← cor + 1</code>. A varredura raster continua do ponto onde parou.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>label0.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">flood-fill com pilha</span>
  </div>
  <div class="code-body">
<pre><span class="kw">def</span> <span class="fn">label0</span>(f, b=np.ones((<span class="num-lit">3</span>,<span class="num-lit">3</span>),dtype=<span class="st">'uint8'</span>)):
    <span class="cm">"""Rotulagem por flood-fill com pilha."""</span>
    h, w = f.shape
    g = np.zeros(f.shape, dtype=<span class="kw">int</span>)
    cor = <span class="num-lit">1</span>
    <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(h):
        <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(w):
            <span class="kw">if</span> f[x,y] <span class="kw">and not</span> g[x,y]:
                pilha = [[x,y]]
                <span class="kw">while</span> pilha:
                    i,j = pilha.pop(); g[i,j] = cor
                    <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f,b,i,j):
                        <span class="kw">if</span> bv <span class="kw">and</span> f[vy,vx] <span class="kw">and not</span> g[vy,vx]:
                            pilha.append([vy,vx])
                cor += <span class="num-lit">1</span>
    <span class="kw">return</span> g</pre>
  </div>
</div>
<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>g = np.zeros(f.shape, dtype=int)</code> — Inicializa a matriz de saída com zeros. Zeros representam o fundo invariável.</div></div>
  <div class="ann-line"><div class="ann-badge num-purple">2</div><div class="ann-text"><code>mm._viz(f, b, i, j)</code> — O iterador morfológico avalia a conectividade. Passando <code>B_cruz</code> a busca expande em 4-vizinhança; passando quadrado (<code>ones</code>) expande em 8-vizinhança.</div></div>
  <div class="ann-line"><div class="ann-badge num-amber">3</div><div class="ann-text"><code>pilha.pop()</code> — Remove o último par de coordenadas inserido, caracterizando um comportamento LIFO de busca em profundidade (DFS) para varrer o objeto de forma contígua.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">4</div><div class="ann-text"><code>cor += 1</code> — O incremento ocorre estritamente fora do laço <code>while</code>, garantindo que o mesmo número marque toda a extensão da componente concluída antes de passar para a próxima semente raster.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 580 820" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:580px;display:block;margin:0 auto">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#534AB7"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
  </defs>
  
  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>
  
  <rect x="160" y="80" width="240" height="48" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="99" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">inicializar saída</text>
  <text x="280" y="116" text-anchor="middle" font-size="12" fill="#085041">g ← zeros(f.shape);  cor ← 1</text>
  
  <rect x="160" y="150" width="240" height="48" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="169" text-anchor="middle" font-size="12" font-weight="500" fill="#534AB7">varredura raster</text>
  <text x="280" y="186" text-anchor="middle" font-size="12" fill="#3C3489">próximo pixel (x, y) em f</text>
  
  <polygon points="280,224 370,252 280,280 190,252" fill="#F1EFE8" stroke="#B4B2A9" stroke-width="1"/>
  <text x="280" y="248" text-anchor="middle" font-size="11.5" fill="#444441">imagem toda</text>
  <text x="280" y="264" text-anchor="middle" font-size="11.5" fill="#444441">varrida?</text>
  
  <ellipse cx="450" cy="252" rx="52" ry="22" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="450" y="257" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>
  
  <polygon points="280,304 380,334 280,364 180,334" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="329" text-anchor="middle" font-size="11.5" fill="#3C3489">f[x,y] ≠ 0  e</text>
  <text x="280" y="344" text-anchor="middle" font-size="11.5" fill="#3C3489">g[x,y] = 0?</text>
  
  <rect x="170" y="388" width="220" height="44" rx="6" fill="#FAEEDA" stroke="#FAC775" stroke-width="1"/>
  <text x="280" y="406" text-anchor="middle" font-size="12" font-weight="500" fill="#854F0B">inserir semente</text>
  <text x="280" y="422" text-anchor="middle" font-size="12" fill="#633806">pilha ← [[x, y]]</text>
  
  <polygon points="280,456 370,486 280,516 190,486" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="491" text-anchor="middle" font-size="11.5" fill="#712B13">pilha vazia?</text>
  
  <rect x="160" y="542" width="240" height="48" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="561" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">i, j ← pilha.pop()</text>
  <text x="280" y="578" text-anchor="middle" font-size="12" fill="#712B13">g[i, j] ← cor</text>
  
  <rect x="150" y="614" width="260" height="60" rx="6" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="634" text-anchor="middle" font-size="12" font-weight="500" fill="#185FA5">inspecionar vizinhos</text>
  <text x="280" y="650" text-anchor="middle" font-size="12" fill="#0C447C">se ativo e não rotulado</text>
  <text x="280" y="666" text-anchor="middle" font-size="12" fill="#0C447C">→ pilha.append([vy, vx])</text>
  
  <rect x="435" y="466" width="110" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="490" y="482" text-anchor="middle" font-size="12" font-weight="500" fill="#534AB7">cor ← cor + 1</text>
  <text x="490" y="496" text-anchor="middle" font-size="10.5" fill="#3C3489">próximo rótulo</text>

  <line x1="280" y1="58" x2="280" y2="80" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="128" x2="280" y2="150" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="198" x2="280" y2="224" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="370" y1="252" x2="398" y2="252" stroke="#3B6D11" stroke-width="1.2" marker-end="url(#arr-g)"/>
  <text x="384" y="245" text-anchor="middle" font-size="11" fill="#3B6D11">sim</text>
  
  <line x1="280" y1="280" x2="280" y2="304" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <path d="M 180,334 L 130,334 L 130,174 L 160,174" fill="none" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-b)"/>
  <text x="152" y="327" text-anchor="middle" font-size="11" fill="#534AB7">não</text>
  
  <line x1="280" y1="364" x2="280" y2="388" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <text x="292" y="378" font-size="11" fill="#534AB7">sim</text>
  
  <line x1="280" y1="432" x2="280" y2="456" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="280" y1="516" x2="280" y2="542" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="292" y="531" font-size="11" fill="#993C1D">não</text>
  
  <line x1="280" y1="590" x2="280" y2="614" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <path d="M 280,674 L 280,694 L 110,694 L 110,486 L 190,486" fill="none" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  
  <line x1="370" y1="486" x2="435" y2="486" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="402" y="479" text-anchor="middle" font-size="11" fill="#993C1D">sim</text>
  
  <path d="M 490,466 L 490,174 L 400,174" fill="none" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-b)"/>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figure 4.15:** Algorithme d


<figure id="fig-04-sim-alg-rotulagem2">
  <img src="imagens/fig-04-sim-alg-rotulagem2.png" alt=" Algorithme d'étiquetage par *flood-fill* avec pile. " style="max-width:80%" />
  <figcaption><strong>Figure 4.15:</strong>  Algorithme d'étiquetage par *flood-fill* avec pile. </figcaption>
</figure>

L'auxiliaire `_viz` itère sur la fenêtre structurante `b` centrée en $(i,j)$, ne générant que les voisins valides à l'intérieur des limites de l'image — la connectivité souhaitée est entièrement déterminée par la forme de `b` passée à l'algorithme.

**Exemple didactique — effet de la connectivité :**

In [20]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-rotulacao * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-rotulacao canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-rotulacao button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-rotulacao button:hover { background: #e8dfcf; }
  #sim-04-rotulacao button.rt_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-rotulacao .rt_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .rt_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .rt_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .rt_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .rt_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .rt_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🪙 Simulateur : Étiquetage des Composantes Connexes</span>
  <span class="rt_pill">remplissage par pile</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="rt_grid_stats">
    <div class="rt_stat_box">
      <div class="rt_stat_label">Pixels Actifs</div>
      <div id="rt_statPx" class="rt_stat_value" style="color:#2980b9;">0</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Composantes</div>
      <div id="rt_statCC" class="rt_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Connectivité</div>
      <div id="rt_statConn" class="rt_stat_value" style="color:#b9770e;">4</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Pas de Raster</div>
      <div id="rt_statStep" class="rt_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas e Legenda -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="rt_ccCanvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Cliquez pour activer/désactiver les pixels · Glissez pour peindre
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:8px; margin-top:10px; justify-content:center;" id="rt_legendBox"></div>
    </div>

    <!-- Painel de Controles Lateral -->
    <div style="flex:1; min-width:220px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Seletor de Conectividade -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Connectivité
        </div>
        <div style="display:flex; gap:6px;">
          <button id="rt_btn4" class="rt_active" onclick="window.rt_setConn(4)" style="flex:1; justify-content:center;">C-4</button>
          <button id="rt_btn8" onclick="window.rt_setConn(8)" style="flex:1; justify-content:center;">C-8</button>
        </div>
        <div id="rt_connDesc" style="font-size:9.5px; color:#8a8371; margin-top:6px; line-height:1.4;">
          4 voisins orthogonaux : N, S, E, O
        </div>
      </div>

      <!-- Seletor de Modo de Visualização -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualisation
        </div>
        <div style="display:flex; gap:6px;">
          <button id="rt_btnLabel" class="rt_active" onclick="window.rt_setMode('label')" style="flex:1; justify-content:center;">Étiquettes</button>
          <button id="rt_btnAnim" onclick="window.rt_setMode('anim')" style="flex:1; justify-content:center;">Animé</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="rt_animCtrl" class="rt_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Pas à Pas
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button onclick="window.rt_animStep(-1)" style="flex:1; justify-content:center;">◀</button>
          <button id="rt_btnPlay" onclick="window.rt_togglePlay()" style="flex:1; justify-content:center;">▶</button>
          <button onclick="window.rt_animStep(1)" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Vit.</label>
          <input type="range" id="rt_speedSlider" min="1" max="10" value="5" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Status da Animação -->
      <div id="rt_animStatus" class="rt_panel" style="display:none;">
        <div id="rt_animTitle" style="font-size:11px; font-weight:700; margin-bottom:3px; color:#26241d;">–</div>
        <div id="rt_animDesc" style="font-size:9.5px; color:#8a8371; line-height:1.4;">–</div>
      </div>

      <!-- Exemplos / Presets -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Exemples Prédéfinis
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button onclick="window.rt_loadPreset('diagonal')" style="justify-content:flex-start;">■ Diagonale (C-4 vs C-8)</button>
          <button onclick="window.rt_loadPreset('letters')" style="justify-content:flex-start;">A Lettres Séparées</button>
          <button onclick="window.rt_loadPreset('ring')" style="justify-content:flex-start;">◎ Anneau avec Trou</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button onclick="window.rt_clearGrid()" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Effacer la Grille
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04RotulacaoCC(root){
    if (!root || root.dataset.sim04RotulacaoCCInit) return;
    root.dataset.sim04RotulacaoCCInit = "1";

    const rt_COLS = 12, rt_ROWS = 12, rt_CELL = 22, rt_PAD = 10;
    const rt_W = rt_COLS * rt_CELL + rt_PAD * 2, rt_H = rt_ROWS * rt_CELL + rt_PAD * 2;
    const rt_cv = root.querySelector('#rt_ccCanvas');
    rt_cv.width = rt_W; 
    rt_cv.height = rt_H;
    const rt_ctx = rt_cv.getContext('2d');

    const rt_PALETTE = [
      ['#2980b9', '#ebf4fd', '#a9cce3', '#042c53'],
      ['#27ae60', '#eafaf1', '#a3e4d7', '#04342C'],
      ['#b9770e', '#fef5e7', '#f8c471', '#412402'],
      ['#c0392b', '#fdecea', '#f5b7b1', '#4A1B0C'],
      ['#8e44ad', '#f5eef8', '#d7bde2', '#4B1528'],
      ['#16a085', '#e8f8f5', '#a3e4d7', '#0e6251'],
      ['#d35400', '#fbeee6', '#f5cba7', '#7e5109'],
      ['#2c3e50', '#ebedef', '#bdc3c7', '#17202a']
    ];

    let rt_grid = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
    let rt_connectivity = 4;
    let rt_mode = 'label';
    let rt_painting = false;
    let rt_paintVal = 1;

    let rt_animSteps = [];
    let rt_animIdx = 0;
    let rt_playing = false;
    let rt_playTimer = null;

    function rt_neighbors(r, c, conn) {
      const n = [[r-1,c],[r+1,c],[r,c-1],[r,c+1]];
      if (conn === 8) n.push([r-1,c-1],[r-1,c+1],[r+1,c-1],[r+1,c+1]);
      return n.filter(([nr,nc]) => nr>=0 && nr<rt_ROWS && nc>=0 && nc<rt_COLS);
    }

    function rt_label(g, conn) {
      const lbl = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      let c = 1;
      for (let r=0; r<rt_ROWS; r++) {
        for (let cc=0; cc<rt_COLS; cc++) {
          if (g[r][cc] && !lbl[r][cc]) {
            const stack = [[r,cc]];
            while (stack.length) {
              const [i,j] = stack.pop();
              if (lbl[i][j]) continue;
              lbl[i][j] = c;
              for (const [ni,nj] of rt_neighbors(i, j, conn))
                if (g[ni][nj] && !lbl[ni][nj]) stack.push([ni,nj]);
            }
            c++;
          }
        }
      }
      return lbl;
    }

    function rt_buildAnimSteps(g, conn) {
      const steps = [];
      const lbl = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      let c = 1;
      for (let r=0; r<rt_ROWS; r++) {
        for (let cc=0; cc<rt_COLS; cc++) {
          steps.push({type:'raster', r, cc, lbl:lbl.map(a => [...a]), c});
          if (g[r][cc] && !lbl[r][cc]) {
            steps.push({type:'found', r, cc, c, lbl:lbl.map(a => [...a])});
            const stack = [[r,cc]];
            while (stack.length) {
              const [i,j] = stack.pop();
              if (lbl[i][j]) continue;
              lbl[i][j] = c;
              steps.push({type:'fill', i, j, c, lbl:lbl.map(a => [...a])});
              for (const [ni,nj] of rt_neighbors(i, j, conn))
                if (g[ni][nj] && !lbl[ni][nj]) stack.push([ni,nj]);
            }
            c++;
          }
        }
      }
      steps.push({type:'done', lbl:lbl.map(a => [...a]), c:c-1});
      return steps;
    }

    function rt_getColor(c) {
      const p = rt_PALETTE[(c-1) % rt_PALETTE.length];
      return { fill: p[1], stroke: p[0], text: p[0] };
    }

    function rt_drawLabel() {
      const lbl = rt_label(rt_grid, rt_connectivity);
      const numCC = Math.max(0, ...lbl.flat());
      rt_ctx.clearRect(0, 0, rt_W, rt_H);

      for (let r=0; r<rt_ROWS; r++) {
        for (let c=0; c<rt_COLS; c++) {
          const x = rt_PAD + c * rt_CELL, y = rt_PAD + r * rt_CELL;
          const l = lbl[r][c];
          const active = rt_grid[r][c];

          if (active && l) {
            const col = rt_getColor(l);
            rt_ctx.fillStyle = col.fill; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = col.stroke; rt_ctx.lineWidth = 1.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = col.text;
            rt_ctx.font = 'bold 11px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText(l, x+rt_CELL/2, y+rt_CELL/2);
          } else {
            rt_ctx.fillStyle = '#fafaf7'; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = '#e4dcc8'; rt_ctx.lineWidth = 0.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = '#8a8371';
            rt_ctx.font = '9px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText('0', x+rt_CELL/2, y+rt_CELL/2);
          }

          if (r===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(c, x+rt_CELL/2, rt_PAD/2); }
          if (c===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(r, rt_PAD/2, y+rt_CELL/2); }
        }
      }

      const numPx = rt_grid.flat().filter(Boolean).length;
      root.querySelector('#rt_statPx').textContent = numPx;
      root.querySelector('#rt_statCC').textContent = numCC;
      root.querySelector('#rt_statConn').textContent = rt_connectivity;
      root.querySelector('#rt_statStep').textContent = '–';
      rt_buildLegend(numCC);
    }

    function rt_drawAnimFrame() {
      if (!rt_animSteps.length) return;
      const step = rt_animSteps[Math.min(rt_animIdx, rt_animSteps.length-1)];
      const lbl = step.lbl;
      const numCC = step.c-1 || 0;

      rt_ctx.clearRect(0, 0, rt_W, rt_H);

      for (let r=0; r<rt_ROWS; r++) {
        for (let c=0; c<rt_COLS; c++) {
          const x = rt_PAD + c * rt_CELL, y = rt_PAD + r * rt_CELL;
          const l = lbl[r][c];
          const active = rt_grid[r][c];

          if (active && l) {
            const col = rt_getColor(l);
            rt_ctx.fillStyle = col.fill; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = col.stroke; rt_ctx.lineWidth = 1.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = col.text;
            rt_ctx.font = 'bold 11px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText(l, x+rt_CELL/2, y+rt_CELL/2);
          } else if (active) {
            rt_ctx.fillStyle = '#fef5e7';
            rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = '#b9770e'; rt_ctx.lineWidth = 1; rt_ctx.setLineDash([2,2]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.setLineDash([]);
            rt_ctx.fillStyle = '#b9770e';
            rt_ctx.font = '9px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText('?', x+rt_CELL/2, y+rt_CELL/2);
          } else {
            rt_ctx.fillStyle = '#fafaf7';
            rt_ctx.strokeStyle = '#e4dcc8'; rt_ctx.lineWidth = 0.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
          }

          if (r===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(c, x+rt_CELL/2, rt_PAD/2); }
          if (c===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(r, rt_PAD/2, y+rt_CELL/2); }
        }
      }

      if (step.type==='raster' || step.type==='found') {
        const x = rt_PAD + step.cc * rt_CELL, y = rt_PAD + step.r * rt_CELL;
        rt_ctx.strokeStyle = '#c0392b'; rt_ctx.lineWidth = 2; rt_ctx.setLineDash([]);
        rt_ctx.strokeRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
      }
      if (step.type==='fill') {
        const x = rt_PAD + step.j * rt_CELL, y = rt_PAD + step.i * rt_CELL;
        rt_ctx.strokeStyle = '#b9770e'; rt_ctx.lineWidth = 2; rt_ctx.setLineDash([]);
        rt_ctx.strokeRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
      }

      const descriptions = {
        raster: 'Varredura: (' + step.r + ',' + (step.cc||0) + ') — buscando não rotulado',
        found: 'Pixel em (' + step.r + ',' + (step.cc||0) + ')! Flood-fill: C' + step.c,
        fill: 'Preenchimento: (' + (step.i||0) + ',' + (step.j||0) + ') rotulado C' + step.c,
        done: 'Concluído! ' + step.c + ' componente(s).'
      };
      const titles = { raster: 'Varredura', found: 'Semente!', fill: 'Preenchendo...', done: 'Pronto!' };
      
      root.querySelector('#rt_animTitle').textContent = titles[step.type] || '–';
      root.querySelector('#rt_animDesc').textContent = descriptions[step.type] || '–';
      root.querySelector('#rt_statStep').textContent = (rt_animIdx+1) + '/' + rt_animSteps.length;
      root.querySelector('#rt_statCC').textContent = numCC;
      root.querySelector('#rt_statPx').textContent = rt_grid.flat().filter(Boolean).length;
      rt_buildLegend(numCC);
    }

    function rt_buildLegend(n) {
      const box = root.querySelector('#rt_legendBox');
      box.innerHTML = '';
      if (n === 0) {
        box.innerHTML = '<span style="font-size:9.5px; color:#8a8371;">Nenhum píxel ativo</span>';
        return;
      }
      for (let i=1; i<=n; i++) {
        const p = rt_PALETTE[(i-1) % rt_PALETTE.length];
        const span = document.createElement('span');
        span.style.cssText = 'display:flex; align-items:center; gap:4px; font-size:9.5px; color:#5e5a4a;';
        span.innerHTML = '<span style="width:12px; height:12px; border-radius:3px; background:' + p[1] + '; border:1px solid ' + p[0] + '; display:inline-block"></span><span style="color:' + p[0] + '; font-weight:bold">C' + i + '</span>';
        box.appendChild(span);
      }
    }

    function rt_redraw() {
      if (rt_mode === 'label') rt_drawLabel();
      else rt_drawAnimFrame();
    }

    window.rt_setConn = function(c) {
      rt_connectivity = c;
      root.querySelector('#rt_btn4').classList.toggle('rt_active', c===4);
      root.querySelector('#rt_btn8').classList.toggle('rt_active', c===8);
      root.querySelector('#rt_statConn').textContent = c;
      root.querySelector('#rt_connDesc').textContent = c===4 ? '4 vizinhos ortogonais: N, S, L, O' : '8 vizinhos (inclui diagonais)';
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    };

    window.rt_setMode = function(m) {
      rt_mode = m;
      window.rt_stopPlay();
      root.querySelector('#rt_btnLabel').classList.toggle('rt_active', m==='label');
      root.querySelector('#rt_btnAnim').classList.toggle('rt_active', m==='anim');
      root.querySelector('#rt_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      root.querySelector('#rt_animStatus').style.display = m==='anim' ? 'block' : 'none';
      if (m==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    };

    window.rt_animStep = function(d) {
      rt_animIdx = Math.max(0, Math.min(rt_animSteps.length-1, rt_animIdx+d));
      rt_drawAnimFrame();
    };

    window.rt_togglePlay = function() {
      if (rt_playing) window.rt_stopPlay(); else window.rt_startPlay();
    };

    window.rt_startPlay = function() {
      rt_playing = true;
      root.querySelector('#rt_btnPlay').textContent = '⏸';
      function tick() {
        if (rt_animIdx >= rt_animSteps.length-1) { window.rt_stopPlay(); return; }
        rt_animIdx++;
        rt_drawAnimFrame();
        const spd = +root.querySelector('#rt_speedSlider').value;
        const delay = Math.round(1100 - spd * 100);
        rt_playTimer = setTimeout(tick, delay);
      }
      tick();
    };

    window.rt_stopPlay = function() {
      rt_playing = false;
      if (rt_playTimer) { clearTimeout(rt_playTimer); rt_playTimer = null; }
      root.querySelector('#rt_btnPlay').textContent = '▶';
    };

    window.rt_clearGrid = function() {
      rt_grid = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      rt_animIdx = 0; rt_animSteps = [];
      window.rt_stopPlay();
      rt_redraw();
    };

    const rt_PRESETS = {
      diagonal: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        for (let i=1; i<=5; i++) g[i][i]=1;
        [[2,8],[2,9],[3,8],[3,9],[3,10], [6,1],[6,2],[7,1],[7,2]].forEach(([r,c]) => g[r][c]=1);
        return g;
      },
      letters: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        [[1,1],[2,1],[3,1],[4,1],[5,1],[5,2],[5,3], [1,5],[1,6],[1,7],[2,5],[2,7],[3,5],[3,7],[4,5],[4,7],[5,5],[5,6],[5,7], [1,9],[1,10],[2,9],[3,9],[4,9],[5,9],[5,10]].forEach(([r,c]) => g[r][c]=1);
        return g;
      },
      ring: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        for (let c=2; c<=9; c++) { g[2][c]=1; g[8][c]=1; }
        for (let r=3; r<=7; r++) { g[r][2]=1; g[r][9]=1; }
        for (let c=4; c<=7; c++) { g[4][c]=1; g[6][c]=1; }
        for (let r=5; r<=5; r++) { g[r][4]=1; g[r][7]=1; }
        return g;
      }
    };

    window.rt_loadPreset = function(name) {
      rt_grid = rt_PRESETS[name]();
      rt_animIdx = 0; window.rt_stopPlay();
      if (rt_mode==='anim') rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity);
      rt_redraw();
    };

    function rt_cellAt(e) {
      const rect = rt_cv.getBoundingClientRect();
      const sx = rt_W / rect.width, sy = rt_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - rt_PAD) / rt_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - rt_PAD) / rt_CELL);
      return {r, c};
    }

    rt_cv.addEventListener('mousedown', function(e){
      const {r, c} = rt_cellAt(e);
      if (r<0 || r>=rt_ROWS || c<0 || c>=rt_COLS) return;
      rt_painting = true;
      rt_paintVal = rt_grid[r][c] ? 0 : 1;
      rt_grid[r][c] = rt_paintVal;
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    });

    rt_cv.addEventListener('mousemove', function(e){
      if (!rt_painting) return;
      const {r, c} = rt_cellAt(e);
      if (r<0 || r>=rt_ROWS || c<0 || c>=rt_COLS || rt_grid[r][c] === rt_paintVal) return;
      rt_grid[r][c] = rt_paintVal;
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    });
    
    document.addEventListener('mouseup', () => rt_painting = false);

    rt_grid = rt_PRESETS.diagonal();
    rt_drawLabel();
  }

  function tryInitSim04RotulacaoCC(){
    var root = document.getElementById('sim-04-rotulacao');
    if (root) initSim04RotulacaoCC(root); else setTimeout(tryInitSim04RotulacaoCC, 200);
  }
  tryInitSim04RotulacaoCC();
})();
</script>
""")

**Figure 4.16:** Simulateur interactif d


<figure id="fig-04-sim-04-rotulacao">
  <img src="imagens/fig-04-sim-04-rotulacao.png" alt=" Simulateur interactif d'étiquetage de composantes connexes (*connected component labeling*) : visualisation de l'expansion flood-fill, connectivité-4 et connectivité-8. " style="max-width:80%" />
  <figcaption><strong>Figure 4.16:</strong>  Simulateur interactif d'étiquetage de composantes connexes (*connected component labeling*) : visualisation de l'expansion flood-fill, connectivité-4 et connectivité-8. </figcaption>
</figure>

In [21]:
# Image binaire 10×10 avec des composantes adjacentes en diagonale
f = np.array([
    [0,0,0,0,0,0,0,0,0,0],
    [0,1,0,0,0,0,0,0,0,0],
    [0,0,1,0,0,0,0,0,0,0],  # diagonale avec la ligne précédente
    [0,0,0,1,0,0,1,1,0,0],
    [0,0,0,0,0,0,1,1,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,1,1,1,0,0,0,0,0,0],
    [0,1,0,1,0,0,0,1,0,0],
    [0,1,1,1,0,0,0,0,1,0],  # diagonale avec la ligne précédente
    [0,0,0,0,0,0,0,0,0,0]], dtype=np.uint8) * 255

# Élément structurant en croix (connectivité-4) et carré (connectivité-8)
B4 = mm.secross()   # connectivité-4
B8 = mm.sebox()     # connectivité-8

# Étiquetage avec label0 (didactique) et label (cv2)
lbl4_didatico = mm.label0(f, B4)
lbl8_didatico = mm.label0(f, B8)

_, lbl4_cv2 = cv2.connectedComponents(f, connectivity=4)
_, lbl8_cv2 = cv2.connectedComponents(f, connectivity=8)

# Validation croisée
print(f"✅ label0 (C4) == cv2 (C4) : {np.array_equal(lbl4_didatico, lbl4_cv2)}")
print(f"✅ label0 (C8) == cv2 (C8) : {np.array_equal(lbl8_didatico, lbl8_cv2)}")
print(f"   Composantes C4 : {lbl4_cv2.max()}  |  Composantes C8 : {lbl8_cv2.max()}")

# Normalisation pour visualisation
def norm_label(lbl):
    out = np.zeros_like(lbl, dtype=np.uint8)
    for i, v in enumerate(np.unique(lbl)[1:], 1):
        out[lbl == v] = int(i * 255 / lbl.max())
    return out

mm.show(
    [f, norm_label(lbl4_cv2), norm_label(lbl8_cv2)],
    titles=[
        "f originale",
        f"Étiquetage C4\n({lbl4_cv2.max()} composantes)",
        f"Étiquetage C8\n({lbl8_cv2.max()} composantes)"
    ],
    cols=3, figsize=(12, 4), axis=True
)

✅ label0 (C4) == cv2 (C4) : True
✅ label0 (C8) == cv2 (C8) : True
   Composantes C4 : 7  |  Composantes C8 : 4


<Figure size 1800x600 with 3 Axes>

**Figure 4.17:** Effet de la connectivité sur l


### 4.4.2 Transformée de Distance

La **Transformée de Distance** (TD) est un opérateur qui, appliqué à une image binaire $f$, produit une image en niveaux de gris $D$ dans laquelle chaque pixel appartenant à l'objet ($f(x,y)\neq 0$) reçoit comme valeur la distance géométrique jusqu'au pixel de fond ($f(x',y')=0$) le plus proche :

$$
D(x,y) = \min_{(x',y') \,:\, f(x',y')=0} \; d\bigl((x,y),\,(x',y')\bigr)
$$

où $d(\cdot,\cdot)$ est une métrique de distance — typiquement la distance euclidienne ($L_2$). Le résultat est une représentation topographique des objets : les pixels situés à l'intérieur prennent des valeurs élevées, tandis que les pixels proches des bords présentent de faibles valeurs de distance. Les **maxima locaux** de $D$ correspondent aux points les plus éloignés du bord de l'objet, souvent proches de leurs centres géométriques ou de leurs centres d'inscription maximale — propriété particulièrement utile pour la génération automatique de marqueurs dans l'algorithme *watershed*.

> ### 📝 Définition formelle utilisant les érosions
>
> La TD admet également une interprétation morphologique itérative, selon l'algorithme de la [Figure 4.18](#fig-04-sim-alg-distancia2). Considérons un élément structurant $b$ dont la valeur centrale est nulle et dont les voisins possèdent des coûts négatifs associés au déplacement. En appliquant des érosions successives avec cet élément structurant particulier, les valeurs des pixels des objets (qui doivent prendre la distance maximale possible de l'image) sont progressivement réduites selon les coûts définis par $b$. La valeur accumulée de cette propagation représente alors la distance au fond selon la métrique induite par l'élément structurant.

Cette interprétation est implémentée dans `mm.dist1()`, qui accumule des érosions successives en utilisant l'opération `mm.ero1()`. Quant à `mm.dist()`, elle délègue le calcul de la distance euclidienne à l'opérateur optimisé d'OpenCV `cv2.distanceTransform(f, cv2.DIST_L2, 5)`, où `f` est l'image binaire d'entrée, `cv2.DIST_L2` spécifie la métrique euclidienne ($L_2$) et `5` indique l'utilisation d'un masque 5×5 pour approximer la distance avec une haute précision.

La fonction `dist1` produit une transformée de distance discrète dont la métrique est déterminée par la géométrie et les poids de l'élément structurant utilisé. Par exemple, en utilisant un élément structurant en croix avec un coût unitaire pour les quatre voisins orthogonaux, on obtient la distance de ***Manhattan*** ($L_1$). D'autres choix de voisinage et de poids induisent des métriques différentes. Quant à `mm.dist()`, elle calcule une approximation efficace de la distance euclidienne ($L_2$).

En exigeant des érosions successives sur toute l'image, l'approche `dist1` possède un coût computationnel significativement plus élevé que `mm.dist()`, étant employée dans ce livre principalement à des fins pédagogiques et pour mettre en évidence la relation entre la morphologie mathématique et les transformées de distance.

In [22]:
# @title { display-mode: "form" }
# Préfixe exclusif pour éviter tout conflit avec d'autres cellules du notebook
PREFIX = "dist2"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#E6F1FB;color:#185FA5}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.metric-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:16px}}
.metric-card{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:12px 14px;text-align:left}}
.metric-title{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px}}
.metric-formula{{font-family:var(--font-mono);font-size:12px;color:var(--color-text-primary);margin-bottom:4px}}
.metric-desc{{font-size:12px;color:var(--color-text-secondary);line-height:1.5}}
.metric-elem{{display:inline-grid;grid-template-columns:repeat(3,16px);grid-template-rows:repeat(3,16px);gap:2px;margin-top:6px}}
.mc{{width:16px;height:16px;border-radius:2px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500}}
.mc-on{{background:#E6F1FB;color:#185FA5}}
.mc-off{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}
.mc-ctr{{background:#185FA5;color:#fff}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Transformada de distância por erosões numéricas sucessivas — painel interativo</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-ripple" aria-hidden="true"></i></div>
    <div><div class="algo-title">Transformada de distância por erosão numérica</div><div class="algo-sub">Propagação matemática de distâncias via elemento estruturante com pesos</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Inicializar a imagem de trabalho fazendo uma cópia da original: <code>g ← f.copy()</code>. Os pixels de fundo (0) servem como fontes de distância nula.</div></div>
    <div class="step">
      <div class="step-num num-coral">2</div>
      <div class="step-text">
        Entrar em um laço infinito de erosões com pesos (ponto fixo):
        <div class="step-sub">
          <div class="step-sub-item">Salvar estado anterior: <code>f ← g.copy()</code>.</div>
          <div class="step-sub-item">Erodir: <code>g ← ero1(g, b)</code>, aplicando a subtração local de pesos e computando o valor mínimo para cada vizinhança.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-blue">3</div><div class="step-text">Verificar convergência: se <code>f</code> for idêntica a <code>g</code> (<code>array_equal</code>), a frente de onda de distâncias se estabilizou. Romper o laço (<code>break</code>).</div></div>
    <div class="step"><div class="step-num num-blue">4</div><div class="step-text">Retornar a matriz modificada <code>g</code> contendo o mapa exato de distâncias.</div></div>
    <div class="note">Nesta abordagem morfológica numérica, não há incremento artificial ou contador. A distância propaga-se de fora para dentro porque a erosão contínua puxa o valor <code>0</code> do fundo e o decrementa matematicamente (subtraindo os pesos negativos como <code>-1</code>), fazendo com que os valores escalem radialmente.</div>
  </div>
</div>

<div class="metric-grid">
  <div class="metric-card">
    <div class="metric-title" style="color:#185FA5">Cruz — L₁ (Manhattan)</div>
    <div class="metric-formula">B_cruz [y,x]</div>
    <div class="metric-desc">Pesos: Centro=0, Lados=-1, Cantos=-inf</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#854F0B">Erosão de Cinzas</div>
    <div class="metric-formula">f[vy,vx] - bv</div>
    <div class="metric-desc">Subtrai o peso e busca o valor mínimo local</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#0F6E56">Convergência</div>
    <div class="metric-formula">f == g</div>
    <div class="metric-desc">Para quando nenhum pixel muda de valor</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Clonar imagem de entrada: <code>g ← f.copy()</code>. O objeto possui intensidade alta (255) e o fundo possui intensidade 0.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Mapeamento Local (ero1)</div>
      <div class="card-text">Para cada coordenada <code>(y, x)</code>, buscar o mínimo valor da operação <code>f[vy, vx] - bv</code> aplicada à sua vizinhança estruturante.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Loop Iterativo</div>
      <div class="card-text">Atualizar sequencialmente: <code>f = g.copy()</code> seguido de <code>g = ero1(g, b)</code>. Os valores nulos propagam-se para o interior do objeto.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Critério de Parada</div>
      <div class="card-text">Se <code>np.array_equal(f, g)</code>, significa que o mapa de distâncias atingiu o equilíbrio estável e a propagação terminou.</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Cópia de Trabalho</div><div class="tl-desc">Prepara a matriz inicial `g`.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div>
      <div class="tl-title">Loop de Erosão de Escala</div>
      <div class="while-box">
        <div class="while-head">while True</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Guarda estado: <code>f ← g.copy()</code></div>
          <div class="step-sub-item">Aplica erosão com pesos: <code>g ← ero1(g, b)</code></div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Estabilização Espacial</div><div class="tl-desc">Condição de parada acionada assim que <code>np.array_equal(f, g)</code> se torna verdadeiro.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">04</span></div>
    <div><div class="tl-title">Retorno Numérico</div><div class="tl-desc">Retorna <code>g</code> contendo as distâncias calculadas pela subtração cumulativa dos pesos.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>morph_dist.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">Erosão numérica iterativa</span>
  </div>
  <div class="code-body">
<pre><span class="kw">@staticmethod</span>
<span class="kw">def</span> <span class="fn">ero1</span>(f, b):
    g = np.empty_like(f)
    <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">0</span>]):
        <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">1</span>]):
            g[y,x] = <span class="num-lit">255</span>
            <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f,b,y,x):
                <span class="kw">if</span> np.isinf(bv): <span class="kw">continue</span> 
                val = int(f[vy,vx]) - int(bv)
                <span class="kw">if</span> g[y,x] > val: 
                    g[y,x] = max(0, val)
    <span class="kw">return</span> g

<span class="kw">@staticmethod</span>
<span class="kw">def</span> <span class="fn">dist1</span>(f, b):
    g = f.copy()
    <span class="kw">while</span> <span class="fn">True</span>:
        f = g.copy()
        g = mm.ero1(g, b)
        <span class="kw">if</span> np.array_equal(f, g): 
            <span class="kw">break</span>
    <span class="kw">return</span> g</pre>
  </div>
</div>

<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>g[y,x] = 255</code> — Inicializa o elemento com o valor máximo antes de computar o operador de mínimo da erosão.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">2</div><div class="ann-text"><code>f[vy,vx] - bv</code> — Subtrai o peso associado da vizinhança. Como os pesos da cruz externa são negativos (ex: <code>-1</code>), a operação torna-se uma adição matemática (<code>f[vy,vx] - (-1) = f[vy,vx] + 1</code>) propagando a distância a partir das bordas zeradas.</div></div>
  <div class="ann-line"><div class="ann-badge num-blue">3</div><div class="ann-text"><code>np.array_equal(f, g)</code> — Critério de convergência exato por estabilização de ponto fixo.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 560 720" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:560px;display:block">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#185FA5"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
  </defs>

  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>

  <rect x="150" y="85" width="260" height="46" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="104" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">inicializar mapa</text>
  <text x="280" y="120" text-anchor="middle" font-size="12" fill="#085041">g ← f.copy()</text>

  <rect x="150" y="165" width="260" height="46" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="184" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">salvar estado anterior</text>
  <text x="280" y="200" text-anchor="middle" font-size="12" fill="#712B13">f ← g.copy()</text>

  <rect x="150" y="245" width="260" height="46" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="264" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">executar erosão com pesos</text>
  <text x="280" y="280" text-anchor="middle" font-size="12" fill="#712B13">g ← ero1(g, b)</text>

  <polygon points="280,325 390,355 280,385 170,355" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="351" text-anchor="middle" font-size="12" fill="#0C447C">array_equal(f, g)</text>
  <text x="280" y="367" text-anchor="middle" font-size="12" fill="#0C447C">estabilizou?</text>

  <line x1="170" y1="355" x2="100" y2="355" stroke="#993C1D" stroke-width="1.2"/>
  <line x1="100" y1="355" x2="100" y2="188" stroke="#993C1D" stroke-width="1.2"/>
  <line x1="100" y1="188" x2="150" y2="188" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="120" y="345" text-anchor="middle" font-size="11" fill="#993C1D">não</text>

  <line x1="280" y1="385" x2="280" y2="430" stroke="#0F6E56" stroke-width="1.2" marker-end="url(#arr-g)"/>
  <text x="295" y="405" font-size="11" fill="#0F6E56">sim</text>

  <rect x="150" y="430" width="260" height="46" rx="6" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="458" text-anchor="middle" font-size="12" font-weight="500" fill="#27500A">retornar g</text>

  <ellipse cx="280" cy="525" rx="50" ry="20" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="529" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>
  <line x1="280" y1="476" x2="280" y2="505" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>

  <line x1="280" y1="58" x2="280" y2="85" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="131" x2="280" y2="165" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="211" x2="280" y2="245" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="291" x2="280" y2="325" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>

  <text x="280" y="600" text-anchor="middle" font-size="11.5" fill="#888780" font-style="italic">Ponto Fixo Numérico: A distância emerge da propagação matemática do valor zero.</text>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figure 4.18:** Algorithme de la Transformée de Distance.


<figure id="fig-04-sim-alg-distancia2">
  <img src="imagens/fig-04-sim-alg-distancia2.png" alt=" Algorithme de la Transformée de Distance. " style="max-width:80%" />
  <figcaption><strong>Figure 4.18:</strong>  Algorithme de la Transformée de Distance. </figcaption>
</figure>

A [Figure 4.19](#fig-04-sim-04-distancia) apresenta um simulador interativo da TD: é possível posicionar o cursor sobre diferentes pixels do objeto e observar, em tempo real, o valor da distância associado àquela posição, isto é, a distância até o pixel de fundo mais próximo. A [Figure 4.20](#fig-04-distancia-didatico) apresenta um exemplo prático dessa execução em ambiente Python.

A [Figure 4.19](#fig-04-sim-04-distancia) apresenta um simulador interativo da transformada de distância (TD): é possível posicionar o cursor sobre diferentes pixels do objeto e observar, em tempo real, o valor da distância associado a essa posição, ou seja, a distância até o pixel de fundo mais próximo. A [Figure 4.20](#fig-04-distancia-didatico) apresenta um exemplo prático dessa execução em ambiente Python.

In [23]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-distancia * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-distancia canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-distancia button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-distancia button:hover { background: #e8dfcf; }
  #sim-04-distancia button.sim04_td_act { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-distancia .sim04_td_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_td_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim04_td_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .sim04_td_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim04_td_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim04_td_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  .sim04_td_se_grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 4px; max-width: 120px; margin: 6px auto; }
  .sim04_td_se_btn { padding: 6px!important; font-size: 10px!important; font-family: monospace; font-weight: 700; }
  .sim04_td_se_btn.sim04_td_inf { color: #8a8371!important; background: #fafaf7!important; border-color: #e4dcc8!important; font-weight: normal; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🗺️ Simulateur : Transformée de Distance (TD)</span>
  <span class="sim04_td_pill">Frontières à +∞ (144)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="sim04_td_grid_stats">
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Pixels Actifs</div>
      <div id="sim04_td_statPx" class="sim04_td_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Distance Max.</div>
      <div id="sim04_td_statMax" class="sim04_td_stat_value" style="color:#2980b9;">0</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Métrique Actuelle</div>
      <div id="sim04_td_statMetric" class="sim04_td_stat_value" style="color:#b9770e; font-size:11px;">L∞ (Tchebychev)</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Itération (k)</div>
      <div id="sim04_td_statStep" class="sim04_td_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas e Legenda -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="sim04_td_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Cliquez pour activer/désactiver les pixels · Glissez pour peindre
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:6px; margin-top:10px; align-items:center; justify-content:center;" id="sim04_td_legendBox"></div>
    </div>

    <!-- Painel de Controles Lateral -->
    <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Editor do Elemento Estruturante (b) -->
      <div class="sim04_td_panel" style="text-align:center;">
        <div style="font-size:11px; font-weight:700; margin-bottom:2px; text-align:left; color:#5e5a4a;">
          Élément Structurant (b)
        </div>
        <div style="font-size:9.5px; color:#8a8371; margin-bottom:6px; text-align:left;">Cliquez pour modifier les poids :</div>
        
        <div class="sim04_td_se_grid">
          <button id="sim04_td_se_0_0" class="sim04_td_se_btn sim04_td_active" data-se="0,0">-1</button>
          <button id="sim04_td_se_0_1" class="sim04_td_se_btn sim04_td_active" data-se="0,1">-1</button>
          <button id="sim04_td_se_0_2" class="sim04_td_se_btn sim04_td_active" data-se="0,2">-1</button>
          
          <button id="sim04_td_se_1_0" class="sim04_td_se_btn sim04_td_active" data-se="1,0">-1</button>
          <button id="sim04_td_se_1_1" class="sim04_td_se_btn" style="background:#e4dcc8; cursor:not-allowed; color:#8a8371;" disabled>0</button>
          <button id="sim04_td_se_1_2" class="sim04_td_se_btn sim04_td_active" data-se="1,2">-1</button>
          
          <button id="sim04_td_se_2_0" class="sim04_td_se_btn sim04_td_active" data-se="2,0">-1</button>
          <button id="sim04_td_se_2_1" class="sim04_td_se_btn sim04_td_active" data-se="2,1">-1</button>
          <button id="sim04_td_se_2_2" class="sim04_td_se_btn sim04_td_active" data-se="2,2">-1</button>
        </div>
        
        <div style="display:flex; flex-wrap:wrap; gap:4px; margin-top:8px;">
          <button data-se-preset="cross" style="flex:1; font-size:9.5px;">L1 (Croix)</button>
          <button data-se-preset="square" style="flex:1; font-size:9.5px;">L∞ (Carré)</button>
          <button data-se-preset="chamfer" style="flex:1; font-size:9.5px; min-width:90px;">Chamfer 3-4</button>
        </div>
      </div>

      <!-- Modo de Visualização -->
      <div class="sim04_td_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualisation
        </div>
        <div style="display:flex; gap:6px;">
          <button id="sim04_td_btnLabel" class="sim04_td_active" data-mode="label" style="flex:1; justify-content:center;">Final (TD)</button>
          <button id="sim04_td_btnAnim" data-mode="anim" style="flex:1; justify-content:center;">Animé</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="sim04_td_animCtrl" class="sim04_td_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Propagation Pas à Pas
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button data-action="prev" style="flex:1; justify-content:center;">◀</button>
          <button id="sim04_td_btnPlay" data-action="play" style="flex:1; justify-content:center;">▶</button>
          <button data-action="next" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Vit.</label>
          <input type="range" id="sim04_td_speedSlider" min="1" max="10" value="5" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Exemplos f -->
      <div class="sim04_td_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Exemples Prédéfinis
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button data-preset="square" style="justify-content:flex-start;">■ Carré Plein</button>
          <button data-preset="lshape" style="justify-content:flex-start;">╚ Forme en L</button>
          <button data-preset="ring" style="justify-content:flex-start;">◎ Anneau avec Trou</button>
          <button data-preset="corner0" style="justify-content:flex-start;">↘ Fond en (0,0)</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button data-action="clear" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Effacer la Grille
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04TD(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const td_COLS = 12, td_ROWS = 12, td_CELL = 22, td_PAD = 10;
    const td_W = td_COLS * td_CELL + td_PAD * 2, td_H = td_ROWS * td_CELL + td_PAD * 2;
    const td_cv = root.querySelector('#sim04_td_Canvas');
    td_cv.width = td_W; 
    td_cv.height = td_H;
    const td_ctx = td_cv.getContext('2d');
    
    const td_MAX_DIST = 144;

    const td_COLORS = ['#ebf4fd', '#a9cce3', '#73c6b6', '#52be80', '#27ae60', '#2980b9', '#1b4f72', '#5b2c6f', '#4a235a'];
    function td_getColor(d, maxD) {
      if(d === 0) return { fill: 'transparent', text: '#8a8371', stroke: '#e4dcc8' };
      if(d >= td_MAX_DIST) return { fill: '#fdecea', text: '#c0392b', stroke: '#f5b7b1' };
      
      let ratio = maxD > 1 ? (d - 1) / (maxD - 1) : 0;
      let idx = Math.max(0, Math.min(td_COLORS.length - 1, Math.floor(ratio * (td_COLORS.length - 1))));
      
      const bg = td_COLORS[idx];
      const text = (idx > 4) ? '#ffffff' : '#26241d';
      return { fill: bg, text: text, stroke: '#2980b9' };
    }

    let td_grid = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
    
    const td_VALS = [-1, -2, -3, -4, -5, -6, -7, -8, -9, -Infinity];
    let td_b = [
      [-1, -1, -1],
      [-1,  0, -1],
      [-1, -1, -1]
    ];
    
    let td_mode = 'label';
    let td_painting = false;
    let td_paintVal = 1;

    let td_animSteps = [];
    let td_animIdx = 0;
    let td_playing = false;
    let td_playTimer = null;

    function td_compute() {
      let f = td_grid;
      let g = Array.from({length: td_ROWS}, (_, r) => 
        Array.from({length: td_COLS}, (_, c) => f[r][c] ? td_MAX_DIST : 0)
      );
      let steps = [];
      let k = 0;

      const copy = arr => arr.map(row => [...row]);
      steps.push({ g: copy(g), eroded: Array.from({length:td_ROWS}, ()=>new Array(td_COLS).fill(0)), k: k });

      let activePx = f.flat().reduce((a,b)=>a+b, 0);
      if(activePx === 0) return { D: g, steps, max: 0 };

      while (true) {
        k++;
        let nextG = Array.from({length: td_ROWS}, () => new Array(td_COLS).fill(0));
        let changed = false;
        let erodedCells = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));

        for (let r=0; r<td_ROWS; r++) {
          for (let c=0; c<td_COLS; c++) {
            if (f[r][c] === 1) {
              let min_val = g[r][c];
              
              for (let dr=-1; dr<=1; dr++) {
                for (let dc=-1; dc<=1; dc++) {
                  let b_val = td_b[dr+1][dc+1];
                  if (b_val !== -Infinity && !(dr===0 && dc===0)) {
                    let nr = r + dr, nc = c + dc;
                    let neighbor_val = td_MAX_DIST;
                    
                    if (nr >= 0 && nr < td_ROWS && nc >= 0 && nc < td_COLS) {
                      neighbor_val = g[nr][nc];
                    }
                    
                    let val = neighbor_val - b_val; 
                    if (val < min_val) {
                      min_val = val;
                    }
                  }
                }
              }
              nextG[r][c] = min_val;
              if (min_val !== g[r][c]) {
                changed = true;
                erodedCells[r][c] = 1;
              }
            } else {
              nextG[r][c] = 0;
            }
          }
        }
        
        if (!changed) {
          steps.push({ g: copy(nextG), eroded: erodedCells, k: k, done: true });
          break;
        }
        
        steps.push({ g: copy(nextG), eroded: erodedCells, k: k });
        g = nextG;
      }
      
      let maxD = 0;
      for(let r=0; r<td_ROWS; r++){
        for(let c=0; c<td_COLS; c++){
           if(f[r][c] === 1 && g[r][c] < td_MAX_DIST && g[r][c] > maxD) maxD = g[r][c];
        }
      }
      return { D: g, steps, max: maxD };
    }

    function td_drawFinal() {
      const res = td_compute();
      const D = res.D;
      td_ctx.clearRect(0, 0, td_W, td_H);

      for (let r=0; r<td_ROWS; r++) {
        for (let c=0; c<td_COLS; c++) {
          const x = td_PAD + c * td_CELL, y = td_PAD + r * td_CELL;
          const d = D[r][c];
          const active = td_grid[r][c];

          if (active) {
            const col = td_getColor(d, res.max);
            td_ctx.fillStyle = col.fill; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            td_ctx.strokeStyle = col.stroke; td_ctx.lineWidth = 1; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = col.text;
            td_ctx.font = d >= td_MAX_DIST ? 'bold 9px monospace' : 'bold 11px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText(d >= td_MAX_DIST ? '144' : d, x+td_CELL/2, y+td_CELL/2);
          } else {
            td_ctx.fillStyle = '#fafaf7'; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            td_ctx.strokeStyle = '#e4dcc8'; td_ctx.lineWidth = 0.5; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = '#8a8371';
            td_ctx.font = '9px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText('0', x+td_CELL/2, y+td_CELL/2);
          }

          if (r===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(c, x+td_CELL/2, td_PAD/2); }
          if (c===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(r, td_PAD/2, y+td_CELL/2); }
        }
      }

      root.querySelector('#sim04_td_statPx').textContent = td_grid.flat().filter(Boolean).length;
      root.querySelector('#sim04_td_statMax').textContent = res.max;
      root.querySelector('#sim04_td_statStep').textContent = 'Finalizado';
      td_buildLegend(res.D, res.max);
    }

    function td_drawAnimFrame() {
      if (!td_animSteps.length) return;
      const step = td_animSteps[Math.min(td_animIdx, td_animSteps.length-1)];
      const D = step.g;
      const eroded = step.eroded;
      
      let currentMaxD = 0;
      for(let r=0; r<td_ROWS; r++) {
        for(let c=0; c<td_COLS; c++) {
          if(td_grid[r][c] && D[r][c] < td_MAX_DIST && D[r][c] > currentMaxD) currentMaxD = D[r][c];
        }
      }

      td_ctx.clearRect(0, 0, td_W, td_H);

      for (let r=0; r<td_ROWS; r++) {
        for (let c=0; c<td_COLS; c++) {
          const x = td_PAD + c * td_CELL, y = td_PAD + r * td_CELL;
          const d = D[r][c];
          const active = td_grid[r][c];

          if (active) {
            let maxValArray = td_animSteps[td_animSteps.length-1].g.flat().filter(v=>v<td_MAX_DIST);
            const globalMax = maxValArray.length > 0 ? Math.max(...maxValArray) : 1;
            const col = td_getColor(d, globalMax);
            td_ctx.fillStyle = col.fill; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            
            if(eroded[r][c] === 1) {
               td_ctx.strokeStyle = '#b9770e'; td_ctx.lineWidth = 2; td_ctx.setLineDash([2,2]);
               td_ctx.fillStyle = 'rgba(185, 119, 14, 0.15)'; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            } else {
               td_ctx.strokeStyle = col.stroke; td_ctx.lineWidth = 1; td_ctx.setLineDash([]);
            }
            
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = col.text;
            td_ctx.font = d >= td_MAX_DIST ? 'bold 9px monospace' : 'bold 11px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText(d >= td_MAX_DIST ? '144' : d, x+td_CELL/2, y+td_CELL/2);
            td_ctx.setLineDash([]);
          } else {
            td_ctx.fillStyle = '#fafaf7';
            td_ctx.strokeStyle = '#e4dcc8'; td_ctx.lineWidth = 0.5; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = '#8a8371';
            td_ctx.font = '9px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText('0', x+td_CELL/2, y+td_CELL/2);
          }

          if (r===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(c, x+td_CELL/2, td_PAD/2); }
          if (c===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(r, td_PAD/2, y+td_CELL/2); }
        }
      }

      root.querySelector('#sim04_td_statMax').textContent = currentMaxD;
      root.querySelector('#sim04_td_statPx').textContent = td_grid.flat().filter(Boolean).length;
      root.querySelector('#sim04_td_statStep').textContent = step.done ? 'Concluído' : step.k;
      td_buildLegend(D, currentMaxD);
    }

    function td_buildLegend(D_matrix, maxD) {
      const box = root.querySelector('#sim04_td_legendBox');
      box.innerHTML = '';
      
      let uniqueD = new Set();
      D_matrix.forEach(row => row.forEach(val => { 
        if(val > 0 && val < td_MAX_DIST) uniqueD.add(val); 
      }));
      let sortedD = Array.from(uniqueD).sort((a,b) => a-b);
      
      if (sortedD.length === 0) {
        box.innerHTML = '<span style="font-size:9.5px; color:#8a8371;">Sem propagação visível</span>';
        return;
      }
      
      let displayArr = sortedD;
      if (sortedD.length > 12) {
          displayArr = sortedD.filter((_, i) => i === 0 || i === sortedD.length-1 || i%Math.ceil(sortedD.length/10) === 0);
      }
      
      box.innerHTML = '<span style="font-size:9.5px; color:#8a8371; width:100%;">Cores (exceto 144):</span>';
      
      displayArr.forEach(d => {
        const col = td_getColor(d, maxD);
        const span = document.createElement('span');
        span.style.cssText = 'display:flex; align-items:center; gap:4px; font-size:9.5px; color:#5e5a4a;';
        span.innerHTML = '<span style="width:12px; height:12px; border-radius:3px; background:' + col.fill + '; border:1px solid ' + col.stroke + '; display:inline-block"></span><span>' + d + '</span>';
        box.appendChild(span);
      });
    }

    function td_updateSEUI() {
      let typeName = "Personalizada";
      let isCross = td_b[0][1]===-1 && td_b[1][0]===-1 && td_b[1][2]===-1 && td_b[2][1]===-1 && td_b[0][0]===-Infinity && td_b[0][2]===-Infinity && td_b[2][0]===-Infinity && td_b[2][2]===-Infinity;
      let isSquare = td_b.every((r, i) => r.every((v, j) => (i===1 && j===1) ? true : v===-1));
      let isChamfer = td_b[0][1]===-3 && td_b[1][0]===-3 && td_b[1][2]===-3 && td_b[2][1]===-3 && td_b[0][0]===-4 && td_b[0][2]===-4 && td_b[2][0]===-4 && td_b[2][2]===-4;

      if (isCross) typeName = "L1 (City-Block)";
      else if (isSquare) typeName = "L∞ (Chebyshev)";
      else if (isChamfer) typeName = "Chamfer 3-4";

      root.querySelector('#sim04_td_statMetric').textContent = typeName;

      for (let r=0; r<3; r++) {
        for (let c=0; c<3; c++) {
          if(r===1 && c===1) continue;
          const btn = root.querySelector('#sim04_td_se_' + r + '_' + c);
          const val = td_b[r][c];
          
          if(val !== -Infinity) {
            btn.textContent = String(val);
            btn.className = 'sim04_td_se_btn sim04_td_active';
          } else {
            btn.textContent = '-∞';
            btn.className = 'sim04_td_se_btn sim04_td_inf';
          }
        }
      }
    }

    function td_toggleSE(r, c) {
      if(r===1 && c===1) return;
      
      let currentVal = td_b[r][c];
      let idx = td_VALS.indexOf(currentVal);
      let nextVal = td_VALS[(idx + 1) % td_VALS.length];
      
      td_b[r][c] = nextVal;
      td_updateSEUI();
      td_refresh();
    };

    function td_setPresetSE(type) {
      for (let r=0; r<3; r++) for (let c=0; c<3; c++) td_b[r][c] = -Infinity;
      td_b[1][1] = 0;

      if (type === 'cross') {
        td_b[0][1] = td_b[1][0] = td_b[1][2] = td_b[2][1] = -1;
      } else if (type === 'square') {
        for (let r=0; r<3; r++) for (let c=0; c<3; c++) if(!(r===1 && c===1)) td_b[r][c] = -1;
      } else if (type === 'chamfer') {
        td_b[0][1] = td_b[1][0] = td_b[1][2] = td_b[2][1] = -3;
        td_b[0][0] = td_b[0][2] = td_b[2][0] = td_b[2][2] = -4;
      }
      
      td_updateSEUI();
      td_refresh();
    };

    function td_refresh() {
      if (td_mode === 'anim') {
        td_animSteps = td_compute().steps;
        td_animIdx = 0;
        td_drawAnimFrame();
      } else {
        td_drawFinal();
      }
    }

    function td_setMode(m) {
      td_mode = m;
      td_stopPlay();
      root.querySelector('#sim04_td_btnLabel').classList.toggle('sim04_td_active', m==='label');
      root.querySelector('#sim04_td_btnAnim').classList.toggle('sim04_td_active', m==='anim');
      root.querySelector('#sim04_td_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      td_refresh();
    }

    function td_animStep(d) {
      td_animIdx = Math.max(0, Math.min(td_animSteps.length-1, td_animIdx+d));
      td_drawAnimFrame();
    }

    function td_togglePlay() {
      if (td_playing) td_stopPlay(); else td_startPlay();
    }

    function td_startPlay() {
      td_playing = true;
      root.querySelector('#sim04_td_btnPlay').textContent = '⏸';
      function tick() {
        if (td_animIdx >= td_animSteps.length-1) { td_stopPlay(); return; }
        td_animIdx++;
        td_drawAnimFrame();
        const spd = +root.querySelector('#sim04_td_speedSlider').value;
        const delay = Math.round(1200 - spd * 100);
        td_playTimer = setTimeout(tick, delay);
      }
      tick();
    }

    function td_stopPlay() {
      td_playing = false;
      if (td_playTimer) { clearTimeout(td_playTimer); td_playTimer = null; }
      root.querySelector('#sim04_td_btnPlay').textContent = '▶';
    }

    function td_clearGrid() {
      td_grid = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
      td_stopPlay();
      td_refresh();
    }

    const td_PRESETS = {
      square: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let r=2; r<=9; r++) for(let c=2; c<=9; c++) g[r][c] = 1;
        return g;
      },
      lshape: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let r=2; r<=9; r++) { g[r][2]=1; g[r][3]=1; g[r][4]=1; }
        for(let c=5; c<=9; c++) { g[7][c]=1; g[8][c]=1; g[9][c]=1; }
        return g;
      },
      ring: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let c=2; c<=9; c++) { g[2][c]=1; g[3][c]=1; g[8][c]=1; g[9][c]=1; }
        for(let r=4; r<=7; r++) { g[r][2]=1; g[r][3]=1; g[r][8]=1; g[r][9]=1; }
        return g;
      },
      corner0: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(1));
        g[0][0] = 0;
        return g;
      }
    };

    function td_loadPreset(name) {
      td_grid = td_PRESETS[name]();
      td_stopPlay();
      td_refresh();
    }

    function td_cellAt(e) {
      const rect = td_cv.getBoundingClientRect();
      const sx = td_W / rect.width, sy = td_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - td_PAD) / td_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - td_PAD) / td_CELL);
      return {r, c};
    }

    td_cv.addEventListener('mousedown', function(e){
      const {r, c} = td_cellAt(e);
      if (r<0 || r>=td_ROWS || c<0 || c>=td_COLS) return;
      td_painting = true;
      td_paintVal = td_grid[r][c] ? 0 : 1;
      td_grid[r][c] = td_paintVal;
      td_refresh();
    });

    td_cv.addEventListener('mousemove', function(e){
      if (!td_painting) return;
      const {r, c} = td_cellAt(e);
      if (r<0 || r>=td_ROWS || c<0 || c>=td_COLS || td_grid[r][c] === td_paintVal) return;
      td_grid[r][c] = td_paintVal;
      td_refresh();
    });
    
    document.addEventListener('mouseup', () => td_painting = false);

    root.querySelectorAll('[data-se]').forEach(btn => {
      btn.addEventListener('click', function() {
        const parts = this.getAttribute('data-se').split(',');
        td_toggleSE(parseInt(parts[0], 10), parseInt(parts[1], 10));
      });
    });

    root.querySelectorAll('[data-se-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_setPresetSE(this.getAttribute('data-se-preset'));
      });
    });

    root.querySelectorAll('[data-mode]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_setMode(this.getAttribute('data-mode'));
      });
    });

    root.querySelectorAll('[data-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_loadPreset(this.getAttribute('data-preset'));
      });
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'clear') td_clearGrid();
        if(act === 'prev') td_animStep(-1);
        if(act === 'next') td_animStep(1);
        if(act === 'play') td_togglePlay();
      });
    });

    td_grid = td_PRESETS.corner0();
    td_updateSEUI();
    td_refresh();
  }

  function tryInitSim04TD(){
    var root = document.getElementById('sim-04-distancia');
    if (root) initSim04TD(root); else setTimeout(tryInitSim04TD, 200);
  }
  tryInitSim04TD();
})();
</script>
""")

**Figure 4.19:** Simulateur interactif de la Transformée de Distance (TD) itérative via érosion en niveaux de gris. Les pixels hors de l


<figure id="fig-04-sim-04-distancia">
  <img src="imagens/fig-04-sim-04-distancia.png" alt=" Simulateur interactif de la Transformée de Distance (TD) itérative via érosion en niveaux de gris. Les pixels hors de l'image prennent la valeur maximale (144), propageant les coûts depuis le fond interne. " style="max-width:80%" />
  <figcaption><strong>Figure 4.19:</strong>  Simulateur interactif de la Transformée de Distance (TD) itérative via érosion en niveaux de gris. Les pixels hors de l'image prennent la valeur maximale (144), propageant les coûts depuis le fond interne. </figcaption>
</figure>

In [24]:
import numpy as np

# Image binaire 10×10 avec un seul pixel de fond (0,0) et le reste comme objet (255)
f = np.ones((10,10), dtype=np.uint8) * 255
f[0,0] = 0

B_cruz = np.array([
    [-np.inf, -1, -np.inf],
    [-1,      0, -1],
    [-np.inf, -1, -np.inf]
], dtype=float)

d_iter = mm.dist1(f, B_cruz)
d_l2   = mm.dist(f)

print(f"Max. dist1 (érosions) : {d_iter.max()} px")
print(f"Max. dist  (L2)      : {d_l2.max():.2f} px")
print(f"Position du maximum dist1 : {np.where(d_iter == d_iter.max())}")
print(f"Position du maximum dist  : {np.where(d_l2 == d_l2.max())}")

mm.show(
    [f,        d_iter,                    d_l2],
    titles=["f original", "mm.dist1\n(érosions avec croix)", "mm.dist\n(L2 euclidienne)"],
    cols=3, figsize=(12, 4), axis=True
)

Max. dist1 (érosions) : 18 px
Max. dist  (L2)      : 12.00 px
Position du maximum dist1 : (array([9]), array([9]))
Position du maximum dist  : (array([9]), array([9]))


<Figure size 1800x600 with 3 Axes>

**Figure 4.20:** Transformée de distance sur une image binaire 10×10. Gauche : image originale (*foreground* = 255). Centre : mm.dist1 itérative (érosions avec élément croix). Droite : mm.dist (L2 euclidienne).


L’annotation des valeurs numériques directement sur les pixels permet de vérifier comment `dist1` propage les distances selon la métrique induite par la fonction structurante utilisée. Dans le cas de l’élément en croix à coût unitaire, les valeurs obtenues correspondent à la distance de *Manhattan* ($L_1$). Bien que `dist1` et `mm.dist` produisent des valeurs numériques distinctes en adoptant des métriques différentes, les deux transformées préservent la structure topographique des objets, faisant en sorte que leurs maxima se produisent dans des régions centrales similaires. Cette propriété justifie l’utilisation de `mm.dist` dans des applications pratiques, en raison de sa grande efficacité computationnelle.

### 4.4.3 Transformada de Distância Euclidiana em quatro passos

Le simulateur de la [Figure 4.21](#fig-04-sim-04-tde) implémente l'algorithme de la TDE de Lotufo (2001) en deux étapes. Dans la première étape, la fonction `edt1` effectue une transformation unidimensionnelle verticale de manière séquentielle (*in-place*), parcourant chaque colonne en *raster* (↓) et en *anti-raster* (↑) pour calculer les distances dans la direction verticale (deux étapes : Sud et Nord). Dans la deuxième étape, la fonction `edt2` utilise ce résultat comme entrée et effectue une propagation horizontale par files : pour chaque ligne de la matrice, deux files de priorité, `Eq` et `Wq`, sont initialisées en parcourant les indices de colonne dans des directions opposées (`Eq` de `W-1` à `1`, `Wq` de `2` à `W`), de sorte que chaque pixel mis à jour met immédiatement en file ses voisins pour retraitement dans la même itération (deux étapes supplémentaires : Est et Ouest). Ce mécanisme de file permet d'appliquer des érosions successives avec des poids impairs croissants (`b = 1, 3, 5, ...`, incrémenté à chaque itération de la boucle externe) sans que la propagation ne reste piégée dans des valeurs obsolètes, car chaque itération résout complètement la chaîne de dépendances horizontale avant l'incrément suivant de `b`. La convergence de cette propagation produit la Transformada de Distância Euclidiana  sur toute la matrice, combinant l'information verticale obtenue dans `edt1` avec la propagation horizontale par file réalisée dans `edt2`.

In [25]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-tde" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-tde * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-tde canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; margin: 0 auto; }
  #sim-04-tde button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all .15s ease; display: inline-flex; align-items: center; gap: 5px; font-weight: 600; }
  #sim-04-tde button:hover { background: #e8dfcf; }
  #sim-04-tde button.sim04_edt_act { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-tde .sim04_edt_smtxt { font-size: 10.5px; color: #8a8371; line-height: 1.4; }
  #sim-04-tde .sim04_edt_mono { font-family: monospace; }
  #sim-04-tde .sim04_edt_statcard { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 8px 10px; text-align: center; min-width: 0; }
  #sim-04-tde .sim04_edt_statlabel { font-size: 9.5px; text-transform: uppercase; letter-spacing: .04em; color: #8a8371; margin-bottom: 2px; font-weight: 700; }
  #sim-04-tde .sim04_edt_statval { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim04_edt_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📐 EDT² 2D Corrigé · Matrice 4×4</span>
  <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Convergence Exacte</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição Teórica -->
  <div class="sim04_edt_panel" style="margin-bottom:14px;">
    <div style="font-size:11.5px; font-weight:700; color:#26241d; margin-bottom:2px;">Étape 1 : edt1 Vertical In-place · Étape 2 : edt2 Horizontal In-place</div>
    <div class="sim04_edt_smtxt">Utilise la même structure raster/anti-raster décrite dans l'article pour la propagation de distances exactes (exemple de l'article, p. 103).</div>
  </div>

  <!-- Cards de Estatísticas -->
  <div style="display:grid; grid-template-columns:repeat(3, 1fr); gap:10px; margin-bottom:14px;">
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Étape Actuelle</div><div class="sim04_edt_statval" id="sim04_edt_sStep" style="color:#c0392b;">–</div></div>
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Phase</div><div class="sim04_edt_statval" id="sim04_edt_sPhase" style="color:#2980b9; font-size:12px;">–</div></div>
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">b Actuel</div><div class="sim04_edt_statval" id="sim04_edt_sB" style="color:#b9770e;">–</div></div>
  </div>

  <!-- Canvas da Matriz 4x4 -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim04_edt_cvs"></canvas>
  </div>

  <!-- Barra de Ações -->
  <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:12px; align-items:center; justify-content:center;">
    <button id="sim04_edt_btnFinal" class="sim04_edt_act" data-action="final">Résultat Final</button>
    <button id="sim04_edt_btnStep" data-action="step">Pas à Pas</button>
    <div style="width:1px; background:#e4dcc8; height:20px;"></div>
    <button data-action="clear" style="border-color:#f5b7b1; color:#c0392b; background:#fdecea;">× Effacer</button>
  </div>

  <!-- Painel de Passo a Passo (Controles de Animação) -->
  <div id="sim04_edt_stepCtrl" style="display:none;" class="sim04_edt_panel">
    <div style="display:flex; gap:6px; align-items:center; flex-wrap:wrap;">
      <button data-action="prev">◀ Précédent</button>
      <button id="sim04_edt_btnPlay" data-action="play">▶ Lecture</button>
      <button data-action="next">Suivant ▶</button>
      <span class="sim04_edt_smtxt" style="margin-left:4px; font-weight:700;">Vit. :</span>
      <input type="range" id="sim04_edt_speedSlider" min="1" max="10" value="5" step="1" style="width:70px; cursor:pointer;">
      <span id="sim04_edt_stepLabel" class="sim04_edt_mono" style="font-size:10.5px; color:#5e5a4a; flex:1; text-align:right; min-width:0; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;"></span>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04EdtConvergencia(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const COLS=4, ROWS=4, CELL=50, PAD=20, INF=9999;
    const cv = root.querySelector('#sim04_edt_cvs');
    const CTX = cv.getContext('2d');

    cv.width = COLS * CELL + PAD * 2;
    cv.height = ROWS * CELL + PAD * 2;

    let grid = [
      [1, 1, 0, 1],
      [1, 1, 1, 1],
      [1, 0, 1, 1],
      [0, 1, 1, 1]
    ];

    let mode='final';
    let steps=[], stepIdx=0;
    let playing=false, playTimer=null;

    function getCol(d) {
      if(d === 0) return { fill: 'transparent', txt: '#8a8371', bdr: '#e4dcc8' };
      if(d >= INF) return { fill: '#fdecea', txt: '#c0392b', bdr: '#f5b7b1' };
      const colors = ['#ebf4fd', '#a9cce3', '#73c6b6', '#52be80', '#27ae60', '#2980b9', '#1b4f72'];
      let idx = Math.min(colors.length - 1, Math.floor(d / 1.5));
      return { fill: colors[idx], txt: idx > 3 ? '#ffffff' : '#26241d', bdr: colors[Math.min(idx+1, colors.length-1)] };
    }

    function drawMatrix(matrix, opts={}) {
      CTX.clearRect(0, 0, cv.width, cv.height);
      for(let r=0; r<ROWS; r++) {
        for(let c=0; c<COLS; c++) {
          const px = PAD + c * CELL; const py = PAD + r * CELL;
          const d = matrix[r][c]; const col = getCol(d);

          CTX.fillStyle = col.fill; CTX.fillRect(px+1, py+1, CELL-2, CELL-2);

          const isCur = opts.cursor && opts.cursor.r === r && opts.cursor.c === c;
          const isChg = opts.changed && opts.changed[r][c];

          CTX.strokeStyle = isCur ? '#c0392b' : isChg ? '#b9770e' : col.bdr;
          CTX.lineWidth = isCur ? 3 : isChg ? 2.5 : 0.5;
          CTX.strokeRect(px+0.5, py+0.5, CELL-1, CELL-1);

          CTX.fillStyle = col.txt; CTX.font = 'bold 14px monospace';
          CTX.textAlign = 'center'; CTX.textBaseline = 'middle';
          CTX.fillText(d >= INF ? '∞' : String(d), px + CELL/2, py + CELL/2);

          if (r === 0) { CTX.fillStyle='#8a8371'; CTX.font='9px monospace'; CTX.fillText('c'+c, px+CELL/2, PAD/2); }
          if (c === 0) { CTX.fillStyle='#8a8371'; CTX.font='9px monospace'; CTX.fillText('r'+r, PAD/2, py+CELL/2); }
        }
      }
    }

    function copyMat(m) { return m.map(row => [...row]); }

    function computeEdt1(gInit) {
      const all = [];
      let g = copyMat(gInit);
      all.push({ phase: 'edt1_init', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'edt1 · Matriz de entrada inicializada' });

      for(let c=0; c<COLS; c++) {
        let b = 1;
        for(let r=1; r<ROWS; r++) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r-1][c] < INF) {
            if(g[r][c] > g[r-1][c] + b) {
              g[r][c] = g[r-1][c] + b; chg[r][c] = 1;
              all.push({ phase: 'edt1_raster', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt1 raster ↓ · f(' + r + ',' + c + ') = f(' + (r-1) + ',' + c + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt1_raster', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt1 raster ↓ · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      for(let c=0; c<COLS; c++) {
        let b = 1;
        for(let r=ROWS-2; r>=0; r--) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r+1][c] < INF) {
            if(g[r][c] > g[r+1][c] + b) {
              g[r][c] = g[r+1][c] + b; chg[r][c] = 1;
              all.push({ phase: 'edt1_anti', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt1 anti-raster ↑ · f(' + r + ',' + c + ') = f(' + (r+1) + ',' + c + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt1_anti', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt1 anti-raster ↑ · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }
      return { steps: all, g: copyMat(g) };
    }

    function computeEdt2(gIn) {
      const all = [];
      let g = copyMat(gIn);
      all.push({ phase: 'edt2_init', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'edt2 · Estado inicial f_v antes da propagação horizontal' });

      for(let r=0; r<ROWS; r++) {
        let b = 1;
        for(let c=1; c<COLS; c++) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r][c-1] < INF) {
            if(g[r][c] > g[r][c-1] + b) {
              g[r][c] = g[r][c-1] + b; chg[r][c] = 1;
              all.push({ phase: 'edt2_raster', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt2 raster → · f(' + r + ',' + c + ') = f(' + r + ',' + (c-1) + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt2_raster', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt2 raster → · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      for(let r=0; r<ROWS; r++) {
        let b = 1;
        for(let c=COLS-2; c>=0; c--) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r][c+1] < INF) {
            if(g[r][c] > g[r][c+1] + b) {
              g[r][c] = g[r][c+1] + b; chg[r][c] = 1;
              all.push({ phase: 'edt2_anti', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt2 anti-raster ← · f(' + r + ',' + c + ') = f(' + r + ',' + (c+1) + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt2_anti', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt2 anti-raster ← · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      all.push({ phase: 'edt2_done', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'Convergência final atingida!' });
      return { steps: all, g: copyMat(g) };
    }

    function computeAll() {
      const gInit = Array.from({length: ROWS}, (_, r) => Array.from({length: COLS}, (_, c) => grid[r][c] ? INF : 0));
      const r1 = computeEdt1(gInit); const r2 = computeEdt2(r1.g);
      return [...r1.steps, ...r2.steps];
    }

    function setMode(m) {
      mode = m; stopPlay();
      root.querySelector('#sim04_edt_btnFinal').className = m === 'final' ? 'sim04_edt_act' : '';
      root.querySelector('#sim04_edt_btnStep').className = m === 'step' ? 'sim04_edt_act' : '';
      root.querySelector('#sim04_edt_stepCtrl').style.display = m === 'step' ? 'block' : 'none';

      if(m === 'step') {
        steps = computeAll(); stepIdx = 0; drawFrame();
      } else {
        const finalSteps = computeAll();
        drawMatrix(finalSteps[finalSteps.length - 1].g);
        root.querySelector('#sim04_edt_sStep').textContent = 'Final';
        root.querySelector('#sim04_edt_sPhase').textContent = 'Concluído';
        root.querySelector('#sim04_edt_sB').textContent = '–';
      }
    }

    function drawFrame() {
      if(!steps.length) return;
      const s = steps[Math.min(stepIdx, steps.length - 1)];
      drawMatrix(s.g, { cursor: s.cursor, changed: s.changed });
      root.querySelector('#sim04_edt_sStep').textContent = (stepIdx + 1) + '/' + steps.length;
      root.querySelector('#sim04_edt_sPhase').textContent = s.phase;
      root.querySelector('#sim04_edt_sB').textContent = s.b > 0 ? s.b : '–';
      root.querySelector('#sim04_edt_stepLabel').textContent = s.label;
    }

    function goStep(d) {
      stepIdx = Math.max(0, Math.min(steps.length - 1, stepIdx + d));
      drawFrame();
    }

    function togglePlay() { playing ? stopPlay() : startPlay(); }
    function startPlay() {
      playing = true; root.querySelector('#sim04_edt_btnPlay').textContent = '⏸ Pausar';
      function tick() {
        if(stepIdx >= steps.length - 1) { stopPlay(); return; }
        stepIdx++; drawFrame();
        playTimer = setTimeout(tick, Math.round(1300 - (+root.querySelector('#sim04_edt_speedSlider').value) * 110));
      }
      tick();
    }
    function stopPlay() { playing = false; if(playTimer) clearTimeout(playTimer); const b = root.querySelector('#sim04_edt_btnPlay'); if(b) b.textContent = '▶ Play'; }

    function clearGrid() { grid = Array.from({length: ROWS}, () => new Array(COLS).fill(0)); refresh(); }
    function refresh() { stopPlay(); if(mode === 'step') { steps = computeAll(); stepIdx = Math.min(stepIdx, steps.length - 1); drawFrame(); } else { setMode('final'); } }

    cv.addEventListener('mousedown', e => {
      const rect = cv.getBoundingClientRect();
      const c = Math.floor(((e.clientX - rect.left) * (cv.width / rect.width) - PAD) / CELL);
      const r = Math.floor(((e.clientY - rect.top) * (cv.height / rect.height) - PAD) / CELL);
      if(r >= 0 && r < ROWS && c >= 0 && c < COLS) { grid[r][c] = grid[r][c] ? 0 : 1; refresh(); }
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'final') setMode('final');
        if(act === 'step') setMode('step');
        if(act === 'clear') clearGrid();
        if(act === 'prev') goStep(-1);
        if(act === 'next') goStep(1);
        if(act === 'play') togglePlay();
      });
    });

    setMode('final');
  }

  function tryInit(){
    var root = document.getElementById('sim-04-tde');
    if (root) initSim04EdtConvergencia(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
""")

**Figure 4.21:** Simulateur interactif 2D (4x4) avec synchronisation stricte des files de propagation horizontale (b) pour obtenir la convergence exacte décrite dans l


<figure id="fig-04-sim-04-tde">
  <img src="imagens/fig-04-sim-04-tde.png" alt=" Simulateur interactif 2D (4x4) avec synchronisation stricte des files de propagation horizontale (b) pour obtenir la convergence exacte décrite dans l'article. " style="max-width:80%" />
  <figcaption><strong>Figure 4.21:</strong>  Simulateur interactif 2D (4x4) avec synchronisation stricte des files de propagation horizontale (b) pour obtenir la convergence exacte décrite dans l'article. </figcaption>
</figure>

#### 4.4.3.1 Transformée de Distance Géodésique

La transformée de distance géodésique associe à chaque pixel la distance minimale jusqu'à un marqueur, sous la contrainte imposée par un masque. Ainsi, la propagation s'effectue exclusivement à travers les pixels autorisés, préservant la connectivité du domaine.

La [Figure 4.22](#fig-04-gdist-menor-caminho) illustre ce processus dans un labyrinthe : (a) le masque `g` ; (b) la distance géodésique `D1` calculée depuis l'entrée ; (c) la distance `D2` calculée depuis la sortie ; et (d) le chemin minimal obtenu à partir de ces deux transformées.

Le chemin optimal est déterminé par la somme des distances (`D1 + D2`). Les pixels appartenant à la trajectoire minimale sont ceux pour lesquels cette somme atteint sa plus petite valeur, définissant une connexion entre l'entrée et la sortie avec une longueur géodésique minimale.

Ce principe permet de résoudre des labyrinthes sans avoir à explorer explicitement toutes les possibilités de parcours. La solution émerge directement de la propagation des distances dans un domaine restreint. Cette approche est particulièrement pertinente pour les labyrinthes de haute complexité, comme ceux construits à partir de structures quasicristallines et de cycles hamiltoniens décrits par Singh (2024).. Dans Zampirolli (2025), ce même formalisme est employé pour résoudre un labyrinthe complexe ; ensuite, la méthode est illustrée sur une version simplifiée du problème.

In [26]:
import numpy as np

# 1 = couloir, 0 = mur
g = np.array([
 [0,1,0,0,0,0,0,0,0,0],
 [0,1,1,1,1,1,0,1,1,1],
 [0,0,0,0,0,1,0,1,0,1],
 [0,1,1,1,0,1,1,1,0,1],
 [0,1,0,1,0,0,0,0,0,1],
 [0,1,0,1,1,1,1,1,1,1],
 [0,1,0,0,0,0,0,0,1,0],
 [0,1,1,1,1,1,1,0,1,0],
 [0,0,0,0,0,0,1,1,1,0],
 [0,0,0,0,0,0,0,0,1,0]
], dtype=np.uint8)

# marqueur de l'entrée
entrada = np.zeros_like(g, dtype=np.uint8)
entrada[0,1] = 1

# marqueur de la sortie
saida = np.zeros_like(g, dtype=np.uint8)
saida[9,8] = 1

# distances géodésiques
D1 = mm.gdist(g, entrada)
D2 = mm.gdist(g, saida)

# somme des distances
S = D1 + D2

# plus petite valeur valide de la somme
dmin = np.min(S[S > 0])

# pixels appartenant à un chemin optimal
caminho = (S == dmin)

print("Distance géodésique minimale :", dmin)

mm.show(
    [g, D1, D2, caminho],
    titles=[
        "Labyrinthe",
        "Distance de l'Entrée",
        "Distance de la Sortie",
        f"Plus court chemin\n(d={dmin})"
    ],
    cols=4,
    figsize=(14,4),
    axis=True
)

Distance géodésique minimale : 15


<Figure size 2100x600 with 4 Axes>

**Figure 4.22:** Plus court chemin géodésique dans un labyrinthe. Les distances géodésiques sont calculées à partir de l


### 4.4.4 Segmentation par *Watershed*

L’algorithme ***Watershed*** interprète une image en niveaux de gris comme une surface topographique, où les valeurs élevées correspondent à des montagnes et les valeurs basses à des vallées ou bassins versants (*catchment basins*). Dans le contexte de la segmentation basée sur des marqueurs, les maxima de la Transformée de Distance sont souvent utilisés pour identifier les régions internes aux objets, fournissant ainsi des graines fiables pour le processus d’inondation.

La segmentation est ensuite réalisée par une simulation conceptuelle d’inondation progressive à partir de ces marqueurs. À mesure que les bassins associés à différentes graines s’étendent, les régions voisines finissent par entrer en contact. À cet instant, des barrières virtuelles, appelées *lignes de partage des eaux*, sont construites, délimitant alors les objets de la scène. Ce mécanisme permet de séparer des objets adjacents ou partiellement superposés, même lorsqu’ils forment une seule composante connexe après le seuillage.

L’implémentation didactique présentée dans ce chapitre explore d’abord le concept de croissance de régions (*region growing*) confinée par un masque binaire, comme détaillé dans l’algorithme interactif de la [Figure 4.23](#fig-04-sim-alg-watershed2).

> ### 📝 Version didactique versus implémentation classique
>
> La fonction `mm.watershed0` n’implémente pas l’algorithme *watershed* classique. Son objectif est d’illustrer, de manière simplifiée, la propagation des marqueurs par croissance de régions (*region growing*), permettant de visualiser comment différentes graines se disputent l’occupation de l’espace disponible. La croissance est délimitée par un masque binaire de support et contrôlée par un suivi de stagnation, produisant un résultat semblable à une partition de Voronoï restreinte à la géométrie des objets d’entrée.
>
> Quant à la fonction `mm.watershed`, elle utilise l’implémentation optimisée d’OpenCV (`cv2.watershed`), qui réalise l’inondation sur une surface topographique définie par l’image d’entrée. Dans ce cas, la propagation des marqueurs est influencée par les valeurs des pixels, faisant en sorte que les lignes de séparation se forment naturellement sur les crêtes du relief.

In [27]:
# @title { display-mode: "form" }
# Préfixe exclusif pour éviter les conflits avec d'autres cellules du notebook
PREFIX = "wat0"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#EBF7F2;color:#0F6E56}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.metric-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:16px}}
.metric-card{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:12px 14px;text-align:left}}
.metric-title{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px}}
.metric-formula{{font-family:var(--font-mono);font-size:12px;color:var(--color-text-primary);margin-bottom:4px}}
.metric-desc{{font-size:12px;color:var(--color-text-secondary);line-height:1.5}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Algoritmo Didático do Watershed Limitado por Máscara — painel interativo</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-bucket" aria-hidden="true"></i></div>
    <div><div class="algo-title">Crescimento de Regiões Confinado por Máscara</div><div class="algo-sub">Inundação concorrente com restrição geométrica de suporte e sincronização síncrona por malha</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Rotular os marcadores sementes em <code>f</code> via <code>mm.label0(f, b)</code>, instanciar a malha dinâmica <code>g ← f.copy()</code> e binarizar a <code>mask</code>.</div></div>
    <div class="step">
      <div class="step-num num-coral">2</div>
      <div class="step-text">
        Enquanto houver pixels não rotulados (<code>while True</code>), reiniciar o controle de atividade <code>mudou ← False</code> e varrer a imagem:
        <div class="step-sub">
          <div class="step-sub-item">Identificar se a coordenada atual é um vazio contido no escopo: <code>g[x,y] == 0 and mask[x,y]</code>.</div>
          <div class="step-sub-item">Avaliar a vizinhança estrutural em <code>mm._viz(f, b, x, y)</code> baseada no estado síncrono estável <code>f</code>.</div>
          <div class="step-sub-item">Se um vizinho possuir rótulo dominante (<code>g[x,y] < f[vy,vx]</code>), a célula em <code>g</code> absorve esse identificador e marca-se <code>mudou ← True</code>.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-purple">3</div><div class="step-text">Verificar ponto fixo: caso uma varredura completa não expanda nenhuma fronteira (<code>not mudou</code>), interrompe-se o laço (<code>break</code>).</div></div>
    <div class="step"><div class="step-num num-blue">4</div><div class="step-text">Atualizar o estado de referência de forma síncrona para a próxima iteração: <code>f ← g.copy()</code>.</div></div>
    <div class="step"><div class="step-num num-blue">5</div><div class="step-text">Se <code>op == 'region'</code>, retornar o mapa de bacias <code>g</code>; caso contrário, extrair as cristas divisórias via <code>mm.gradm(g)</code>.</div></div>
    <div class="note">A sincronização <code>f = g.copy()</code> ao final de cada ciclo impede o crescimento assimétrico ou dependente da ordem da varredura raster (propagação em estilo Jacobi).</div>
  </div>
</div>

<div class="metric-grid">
  <div class="metric-card">
    <div class="metric-title" style="color:#0F6E56">Escopo Geométrico</div>
    <div class="metric-formula">mask[x,y] > 0</div>
    <div class="metric-desc">Restrição binária rígida impedindo o avanço periférico de rótulos.</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#993C1D">Estabilização</div>
    <div class="metric-formula">if not mudou: break</div>
    <div class="metric-desc">Evita loops infinitos interrompendo ao saturar o domínio da máscara.</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#185FA5">Mapeamento Jacobi</div>
    <div class="metric-formula">f = g.copy()</div>
    <div class="metric-desc">Sincronização em bloco após inspeção de todas as coordenadas.</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Geração dos identificadores iniciais pelo mapeamento de componentes conexas e binarização da máscara de suporte.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Expansão Concorrente</div>
      <div class="card-text">Varredura 2D inspecionando vazios internos autorizados. A malha de trabalho <code>g</code> absorve os rótulos lidos da referência estável <code>f</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-purple">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#534AB7">Ponto Fixo Local</div>
      <div class="card-text">A flag <code>mudou</code> monitora mudanças estruturais. Se nenhuma frente avançar, o laço de inundação é finalizado via <code>break</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Sincronização e Saída</div>
      <div class="card-text">Atualização em bloco do estado referencial. A saída pode ser moldada como partições regionais ou linhas de cristas (linhas de watershed).</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Condicionamento Prévio</div><div class="tl-desc">Rotulagem preliminar de marcadores e isolamento booleano do domínio.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div>
      <div class="tl-title">Laço Síncrono Iterativo</div>
      <div class="while-box">
        <div class="while-head">while True</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Redefinição de flag: <code>mudou = False</code></div>
          <div class="step-sub-item">Crescimento condicional: se <code>g[x,y] == 0</code> e estiver na máscara, expande lendo <code>f</code></div>
          <div class="step-sub-item">Controle de estabilidade: <code>if not mudou: break</code></div>
          <div class="step-sub-item">Atualização síncrona: <code>f = g.copy()</code></div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Extração Topológica</div><div class="tl-desc">Retorno condicional das bacias preenchidas ou cálculo morfológico do gradiente de transição.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>mm_watershed.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">Algoritmo com Restrição de Máscara</span>
  </div>
  <div class="code-body">
<pre><span class="kw">def</span> <span class="fn">watershed0</span>(f, mask=<span class="fn">None</span>, b=np.zeros((<span class="num-lit">3</span>,<span class="num-lit">3</span>),dtype=<span class="st">'uint8'</span>), op=<span class="st">'region'</span>):
    f = mm.label0(f, b)
    g = f.copy()
    mask = np.ones_like(f) <span class="kw">if</span> mask <span class="kw">is</span> <span class="fn">None</span> <span class="kw">else</span> (mask > <span class="num-lit">0</span>)
    
    <span class="kw">while</span> <span class="fn">True</span>:
        mudou = <span class="fn">False</span>
        <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">0</span>]):
            <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">1</span>]):
                <span class="kw">if</span> g[x,y] == <span class="num-lit">0</span> <span class="kw">and</span> mask[x,y]:
                    <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f, b, x, y):
                        <span class="kw">if</span> bv <span class="kw">and g[x,y] < f[vy,vx]</span>: 
                            g[x,y] = f[vy,vx]
                            mudou = <span class="fn">True</span>
        <span class="kw">if</span> <span class="kw">not</span> mudou: 
            <span class="kw">break</span>
        f = g.copy()
        
    <span class="kw">return</span> g <span class="kw">if</span> op == <span class="st">'region'</span> <span class="kw">else</span> mm.gradm(g, mm.secross())</pre>
  </div>
</div>

<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>mask = (mask &gt; 0)</code> — Converte a imagem de suporte informada para um mapa Booleano indexável.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">2</div><div class="ann-text"><code>g[x,y] == 0 and mask[x,y]</code> — Filtro ativo: pixels fora da máscara (fundo zero) são ignorados de imediato, confinando as frentes de expansão.</div></div>
  <div class="ann-line"><div class="ann-badge num-purple">3</div><div class="ann-text"><code>if not mudou: break</code> — Mecanismo de escape. Quando todos os espaços internos permitidos forem preenchidos ou estabilizados contra a barreira, o laço aborta de forma limpa.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 580 840" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:580px;display:block;margin:0 auto">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#185FA5"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
    <marker id="arr-p" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#534AB7"/></marker>
  </defs>

  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>

  <rect x="130" y="85" width="300" height="52" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="104" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">rotular sementes e normalizar máscara</text>
  <text x="280" y="121" text-anchor="middle" font-size="11" fill="#085041">f ← mm.label0(f, b) ; g ← f.copy() ; mask ← mask > 0</text>

  <polygon points="280,166 390,196 280,226 170,196" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="201" text-anchor="middle" font-size="12" fill="#0C447C">True?</text>

  <rect x="150" y="256" width="260" height="40" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="281" text-anchor="middle" font-size="12" fill="#712B13">reiniciar ciclo: mudou ← False</text>

  <rect x="120" y="326" width="320" height="72" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="345" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">varredura espacial e expansão</text>
  <text x="280" y="364" text-anchor="middle" font-size="11.5" fill="#712B13">se g[x,y] == 0 e mask[x,y] e f[vy,vx] > g[x,y]:</text>
  <text x="280" y="382" text-anchor="middle" font-size="11.5" font-weight="500" fill="#712B13">g[x,y] ← f[vy,vx] ; mudou ← True</text>

  <polygon points="280,430 380,460 280,490 180,460" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="465" text-anchor="middle" font-size="11.5" fill="#712B13">not mudou?</text>

  <rect x="150" y="520" width="260" height="40" rx="6" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="545" text-anchor="middle" font-size="12" fill="#0C447C">sincronizar malha de ref: f ← g.copy()</text>

  <polygon points="280,600 390,630 280,660 170,630" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="635" text-anchor="middle" font-size="12" fill="#3C3489">op == 'region'?</text>

  <rect x="70" y="695" width="170" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="155" y="719" text-anchor="middle" font-size="11.5" fill="#3C3489">retornar g ( bacias )</text>

  <rect x="320" y="695" width="200" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="420" y="719" text-anchor="middle" font-size="11.5" fill="#3C3489">retornar mm.gradm(g)</text>

  <ellipse cx="280" cy="790" rx="55" ry="22" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="795" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>

  <line x1="280" y1="58" x2="280" y2="85" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="137" x2="280" y2="166" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="280" y1="226" x2="280" y2="256" stroke="#185FA5" stroke-width="1.2" marker-end="url(#arr-b)"/>
  <text x="295" y="240" font-size="11" fill="#185FA5">sim</text>

  <line x1="280" y1="296" x2="280" y2="326" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <line x1="280" y1="398" x2="280" y2="430" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  
  <line x1="280" y1="490" x2="280" y2="520" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="295" y="504" font-size="11" fill="#993C1D">não</text>

  <path d="M 150,540 L 45,540 L 45,196 L 170,196" fill="none" stroke="#185FA5" stroke-width="1.2" marker-end="url(#arr-b)"/>

  <path d="M 380,460 L 460,460 L 460,580 L 280,580 L 280,600" fill="none" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="420" y="450" text-anchor="middle" font-size="11" fill="#993C1D">sim (break)</text>

  <path d="M 390,196 L 500,196 L 500,580 L 280,580" fill="none" stroke="#185FA5" stroke-width="1.2"/>
  <text x="445" y="186" font-size="11" fill="#185FA5">não</text>

  <line x1="170" y1="630" x2="155" y2="630" stroke="#534AB7" stroke-width="1.2"/>
  <line x1="155" y1="630" x2="155" y2="695" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-p)"/>
  <text x="140" y="620" font-size="11" fill="#534AB7">sim</text>

  <line x1="390" y1="630" x2="420" y2="630" stroke="#534AB7" stroke-width="1.2"/>
  <line x1="420" y1="630" x2="420" y2="695" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-p)"/>
  <text x="402" y="620" font-size="11" fill="#534AB7">não</text>

  <line x1="155" y1="735" x2="155" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="155" y1="755" x2="280" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="420" y1="735" x2="420" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="420" y1="755" x2="280" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="280" y1="755" x2="280" y2="768" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figure 4.23:** Algorithme didactique de *Watershed* par croissance de régions limitée par masque.


<figure id="fig-04-sim-alg-watershed2">
  <img src="imagens/fig-04-sim-alg-watershed2.png" alt=" Algorithme didactique de *Watershed* par croissance de régions limitée par masque. " style="max-width:80%" />
  <figcaption><strong>Figure 4.23:</strong>  Algorithme didactique de *Watershed* par croissance de régions limitée par masque. </figcaption>
</figure>

[Figure 4.24](#fig-04-sim-04-watershed) présente un simulateur itératif qui illustre la propagation des marqueurs à travers la région d'intérêt. Chaque marqueur agit comme une source d'inondation qui étend sa zone d'influence jusqu'à rencontrer des régions provenant d'autres graines. Dans l'algorithme d'OpenCV, les pixels appartenant aux lignes de partage sont identifiés par la valeur `-1`, représentant les frontières entre bassins adjacents.

In [28]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-watershed" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-watershed * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-watershed canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-watershed button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-watershed button:hover { background: #e8dfcf; }
  #sim-04-watershed button.ws_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-watershed .ws_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .ws_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .ws_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .ws_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .ws_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .ws_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  .ws_se_grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 4px; max-width: 110px; margin: 6px auto; }
  .ws_se_btn { padding: 6px!important; font-size: 10px!important; font-family: monospace; font-weight: 700; }
  .ws_se_btn.ws_off { color: #8a8371!important; background: #fafaf7!important; border-color: #e4dcc8!important; font-weight: normal; }
  
  .ws_tool_grid { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-top: 6px; }
  .ws_color_dot { width: 10px; height: 10px; border-radius: 2px; display: inline-block; border: 1px solid rgba(0,0,0,0.2); }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">💧 Simulateur : Segmentation par Watershed</span>
  <span class="ws_pill">Propagation avec élément structurant (b)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="ws_grid_stats">
    <div class="ws_stat_box">
      <div class="ws_stat_label">Zone (masque)</div>
      <div id="ws_statPx" class="ws_stat_value" style="color:#8a8371;">0</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Marqueurs</div>
      <div id="ws_statSeeds" class="ws_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Remplissage</div>
      <div id="ws_statFill" class="ws_stat_value" style="color:#2980b9;">0%</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Itération (k)</div>
      <div id="ws_statStep" class="ws_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="ws_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Faites glisser pour dessiner/effacer le masque ou les graines
      </div>
    </div>

    <!-- Painel Lateral de Ferramentas -->
    <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Ferramentas de Desenho -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Outils
        </div>
        <div class="ws_tool_grid">
          <button id="ws_tool_mask" class="ws_active" data-tool="mask">
            <span class="ws_color_dot" style="background:#e4dcc8;"></span> Masque
          </button>
          <button id="ws_tool_erase" data-tool="erase">
            <span class="ws_color_dot" style="background:#fafaf7; border-style:dashed;"></span> Effacer
          </button>
          <button id="ws_tool_s2" data-tool="s2">
            <span class="ws_color_dot" style="background:#2980b9;"></span> Sem. 1
          </button>
          <button id="sim04_ws_tool_s3" data-tool="s3">
            <span class="ws_color_dot" style="background:#27ae60;"></span> Sem. 2
          </button>
          <button id="sim04_ws_tool_s4" data-tool="s4" style="grid-column: span 2;">
            <span class="ws_color_dot" style="background:#b9770e;"></span> Sem. 3
          </button>
        </div>
      </div>

      <!-- Elemento Estruturante (b) -->
      <div class="ws_panel" style="text-align:center;">
        <div style="font-size:11px; font-weight:700; margin-bottom:2px; text-align:left; color:#5e5a4a;">
          Élément structurant (b)
        </div>
        <div class="ws_se_grid">
          <button id="ws_se_0_0" class="ws_se_btn ws_off" data-se="0,0">0</button>
          <button id="ws_se_0_1" class="ws_se_btn ws_active" data-se="0,1">1</button>
          <button id="sim04_ws_se_0_2" class="ws_se_btn ws_off" data-se="0,2">0</button>
          
          <button id="sim04_ws_se_1_0" class="ws_se_btn ws_active" data-se="1,0">1</button>
          <button class="ws_se_btn" style="background:#e4dcc8; cursor:not-allowed; color:#8a8371;" disabled>1</button>
          <button id="sim04_ws_se_1_2" class="ws_se_btn ws_active" data-se="1,2">1</button>
          
          <button id="sim04_ws_se_2_0" class="ws_se_btn ws_off" data-se="2,0">0</button>
          <button id="sim04_ws_se_2_1" class="ws_se_btn ws_active" data-se="2,1">1</button>
          <button id="sim04_ws_se_2_2" class="ws_se_btn ws_off" data-se="2,2">0</button>
        </div>
        <div style="display:flex; gap:4px; margin-top:8px;">
          <button data-se-preset="cross" style="flex:1; font-size:9.5px;">Croix (C-4)</button>
          <button data-se-preset="square" style="flex:1; font-size:9.5px;">Carré (C-8)</button>
        </div>
      </div>

      <!-- Modo de Visualização -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualisation
        </div>
        <div style="display:flex; gap:6px;">
          <button id="ws_btnLabel" class="ws_active" data-mode="label" style="flex:1; justify-content:center;">Final</button>
          <button id="ws_btnAnim" data-mode="anim" style="flex:1; justify-content:center;">Animé</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="ws_animCtrl" class="ws_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Inondation pas à pas
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button data-action="prev" style="flex:1; justify-content:center;">◀</button>
          <button id="ws_btnPlay" data-action="play" style="flex:1; justify-content:center;">▶</button>
          <button data-action="next" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Vit.</label>
          <input type="range" id="ws_speedSlider" min="1" max="10" value="6" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Exemplos Iniciais -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Exemples initiaux
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button data-preset="moedas" style="justify-content:flex-start;">🪙 Pièces en contact</button>
          <button data-preset="celulas" style="justify-content:flex-start;">🦠 Amas de 3</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button data-action="clear" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Tout effacer
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04Watershed(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const ws_COLS = 16, ws_ROWS = 14, ws_CELL = 20, ws_PAD = 10;
    const ws_W = ws_COLS * ws_CELL + ws_PAD * 2, ws_H = ws_ROWS * ws_CELL + ws_PAD * 2;
    const ws_cv = root.querySelector('#ws_Canvas');
    ws_cv.width = ws_W; 
    ws_cv.height = ws_H;
    const ws_ctx = ws_cv.getContext('2d');

    const ws_COLORS = {
      0:  { fill: 'transparent', text: '#8a8371', stroke: '#e4dcc8' },
      1:  { fill: '#fafaf7',     text: '#8a8371', stroke: '#e4dcc8' },
      2:  { fill: '#ebf4fd',     text: '#2980b9', stroke: '#a9cce3' },
      3:  { fill: '#eafaf1',     text: '#27ae60', stroke: '#a3e4d7' },
      4:  { fill: '#fef5e7',     text: '#b9770e', stroke: '#f8c471' }
    };

    let ws_grid = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
    let ws_markers = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
    
    let ws_b = [
      [false, true,  false],
      [true,  true,  true],
      [false, true,  false]
    ];
    
    let ws_currentTool = 'mask'; 
    let ws_mode = 'label';
    let ws_painting = false;

    let ws_animSteps = [];
    let ws_animIdx = 0;
    let ws_playing = false;
    let ws_playTimer = null;

    function ws_compute() {
      let steps = [];
      
      let current = Array.from({length:ws_ROWS}, (_, r) => 
        Array.from({length:ws_COLS}, (_, c) => {
          if (ws_markers[r][c] > 1) return ws_markers[r][c];
          if (ws_grid[r][c] === 1) return 1;
          return 0;
        })
      );
      
      const copy = arr => arr.map(row => [...row]);
      steps.push({ g: copy(current), k: 0 });

      let k = 0;
      while (true) {
        k++;
        let nextG = copy(current);
        let changed = false;

        for (let r = 0; r < ws_ROWS; r++) {
          for (let c = 0; c < ws_COLS; c++) {
            if (current[r][c] === 1) {
              let labels = new Set();
              
              for (let dr = -1; dr <= 1; dr++) {
                for (let dc = -1; dc <= 1; dc++) {
                  if (dr === 0 && dc === 0) continue;
                  if (ws_b[dr+1][dc+1]) { 
                    let nr = r + dr, nc = c + dc;
                    if (nr >= 0 && nr < ws_ROWS && nc >= 0 && nc < ws_COLS) {
                      let val = current[nr][nc];
                      if (val > 1) labels.add(val);
                    }
                  }
                }
              }

              if (labels.size === 1) {
                nextG[r][c] = [...labels][0];
                changed = true;
              } else if (labels.size > 1) {
                let labelsArr = [...labels];
                nextG[r][c] = labelsArr[Math.floor(Math.random() * labelsArr.length)];
                changed = true;
              }
            }
          }
        }

        if (!changed) {
          steps.push({ g: copy(nextG), k: k, done: true });
          break;
        }
        steps.push({ g: copy(nextG), k: k });
        current = nextG;
      }
      
      return { final: current, steps };
    }

    function ws_drawState(matrix, stepInfo) {
      ws_ctx.clearRect(0, 0, ws_W, ws_H);
      let area = 0, filled = 0, numSeeds = new Set();

      for (let r=0; r<ws_ROWS; r++) {
        for (let c=0; c<ws_COLS; c++) {
          const x = ws_PAD + c * ws_CELL, y = ws_PAD + r * ws_CELL;
          const val = matrix[r][c];
          
          if (ws_grid[r][c] === 1) area++;
          if (ws_markers[r][c] > 1) numSeeds.add(ws_markers[r][c]);
          if (val > 1) filled++;

          const col = ws_COLORS[val] || ws_COLORS[0];
          ws_ctx.fillStyle = col.fill; 
          ws_ctx.fillRect(x+1, y+1, ws_CELL-2, ws_CELL-2);
          
          if (val > 1) {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 1; ws_ctx.setLineDash([]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
            ws_ctx.fillStyle = col.text;
            ws_ctx.font = 'bold 10px monospace';
            ws_ctx.textAlign = 'center'; ws_ctx.textBaseline = 'middle';
            ws_ctx.fillText('S' + (val-1), x+ws_CELL/2, y+ws_CELL/2);
          } else if (val === 1) {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 1; ws_ctx.setLineDash([2,2]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
          } else {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 0.5; ws_ctx.setLineDash([]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
          }

          if (r===0) { ws_ctx.fillStyle='#8a8371'; ws_ctx.font='8px monospace'; ws_ctx.textAlign='center'; ws_ctx.fillText(c, x+ws_CELL/2, ws_PAD/2); }
          if (c===0) { ws_ctx.fillStyle='#8a8371'; ws_ctx.font='8px monospace'; ws_ctx.textAlign='center'; ws_ctx.fillText(r, ws_PAD/2, y+ws_CELL/2); }
        }
      }

      let pct = area === 0 ? 0 : Math.round((filled / area) * 100);
      root.querySelector('#ws_statPx').textContent = area;
      root.querySelector('#ws_statSeeds').textContent = numSeeds.size;
      root.querySelector('#ws_statFill').textContent = pct + '%';
      
      if (stepInfo) {
        root.querySelector('#ws_statStep').textContent = stepInfo.done ? 'Concluído' : stepInfo.k;
      } else {
        root.querySelector('#ws_statStep').textContent = 'Finalizado';
      }
    }

    function ws_updateSEUI() {
      for (let r=0; r<3; r++) {
        for (let c=0; c<3; c++) {
          if(r===1 && c===1) continue;
          const btn = root.querySelector('#ws_se_' + r + '_' + c);
          if(!btn) continue;
          if(ws_b[r][c]) {
            btn.textContent = '1';
            btn.className = 'ws_se_btn ws_active';
          } else {
            btn.textContent = '0';
            btn.className = 'ws_se_btn ws_off';
          }
        }
      }
    }

    window.ws_toggleSE = function(r, c) {
      if(r===1 && c===1) return;
      ws_b[r][c] = !ws_b[r][c];
      ws_updateSEUI();
      ws_refresh();
    };

    window.ws_setPresetSE = function(type) {
      for (let r=0; r<3; r++) for (let c=0; c<3; c++) ws_b[r][c] = false;
      ws_b[1][1] = true;
      if (type === 'cross') {
        ws_b[0][1] = ws_b[1][0] = ws_b[1][2] = ws_b[2][1] = true;
      } else if (type === 'square') {
        for (let r=0; r<3; r++) for (let c=0; c<3; c++) ws_b[r][c] = true;
      }
      ws_updateSEUI();
      ws_refresh();
    };

    function ws_refresh() {
      const res = ws_compute();
      if (ws_mode === 'anim') {
        ws_animSteps = res.steps;
        ws_animIdx = 0;
        ws_drawState(ws_animSteps[0].g, ws_animSteps[0]);
      } else {
        ws_drawState(res.final, null);
      }
    }

    function ws_setTool(t) {
      ws_currentTool = t;
      ['mask','erase','s2','s3','s4'].forEach(id => {
        const el = root.querySelector('#ws_tool_' + id);
        if (el) el.classList.toggle('ws_active', id === t);
      });
    }

    function ws_setMode(m) {
      ws_mode = m;
      ws_stopPlay();
      root.querySelector('#ws_btnLabel').classList.toggle('ws_active', m==='label');
      root.querySelector('#ws_btnAnim').classList.toggle('ws_active', m==='anim');
      root.querySelector('#ws_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      ws_refresh();
    }

    function ws_animStep(d) {
      ws_animIdx = Math.max(0, Math.min(ws_animSteps.length-1, ws_animIdx+d));
      ws_drawState(ws_animSteps[ws_animIdx].g, ws_animSteps[ws_animIdx]);
    }

    function ws_togglePlay() {
      if (ws_playing) ws_stopPlay(); else ws_startPlay();
    }

    function ws_startPlay() {
      if (ws_animSteps.length === 0) return;
      ws_playing = true;
      root.querySelector('#ws_btnPlay').textContent = '⏸';
      function tick() {
        if (ws_animIdx >= ws_animSteps.length-1) { ws_stopPlay(); return; }
        ws_animIdx++;
        ws_drawState(ws_animSteps[ws_animIdx].g, ws_animSteps[ws_animIdx]);
        const spd = +root.querySelector('#ws_speedSlider').value;
        const delay = Math.round(1100 - spd * 100);
        ws_playTimer = setTimeout(tick, delay);
      }
      tick();
    }

    function ws_stopPlay() {
      ws_playing = false;
      if (ws_playTimer) { clearTimeout(ws_playTimer); ws_playTimer = null; }
      root.querySelector('#ws_btnPlay').textContent = '▶';
    }

    function ws_clearGrid() {
      ws_grid = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
      ws_markers = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
      ws_stopPlay();
      ws_refresh();
    }

    const ws_PRESETS = {
      moedas: () => {
        let g = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        let m = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        for(let r=3; r<=10; r++) for(let c=2; c<=7; c++) g[r][c] = 1;
        g[3][2]=0; g[3][7]=0; g[10][2]=0; g[10][7]=0; 
        for(let r=4; r<=11; r++) for(let c=6; c<=12; c++) g[r][c] = 1;
        g[4][6]=0; g[4][12]=0; g[11][6]=0; g[11][12]=0;
        m[6][4] = 2;
        m[8][9] = 3;
        return {g, m};
      },
      celulas: () => {
        let g = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        let m = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        for(let r=2; r<=12; r++) for(let c=3; c<=12; c++) {
           if((r>=3 && r<=11) || (c>=4 && c<=11)) g[r][c] = 1;
        }
        g[2][3]=g[2][12]=g[12][3]=g[12][12]=0;
        m[4][6] = 2;
        m[10][5] = 3;
        m[7][10] = 4;
        return {g, m};
      }
    };

    function ws_loadPreset(name) {
      const {g, m} = ws_PRESETS[name]();
      ws_grid = g; ws_markers = m;
      ws_stopPlay();
      ws_refresh();
    }

    function ws_cellAt(e) {
      const rect = ws_cv.getBoundingClientRect();
      const sx = ws_W / rect.width, sy = ws_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - ws_PAD) / ws_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - ws_PAD) / ws_CELL);
      return {r, c};
    }

    function ws_applyTool(r, c) {
      if (ws_currentTool === 'mask') {
        ws_grid[r][c] = 1;
        if (ws_markers[r][c] > 0) ws_markers[r][c] = 0;
      } else if (ws_currentTool === 'erase') {
        ws_grid[r][c] = 0;
        ws_markers[r][c] = 0;
      } else {
        let seedVal = parseInt(ws_currentTool.replace('s',''));
        ws_grid[r][c] = 1;
        ws_markers[r][c] = seedVal;
      }
    }

    ws_cv.addEventListener('mousedown', function(e){
      const {r, c} = ws_cellAt(e);
      if (r<0 || r>=ws_ROWS || c<0 || c>=ws_COLS) return;
      ws_painting = true;
      ws_applyTool(r, c);
      ws_refresh();
    });

    ws_cv.addEventListener('mousemove', function(e){
      if (!ws_painting) return;
      const {r, c} = ws_cellAt(e);
      if (r<0 || r>=ws_ROWS || c<0 || c>=ws_COLS) return;
      ws_applyTool(r, c);
      ws_refresh();
    });
    
    document.addEventListener('mouseup', () => ws_painting = false);

    root.querySelectorAll('[data-tool]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setTool(this.getAttribute('data-tool'));
      });
    });

    root.querySelectorAll('[data-se]').forEach(btn => {
      btn.addEventListener('click', function() {
        const parts = this.getAttribute('data-se').split(',');
        ws_toggleSE(parseInt(parts[0], 10), parseInt(parts[1], 10));
      });
    });

    root.querySelectorAll('[data-se-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setPresetSE(this.getAttribute('data-se-preset'));
      });
    });

    root.querySelectorAll('[data-mode]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setMode(this.getAttribute('data-mode'));
      });
    });

    root.querySelectorAll('[data-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_loadPreset(this.getAttribute('data-preset'));
      });
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'clear') ws_clearGrid();
        if(act === 'prev') ws_animStep(-1);
        if(act === 'next') ws_animStep(1);
        if(act === 'play') ws_togglePlay();
      });
    });

    ws_updateSEUI();
    ws_loadPreset('moedas');
  }

  function tryInitSim04Watershed(){
    var root = document.getElementById('sim-04-watershed');
    if (root) initSim04Watershed(root); else setTimeout(tryInitSim04Watershed, 200);
  }
  tryInitSim04Watershed();
})();
</script>
""")

**Figure 4.24:** Simulateur interactif de l


<figure id="fig-04-sim-04-watershed">
  <img src="imagens/fig-04-sim-04-watershed.png" alt=" Simulateur interactif de l'algorithme *Watershed* par propagation morphologique. Dessinez le masque, positionnez les marqueurs et ajustez l'élément structurant pour observer l'inondation. Lorsque les bassins se rencontrent simultanément, l'égalité est résolue en supposant l'une des régions de manière aléatoire. " style="max-width:80%" />
  <figcaption><strong>Figure 4.24:</strong>  Simulateur interactif de l'algorithme *Watershed* par propagation morphologique. Dessinez le masque, positionnez les marqueurs et ajustez l'élément structurant pour observer l'inondation. Lorsque les bassins se rencontrent simultanément, l'égalité est résolue en supposant l'une des régions de manière aléatoire. </figcaption>
</figure>

#### *Pipeline* Morphologique du *Watershed*

Le *watershed* basé sur marqueurs intègre généralement un flux plus large de segmentation. Dans les images réelles, des étapes de prétraitement sont souvent nécessaires pour améliorer le contraste, réduire les bruits et générer des marqueurs fiables. Ce flux complet est résumé dans la [Tableau 4.6](#tbl-04-watershed-pipeline-real).

<a id="tbl-04-watershed-pipeline-real"></a>

**Tabela 4.6:** *Pipeline* complet du *watershed* basé sur marqueurs pour images réelles.

| Étape | Opération | Finalité |
| --- | --- | --- |
| 1 | CLAHE + Lissage | Amélioration du contraste et réduction du bruit |
| 2 | Seuillage | Séparation initiale entre objet et fond |
| 3 | Ouverture/Fermeture | Suppression des bruits et petites imperfections |
| 4 | Dilatation du masque | Identification du fond sûr (*Sure Background*) |
| 5 | Transformée de distance + Seuil | Identification de l'objet sûr (*Sure Foreground*) |
| 6 | Région incertaine | Différence entre fond sûr et objet sûr |
| 7 | `cv2.watershed` | Propagation des marqueurs dans la région incertaine |


Pour mettre en avant exclusivement les concepts de transformée de distance, marqueurs et inondation topographique, l'exemple de la [Figure 4.25](#fig-04-watershed-didatico) utilise une image binaire synthétique et adopte un flux simplifié, résumé dans la [Tableau 4.7](#tbl-04-watershed-pipeline-simples).

<a id="tbl-04-watershed-pipeline-simples"></a>

**Tabela 4.7:** *Pipeline* simplifié utilisé dans l'exemple didactique de la [Figure 4.25](#fig-04-watershed-didatico).

| Étape | Opération | Finalité |
| --- | --- | --- |
| 1 | Transformée de distance | Construction de la surface topographique |
| 2 | Seuil de la TD | Extraction des marqueurs (*Sure Foreground*) |
| 3 | Dilatation | Détermination du fond sûr (*Sure Background*) |
| 4 | Région incertaine | Différence entre fond et marqueurs |
| 5 | `cv2.watershed` | Propagation des marqueurs et génération des frontières |


In [29]:
import cv2

# 1. Image synthétique et transformée de distance
f_sint = np.zeros((20, 20), dtype=np.uint8)
cv2.circle(f_sint, (6,  10), 5, 255, -1)
cv2.circle(f_sint, (14, 10), 5, 255, -1)
dist = mm.dist(f_sint)

# 2. Marqueurs (pics de distance)
m = (dist > 0.8 * dist.max()).astype(np.uint8) * 255

# 3. Exécution du watershed0 conditionnel (m=marqueurs d'abord, mask=f_sint)
w_reg  = mm.watershed0(m, mask=f_sint, op='region')
w_line = mm.watershed0(m, mask=f_sint, op='line')

#w_reg  = mm.watershedB(m, mask=f_sint, op='region')
#w_line = mm.watershedB(m, mask=f_sint, op='line')

w_reg  = mm.watershed(m, mask=f_sint, op='region')
w_line = mm.watershed(m, mask=f_sint, op='line')

# 4. Affichage des résultats
mm.show([f_sint, dist, m, w_reg, w_line], cols=5, figsize=(16, 4),
        titles=["Original", "Distance", "Marqueurs", "Régions", "Lignes"])

<Figure size 2400x600 with 5 Axes>

**Figure 4.25:** *Pipeline* *watershed* délimité par masque sur image binaire 20×20.


**Application aux pièces superposées :**

In [30]:
img_base = img_coins_gray

# 1. Simuler des pièces superposées/connectées (en utilisant la masque binaire de base)
img_sobrepostas = mm.dil(img_final, mm.sebox(40))

# 2. Ouverture morphologique pour nettoyer les bruits
opening = mm.open(img_sobrepostas, mm.sebox(2))

# 3. Transformée de distance
dist = mm.dist(opening)
dist_vis=(255*(dist/dist.max())).astype(np.uint8) if dist.max() > 0 else dist.astype(np.uint8)

# 4. Pics sûrs (Marqueurs des pièces)
picos = (dist > 0.5 * dist.max()).astype(np.uint8) * 255

# 5. Exécution du Watershed (Ajusté pour utiliser la nouvelle signature)
# Nous passons 'opening' directement dans mask, car elle délimite la portée d'expansion des pièces
ws_region = mm.watershed(picos, mask=opening, op='region')
ws_line   = mm.watershed(picos, mask=opening, op='line')

# 6. Comptage des objets (Ignore le fond 0)
labels = np.unique(ws_region)
labels = labels[labels > 0]
print(f"Objets détectés : {len(labels)}")

# 7. Annotation finale
img_annotated = cv2.cvtColor(img_base, cv2.COLOR_GRAY2BGR)

for idx, label_id in enumerate(labels):
    mask_reg = (ws_region == label_id).astype(np.uint8)
    if mask_reg.sum() < 500: continue
    
    cy, cx = np.mean(np.where(mask_reg), axis=1).astype(int)
    cv2.putText(img_annotated, str(idx + 1), (cx - 25, cy + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 3.2, (0, 255, 0), 8, cv2.LINE_AA)

# Dessine les lignes de séparation en rouge
mask_ann = mm.dil(ws_line, np.ones((11, 11), np.uint8))
img_annotated[mask_ann > 0 ] = [255, 0, 0]

# Affichage
mm.show(
    [img_base, img_sobrepostas, dist_vis, picos, ws_region, img_annotated],
    titles=["Original", "Superposées", "Distance", "Marqueurs", "Watershed", "Annoté"],
    cols=3, rows=2, figsize=(12, 8)
)

Objets détectés : 12


<Figure size 1800x1200 with 6 Axes>

**Figure 4.26:** *Pipeline* *Watershed* pour séparation de pièces superposées : de la masque binaire dilatée jusqu


## 4.5 Extraction de Composants et Descripteurs de Forme

Après la segmentation et le raffinement morphologique, l’étape suivante consiste à identifier individuellement chaque objet présent dans l’image et à extraire ses propriétés géométriques. Cette étape est fondamentale pour les tâches de mesure, de classification et de reconnaissance de formes.

Un **composant connexe** est un ensemble maximal de pixels appartenant à l’objet qui restent mutuellement connectés selon une relation de connectivité préalablement définie (connectivité 4 ou 8). Après l’étiquetage, chaque composant reçoit un identifiant unique, permettant d’analyser ses caractéristiques individuellement.

OpenCV propose deux approches complémentaires pour cette analyse, résumées dans la [Tableau 4.8](#tbl-04-componentes-vs-contornos).

<a id="tbl-04-componentes-vs-contornos"></a>

**Tabela 4.8:** Comparaison entre les approches basées sur les composants connexes et sur les contours.

| | `connectedComponentsWithStats` | `findContours` |
|---|---|---|
| **Retourne** | étiquette par pixel et statistiques par composant | séquence de points décrivant le contour |
| **Descripteurs directs** | aire, *boîte englobante* et centroïde | périmètre, forme et hiérarchie |
| **Objets en contact** | tend à fusionner les régions connectées | tend à produire un unique contour externe |
| **Usage typique** | comptage, filtrage et étiquetage | analyse géométrique et descripteurs de forme |


### 4.5.1 Rotation et Statistiques des Composants

La fonction `cv2.connectedComponentsWithStats` réalise simultanément la rotation des composants connexes et l'extraction de descripteurs de base pour chaque région. L'opérateur renvoie :

- une image de labels (`labels`) ;
- des statistiques géométriques (`stats`) ;
- les coordonnées des centroïdes (`centroids`).

Les statistiques incluent l'aire, la largeur, la hauteur et la position de la *bounding box* minimale alignée sur les axes de l'image.

La [Figure 4.27](#fig-04-componentes) illustre l'application de cet opérateur après le *pipeline* de segmentation des pièces. Chaque composant connexe reçoit une couleur distincte et son aire est annotée directement sur l'image.

In [31]:
# 1. Étiquetage et statistiques
n, labels, stats, centroids = cv2.connectedComponentsWithStats(
    img_final, connectivity=8
)

# 2. Coloration des composants
np.random.seed(4)
colors    = np.random.randint(50, 255, (n, 3), dtype=np.uint8)
colors[0] = [0, 0, 0]                          # fond noir
img_colored   = colors[labels]
img_annotated = img_colored.copy()

# 3. Tableau des descripteurs
print(f"Composants détectés (hors fond) : {n - 1}")
print(f"\n{'ID':>4} {'Área':>8} {'cx':>6} {'cy':>6} {'w':>6} {'h':>6}")
print("-" * 42)
for i in range(1, n):
    area = stats[i, cv2.CC_STAT_AREA]
    cx, cy = int(centroids[i, 0]), int(centroids[i, 1])
    w, h   = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
    print(f"{i:>4} {area:>8} {cx:>6} {cy:>6} {w:>6} {h:>6}")
    for cor, esp in [((0,0,0), 10), ((255,0,0), 5)]:
        cv2.putText(img_annotated, f"{i}: {area}",
                    (cx - 150, cy + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, cor, esp, cv2.LINE_AA)

mm.show(
    [img_coins_gray, img_final, img_colored, img_annotated],
    titles=["Original", "Segmentation finale", "Composants connexes", "Superficies annotées"],
    cols=4, figsize=(18, 6)
)

Composants détectés (hors fond) : 12

  ID     Área     cx     cy      w      h
------------------------------------------
   1   231969   1484    280    553    542
   2   158967    917    250    453    468
   3   139229    250    312    433    419
   4   175550    857    821    474    465
   5   222882   1442    889    539    531
   6   210043    316    977    527    516
   7   343376    934   1559    667    665
   8   213641   1555   1574    530    515
   9   147880    328   1543    432    438
  10   187280    363   2111    487    492
  11   150110   1487   2215    433    444
  12   215387    932   2292    528    522


<Figure size 2700x900 with 4 Axes>

**Figure 4.27:** Composants connexes extraits après le *pipeline* CLAHE → Otsu → nettoyage morphologique. Chaque objet est coloré avec une couleur distincte et annoté avec sa superficie en pixels.


### 4.5.2 Descripteurs de forme

Alors que `connectedComponentsWithStats` opère sur les régions, `cv2.findContours` agit directement sur leurs frontières. L'opérateur retourne, pour chaque objet, une séquence ordonnée de points décrivant son contour.

À partir de cette représentation, il est possible de calculer des descripteurs géométriques qui ne sont pas fournis directement par l'étiquetage des composants :

- **Aire** (`cv2.contourArea`)

- **Périmètre** (`cv2.arcLength`) ;

- **Circularité** ($C = \frac{4\pi A}{P^2}$, où $A$ est l'aire et $P$ le périmètre) ;

- **Approximation polygonale** (`cv2.approxPolyDP`) ;

- **Enveloppe convexe** (*convex hull*) ;

- **Hiérarchie des contours**, permettant de représenter les relations parent-enfant entre les régions et leurs trous internes.

La circularité atteint une valeur maximale égale à 1 pour un cercle parfait et diminue à mesure que l'objet devient plus allongé ou présente des irrégularités sur son bord.

La [Figure 4.28](#fig-04-contornos) présente les contours extraits des pièces segmentées, ainsi que les valeurs de circularité calculées pour chaque objet.

In [32]:
contornos, hierarquia = cv2.findContours(
    img_final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

img_contornos  = cv2.cvtColor(img_coins_gray, cv2.COLOR_GRAY2BGR)
img_circulares = img_contornos.copy()

print(f"Contours détectés : {len(contornos)}")
print(f"\n{'ID':>4} {'Área':>8} {'Perímetro':>10} {'Circularidade':>14}")
print("-" * 42)

for i, cnt in enumerate(contornos, start=1):
    area  = cv2.contourArea(cnt)
    perim = cv2.arcLength(cnt, closed=True)
    circ  = (4 * np.pi * area / perim**2) if perim > 0 else 0
    M     = cv2.moments(cnt)
    cx    = int(M["m10"] / M["m00"]) if M["m00"] > 0 else 0
    cy    = int(M["m01"] / M["m00"]) if M["m00"] > 0 else 0

    print(f"{i:>4} {area:>8.0f} {perim:>10.1f} {circ:>14.3f}")

    cv2.drawContours(img_contornos,  [cnt], -1, (0, 255, 0), 3)
    cv2.drawContours(img_circulares, [cnt], -1, (0, 255, 0), 3)
    for cor, esp in [((0,0,0), 8), ((255,0,0), 3)]:
        cv2.putText(img_circulares, f"{circ:.2f}",
                    (cx - 80, cy + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 3.6, cor, esp, cv2.LINE_AA)

mm.show(
    [img_final, img_contornos, img_circulares],
    titles=["Segmentation finale", "Contours", "Circularité"],
    cols=3, figsize=(18, 6)
)

Contours détectés : 12

  ID     Área  Perímetro  Circularidade
------------------------------------------
   1   214626     1769.6          0.861
   2   149480     1465.1          0.875
   3   186570     1654.9          0.856
   4   147252     1458.6          0.870
   5   212886     1756.3          0.867
   6   342424     2232.5          0.863
   7   209282     1767.7          0.842
   8   222108     1809.7          0.852
   9   174854     1616.4          0.841
  10   138602     1471.5          0.804
  11   158306     1538.7          0.840
  12   231181     1847.4          0.851


<Figure size 2700x900 with 3 Axes>

**Figure 4.28:** Contours extraits avec *findContours*. Chaque pièce est annotée avec sa circularité — des valeurs proches de 1 confirment une forme circulaire.


### 4.5.3 Conexão com a Detecção Moderna de Objetos

Os descritores extraídos nas seções anteriores — especialmente *bounding boxes*, centróides, áreas e medidas de forma — estabelecem uma ponte natural entre a segmentação morfológica clássica e os sistemas modernos de detecção de objetos. Embora as técnicas estudadas neste capítulo utilizem operações sobre pixels e regiões segmentadas, muitas das representações produzidas são diretamente compatíveis com os formatos empregados em modelos contemporâneos de visão computacional.

Detectores baseados em aprendizado profundo, como a família YOLO (*You Only Look Once*) (REDMON, 2016), operam diretamente sobre imagens coloridas e produzem, para cada objeto detectado, uma *bounding box* descrita pelo centro $(cx,cy)$ e pelas dimensões $(w,h)$, além de uma classe e uma pontuação de confiança. Essa representação compartilha a mesma estrutura geométrica básica obtida por `connectedComponentsWithStats`, embora seja produzida por um modelo aprendido e não por segmentação explícita.

A [Figure 4.29](#fig-04-bbox) ilustra como as *bounding boxes* obtidas por morfologia podem ser exportadas no formato YOLO para compor conjuntos de dados utilizados no treinamento ou na avaliação de detectores.

In [33]:
H_img, W_img = img_coins_gray.shape
img_bbox = cv2.cvtColor(img_coins_gray, cv2.COLOR_GRAY2BGR)

CLASSE = 0   # 0 = pièce (unique catégorie dans cet exemple)

print(f"{'cls':>4} {'cx_n':>8} {'cy_n':>8} {'w_n':>8} {'h_n':>8}  ← format YOLO")
print("-" * 54)

yolo_linhas = []
for i in range(1, n):
    x0 = stats[i, cv2.CC_STAT_LEFT]
    y0 = stats[i, cv2.CC_STAT_TOP]
    w  = stats[i, cv2.CC_STAT_WIDTH]
    h  = stats[i, cv2.CC_STAT_HEIGHT]

    cx_n = (x0 + w / 2) / W_img
    cy_n = (y0 + h / 2) / H_img
    w_n  = w / W_img
    h_n  = h / H_img

    yolo_linhas.append(f"{CLASSE} {cx_n:.4f} {cy_n:.4f} {w_n:.4f} {h_n:.4f}")
    print(f"{CLASSE:>4} {cx_n:>8.4f} {cy_n:>8.4f} {w_n:>8.4f} {h_n:>8.4f}")

    cv2.rectangle(img_bbox, (x0, y0), (x0 + w, y0 + h), (0, 255, 0), 4)
    cv2.putText(img_bbox, f"moeda", (x0 + 8, y0 + 60),
                cv2.FONT_HERSHEY_SIMPLEX, 3.8, (255, 0, 0), 5, cv2.LINE_AA)

# Exporter le fichier d'annotation au format YOLO
with open("moedas.txt", "w") as f:
    f.write("\n".join(yolo_linhas))
print("\nAnnotation enregistrée dans pièces.txt")

mm.show(
    [img_coins_gray, img_final, img_bbox],
    titles=["Original", "Segmentation finale", "Boîtes englobantes (format YOLO)"],
    cols=3, figsize=(18, 6)
)

 cls     cx_n     cy_n      w_n      h_n  ← format YOLO
------------------------------------------------------
   0   0.7753   0.1102   0.2880   0.2117
   0   0.4784   0.0984   0.2359   0.1828
   0   0.1331   0.1225   0.2255   0.1637
   0   0.4464   0.3217   0.2469   0.1816
   0   0.7523   0.3471   0.2807   0.2074
   0   0.1664   0.3812   0.2745   0.2016
   0   0.4862   0.6104   0.3474   0.2598
   0   0.8115   0.6154   0.2760   0.2012
   0   0.1724   0.6023   0.2250   0.1711
   0   0.1893   0.8258   0.2536   0.1922
   0   0.7763   0.8652   0.2255   0.1734
   0   0.4854   0.8953   0.2750   0.2039

Annotation enregistrée dans pièces.txt


<Figure size 2700x900 with 3 Axes>

**Figure 4.29:** *Boîtes englobantes* dérivées des composantes connexes superposées à l


Le format YOLO stocke chaque objet dans une ligne contenant cinq champs :

$$
\texttt{classe}\;\;\texttt{cx}\;\;\texttt{cy}\;\;\texttt{w}\;\;\texttt{h}
$$

où $(cx,cy)$ représente le centre de la *bounding box* et $(w,h)$ ses dimensions. Toutes les valeurs géométriques sont normalisées dans l'intervalle $[0,1]$ par rapport à la largeur et à la hauteur de l'image. La **classe** est un identifiant entier associé à une catégorie définie par l'ensemble de données (par exemple, `0 → pièce de monnaie`). Lorsqu'il y a plusieurs catégories — comme la pièce d'or (`0`), la pièce d'argent (`1`) et le disque en plastique (`2`) — il suffit d'attribuer l'identifiant correspondant à chaque objet avant l'exportation, en maintenant exactement le même format d'annotation. Dans l'exemple précédent, ces informations ont été stockées dans le fichier `moedas.txt`.

Le flux présenté dans ce chapitre — segmentation → étiquetage → extraction de *bounding boxes* — correspond conceptuellement à l'étape d'**annotation** (*labeling*) employée dans la construction d'ensembles d'entraînement pour les détecteurs modernes. Des outils spécialisés, comme Label Studio et Roboflow, automatisent ce processus sur des images complexes, mais la logique fondamentale reste la même : associer à chaque objet une région d'intérêt et une classe. Dans des scénarios contrôlés, avec un arrière-plan uniforme et des objets bien séparés, les techniques morphologiques peuvent même générer des annotations automatiquement ou servir de point de départ pour l'étiquetage manuel, réduisant considérablement l'effort de construction de l'ensemble de données. Dans des applications réelles plus complexes, cependant, la validation humaine reste nécessaire pour garantir la qualité des annotations.

> ### 📝 Évaluation : IoU (*Intersection over Union*)
>
> Une méthode simple pour évaluer la qualité d'une segmentation consiste à la comparer à un masque de référence (*ground truth*). La métrique la plus couramment utilisée à cette fin est la **IoU** (*Intersection over Union*) :
>
> <a id="eq-04-iou"></a>
$$
> \text{IoU}=
> \frac{|A\cap B|}
> {|A\cup B|}
> \tag{4.16}
$$

>
> où $A$ représente la segmentation produite par l'algorithme et $B$ la segmentation de référence.
>
> La valeur de l'IoU varie entre 0 et 1. Plus la valeur est élevée, plus la superposition entre les masques est importante. Une IoU égale à 1 indique une correspondance parfaite entre la segmentation obtenue et la référence.
>
> La même métrique est également largement utilisée en détection d'objets, où elle est appliquée aux *bounding boxes* prédites et annotées. Dans ce domaine, des valeurs d'IoU supérieures à 0,5 sont souvent adoptées comme critère minimum pour considérer une détection comme correcte.

> ### 💡 Au-delà de la morphologie
>
> Les techniques étudiées dans ce chapitre segmentent les objets en exploitant la connectivité spatiale, les opérations morphologiques et le relief topographique. Il existe cependant des approches alternatives basées sur le regroupement de caractéristiques, comme l'algorithme *k-means*, les modèles de mélange gaussien (GMM) et des méthodes plus récentes fondées sur l'apprentissage profond. Ces techniques seront reprises dans la Partie II de l'ouvrage, dédiée à la vision par ordinateur.

## 4.6 Résumé

Dans ce chapitre, les principales techniques de segmentation et de morphologie mathématique ont été présentées, concluant ainsi l'étude du TNI dans le domaine spatial.

* **Prétraitement et seuillage :** La combinaison entre l'égalisation adaptative CLAHE et la méthode d'Otsu s'est révélée efficace pour induire une séparation bimodale dans l'histogramme et simplifier la binarisation d'images à éclairage non uniforme.
* **Érosion et dilatation :** Opérateurs morphologiques fondamentaux basés sur la recherche de minima et de maxima locaux dans un voisinage défini par l'élément structurant $B$. Ce sont des opérateurs duaux par le complément, implémentés au moyen des fonctions `mm.ero` et `mm.dil`.
* **Ouverture et fermeture :** Compositions d'érosion et de dilatation permettant de supprimer les bruits, de lisser les contours et de combler les petites lacunes, tout en préservant la structure globale des objets.
* **Reconstruction morphologique :** Processus géodésique itératif qui propage un marqueur dans les limites imposées par un masque, constituant la base d'opérateurs tels que `mm.clohole` et `mm.edgeoff`.
* **Pipeline de nettoyage binaire :** Flux consolidé composé de CLAHE → Otsu → ouverture → `mm.clohole` → ouverture restreinte → `mm.edgeoff`, produisant des masques adaptés à l'analyse quantitative.
* **Morphologie en niveaux de gris :** Extension algébrique basée sur des minima et maxima pondérés, permettant des opérateurs tels que le gradient morphologique et les filtres *top-hat* pour le rehaussement de structures locales.
* **Transformée de distance et Watershed :** La Transformée de distance a permis de générer des marqueurs automatiques pour l'algorithme *watershed*, rendant possible la séparation d'objets adjacents ou partiellement superposés.
* **Composantes connexes et descripteurs :** L'étiquetage de régions (`cv2.connectedComponentsWithStats`) et l'extraction de contours (`cv2.findContours`) ont permis de calculer des descripteurs géométriques tels que l'aire, le centroïde, le périmètre, la circularité et les *bounding boxes*.
* **Lien avec la vision par ordinateur moderne :** Les *bounding boxes* extraites par morphologie ont été exportées au format YOLO, mettant en évidence le lien entre les techniques classiques de segmentation et les systèmes modernes de détection d'objets.

Le chapitre 5 introduira les techniques de traitement dans le **domaine fréquentiel**, abordant la **Transformée de Fourier**, le filtrage spectral et les fondements de la **compression d'images**, notamment la DCT, le JPEG et les *wavelets*.

## 4.7 🤖 Utilisation de Gemini Notebook comme tuteur complémentaire

Dans cette édition, l'utilisation de **Gemini Notebook** est encouragée comme outil d'apprentissage complémentaire. Fondé sur l'intelligence artificielle, le système utilise exclusivement les documents fournis par l'auteur comme source de connaissances, produisant des réponses alignées sur le contenu et l'approche adoptés tout au long de ce chapitre.

> ### ❗ 🎓 Étudiez avec le tuteur intelligent
>
> [🚀 ACCÉDER À GEMINI NOTEBOOK : CHAPITRE 04](https://notebooklm.google.com/notebook/5dafcbfa-ad58-44f9-9707-4f761b0a6c70)
>
> #### 🌐 Langue et langage de programmation
>
> Le projet de ce chapitre dans Gemini Notebook a été construit uniquement avec le texte en **portugais** et les exemples de code en **Python**. Si vous étudiez à partir de l'édition en anglais ou en français, ou si vous suivez le parcours en C++, les réponses du tuteur peuvent ne pas correspondre exactement à la version que vous lisez.
>
> #### ⚠️ Avertissement concernant le contenu généré par l'IA
>
> Bien qu'il s'agisse d'un outil d'aide à l'étude précieux, Gemini Notebook peut éventuellement produire des réponses incomplètes, imprécises ou incorrectes. Il est recommandé de valider les informations en consultant le matériel du chapitre, les livres, les articles scientifiques et d'autres sources académiques fiables. Chaque fois que possible, exécutez et expérimentez les exemples pratiques présentés tout au long du texte afin de consolider la compréhension des concepts.

## 4.8 Liste d'exercices

1. **(10 %)** Implémentez manuellement le critère d’Otsu sans utiliser `cv2.threshold`. Calculez la variance interclasse $\sigma_B^2(T)$ pour tous les seuils $T \in [0,255]$ à l’aide de `mm.hist`, identifiez le seuil optimal $T^*$ et comparez le résultat avec la valeur obtenue par OpenCV. Tracez $\sigma_B^2$ en fonction de $T$ et mettez en évidence le point de maximum.

2. **(15 %)** Appliquez un seuillage adaptatif avec des blocs de taille 11, 31 et 51 à une image présentant un éclairage non uniforme. Comparez les résultats avec le seuillage global d’Otsu et discutez des avantages et des limites de chaque approche.

3. **(15 %)** Exécutez le *pipeline watershed* de l’image de pièces de monnaie en faisant varier le seuil appliqué à la Transformée de Distance ($0,3$, $0,5$ et $0,7$ fois la valeur maximale). Expliquez comment ce paramètre influence la génération des marqueurs, la séparation des objets adjacents et l’apparition de sur-segmentation.

4. **(15 %)** À l’aide de `mm.drawImg`, construisez une démonstration visuelle pas à pas de l’érosion d’une image binaire 7×7 avec un élément structurant carré 3×3. Pour chaque position analysée, indiquez si l’élément structurant est entièrement contenu dans l’objet et justifiez la valeur attribuée au pixel de sortie.

5. **(15 %)** Démontrez expérimentalement la dualité entre l’érosion et la dilatation en vérifiant l’identité $(A \ominus B)^c = A^c \oplus \hat{B}$
   à l’aide de `mm.ero`, `mm.dil` et `mm.bnot`. Calculez la différence pixel à pixel entre les deux membres de l’équation et présentez le résultat à l’aide de `mm.histImg` ou d’une visualisation équivalente.

6. **(15 %)** Implémentez manuellement le gradient morphologique en utilisant uniquement `mm.ero` et `mm.dil`, puis comparez le résultat avec `cv2.morphologyEx(img, cv2.MORPH_GRADIENT, B)`. Évaluez l’effet de différents éléments structurants (carré 3×3, disque 5×5 et ligne 1×9) sur la détection des contours.

7. **(15 %)** Construisez un *pipeline* complet pour le comptage et la classification de pièces de monnaie par taille (petite, moyenne et grande) en utilisant l’aire et la circularité comme descripteurs. Générez manuellement un masque de référence (*ground truth*) et calculez la métrique IoU (*Intersection over Union*) pour évaluer la qualité de la segmentation. Présentez les résultats dans un tableau et au moyen de visualisations produites avec `mm.show`.

## Références du chapitre

La base théorique de ce chapitre repose sur les ouvrages suivants :

* Gonzalez (2018) pour les concepts de segmentation, de seuillage d’Otsu, de transformée de distance, de *watershed*, de morphologie mathématique et de descripteurs de forme.
* Matheron (1975) et Serra (1982) pour la base théorique originale, la formulation algébrique et le développement de la morphologie mathématique.
* Szeliski (2022) pour la segmentation basée sur les régions, l’étiquetage des composantes connexes, le *watershed* basé sur des marqueurs et l’évaluation de la segmentation à l’aide de la métrique IoU.
* Bradski (2008) pour l’utilisation pratique de la bibliothèque OpenCV, y compris des fonctions telles que `cv2.distanceTransform`, `cv2.watershed`, `cv2.connectedComponentsWithStats` et `cv2.findContours`.
* Redmon (2016) pour l’introduction aux détecteurs modernes de la famille YOLO et leur relation avec les descripteurs géométriques tels que les *bounding boxes* extraites par segmentation.
* Singh (2024) pour la construction de labyrinthes complexes basés sur des cycles hamiltoniens sur des pavages quasicristallins, utilisés comme exemple d’application de la transformée de distance géodésique et d’algorithmes de recherche de chemins.
* Zampirolli (2025) pour l’implémentation des opérateurs morphologiques, des transformées géodésiques et de la résolution de labyrinthes par propagation de distances dans des domaines restreints.

## Références du Chapitre


BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

LOTUFO, R.A.; ZAMPIROLLI, F.A. **Fast multidimensional parallel Euclidean distance transform based on mathematical morphology**. 2001.

MATHERON, G. **Random Sets and Integral Geometry**. New York, John Wiley \& Sons, 1975.

REDMON, Joseph *et al*. **You Only Look Once: Unified, Real-Time Object Detection**. 2016.

SERRA, Jean. **Image Analysis and Mathematical Morphology**. London, Academic Press, 1982.

SINGH, S.; LLOYD, J.; FLICKER, F. **Hamiltonian Cycles on Ammann-Beenker Tilings**. 2024.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

ZAMPIROLLI, Francisco de Assis *et al*. **Teaching Hands-On Digital Image Processing with morph.py: Methods and Comprehensive Results**. 2025.

*Référence introuvable pour : staticmethod*